# 데이터셋 구축 — 리스테리아 시료 + 기상·대기·풍향 + 식품 + 토양 + 검사 목적·시료 상태·유통기한

시료 파일 `리스테리아ML/data/listeria_body.parquet`(676,537행)에 식정원 DB(`TOXIN_MODEL_PRED`)의 자료를 붙여 `데이터셋.parquet` 하나로 만든다. 과정과 출력을 이 노트북에 모두 남긴다.

| 항목 | 내용 |
|---|---|
| 입력 | `리스테리아ML/data/listeria_body.parquet` (원본 파일은 바꾸지 않는다) |
| 붙이는 것 | 기상·대기 17개 변수 + 풍향(sin·cos) 2개 — 변수마다 기준일 당일(D-0)~110일 전(D-110) 111칼럼 / 식품(수분활성도·식품pH·염도) / 토양(pH·유기물·유효인산) / 검사 목적·시료 상태·유통기한 변수 |
| 결과 | `데이터셋.parquet`(시료 1행 = 1행, 시료 파일과 같은 676,537행) · `모든_컬럼설명.parquet` |
| 원자료 | `raw/` — DB 에서 받은 그대로. 조인은 이 파일로 한다 |
| 중간 결과 | `중간/` — 변수별 조인 결과 |
| 공통 규칙 | 결측은 채우지 않는다 · 행 키 `lims_data_key` 와 타깃 `judge_result` 를 남긴다 · 기상 칼럼 이름은 `<변수>_d000`(기준일 당일) ~ `<변수>_d110`(110일 전) |

**절 구성**: 1 설정 → 2 시료 파일(2-1 기준일과 좌표가 무엇인지) → 3 DB 원자료 받기 → 4 기상·대기 조인(4-0 변수를 고른 근거) → 5 풍향 조인 → 6 식품 → 7 토양 → 8 검사 목적·시료 상태·유통기한 변수 → 9 합치기·저장 → 10 컬럼설명 → 11 원자료로 다시 계산해 대조

**읽을 때 주의** — 이 데이터셋의 기상·토양 값은 "제조일·제조 장소"의 값이 아니라 "**기준일(`mnfctur_de`) · 시료 주소가 속한 법정동 대표 지점**"의 값이다. 기준일이 무슨 날짜인지(`date_basis`), 주소가 누구의 주소인지(`addr_source`)는 행마다 다르다(2-1절).

## 1. 설정

경로·조인 설정을 이 셀 하나에 모은다.

- `FORCE_DB = False`: `raw/` 에 원자료 파일이 있으면 DB 에 다시 가지 않는다. `True` 면 다시 받는다.
- `FORCE_JOIN = False`: `중간/` 에 조인 결과가 있으면 다시 만들지 않는다. 조인 규칙을 바꾸면 `True` 로 돌린다.

In [1]:
import decimal
import gc
import os
import re
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import psycopg
from dotenv import dotenv_values

SEED = 42
KEY_COL = 'lims_data_key'      # 행 키
DATE_COL = 'mnfctur_de'        # 기준일 (기상·토양 조인의 기준 날짜)
TARGET_COL = 'judge_result'    # 타깃 (검출 / 불검출)

# 경로
ROOT = Path(r'C:\Users\NT751\Desktop\천영진')
BODY_PATH = ROOT / '리스테리아ML' / 'data' / 'listeria_body.parquet'   # 시료 파일 (읽기만 한다)
ENV_PATH = ROOT / '리스테리아ML' / '.env'                                # DB 접속 정보 (값은 출력하지 않는다)
OUT_DIR = ROOT / '리스테리아ML2' / 'data'
RAW_DIR = OUT_DIR / 'raw'      # DB 원자료
MID_DIR = OUT_DIR / '중간'      # 변수별 조인 결과
DATASET_PATH = OUT_DIR / '데이터셋.parquet'
COLDESC_PATH = OUT_DIR / '모든_컬럼설명.parquet'
FORCE_DB = False
FORCE_JOIN = False

# 조인 설정
LAG_MAX = 110                       # 기준일 당일(D-0) ~ 110일 전(D-110) → 변수마다 111칼럼
LAGS = np.arange(0, LAG_MAX + 1)
K_NEAREST = 3                       # 국내: 가까운 관측소 3곳 평균
TOP_CANDIDATES = 60                 # 계산을 빠르게 하려고 좌표마다 거리순 60곳을 먼저 본다. 60곳 안에 그날 값 있는 곳이 3곳보다 적으면 전체 관측소에서 다시 찾는다
ROW_GROUP = 50_000                  # parquet 를 이 행 수씩 나눠 쓴다 (메모리 때문에)
EARTH_R_KM = 6371.0088              # 구면 거리 계산에 쓰는 지구 반지름(km)

def haversine_km(lat1, lon1, lat2, lon2):
    """두 지점 사이의 구면 거리(km). 위도·경도는 도 단위"""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return EARTH_R_KM * 2 * np.arcsin(np.sqrt(a))

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 80)
for d in (RAW_DIR, MID_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('실행 시각:', datetime.now().strftime('%Y-%m-%d %H:%M'))
print('numpy', np.__version__, '| pandas', pd.__version__, '| pyarrow', pa.__version__, '| psycopg', psycopg.__version__)
print('결과 폴더:', OUT_DIR)

실행 시각: 2026-10-08 17:25
numpy 2.5.1 | pandas 3.0.3 | pyarrow 25.0.1 | psycopg 3.3.5
결과 폴더: C:\Users\NT751\Desktop\천영진\리스테리아ML2\data


## 2. 시료 파일 불러오기

- 시료 파일 23칼럼을 그대로 읽는다. 이 23칼럼은 `데이터셋.parquet` 에 모두 남긴다. 행을 지우거나 값을 바꾸지 않는다.
- `src_route` 는 주소에 한글이 있는지로 갈린 값이다(`ready_data.hazard_preprocessed_data.geo_route`, 정의서 "domestic(한글주소→법정동) / overseas(비한글→지오코딩)"). 그래서 한국 주소가 로마자로 적힌 행은 overseas 로 들어가 있다. 이 행들도 시료 파일 값 그대로 둔다(overseas 행으로 조인된다 — 5-2절에서 이 행들로 국내 자료와 해외용 자료를 견준다).

In [2]:
df = pd.read_parquet(BODY_PATH)
N_ROWS = len(df)
BODY_COLS = df.columns.tolist()
assert N_ROWS == 676_537, N_ROWS
assert df[KEY_COL].notna().all() and df[KEY_COL].is_unique, 'lims_data_key 에 결측·중복이 있음'
FILE_KEYS = set(df[KEY_COL])                              # 시료 파일의 행 키 전부 (3-3절에서 DB 와 견줄 때 쓴다)
print(f'시료 파일: {N_ROWS:,}행 × {len(BODY_COLS)}칼럼')
print('칼럼:', BODY_COLS)

is_ov = df['src_route'].astype(str).to_numpy() == 'overseas'
print(f"overseas {is_ov.sum():,}행 중 addr 에 한글이 든 행: {int(df['addr'].astype('string')[is_ov].str.contains('[가-힣]').sum())}")
del is_ov

시료 파일: 676,537행 × 23칼럼
칼럼: ['lims_data_key', 'hazard_label', 'judge_result', 'src_route', 'data_source', 'expr_iem_nm', 'result_value', 'result_value_num', 'judge_stdr', 'judge_stdr_num', 'inspct_instt_code', 'prdlst_code', 'train_prdlst_code', 'train_prdlst_nm', 'prduct_nm', 'mnfctur_cmpny_nm', 'mnfctur_de', 'date_basis', 'addr', 'addr_source', 'legal_dong_code', 'lat', 'lon']
overseas 47,642행 중 addr 에 한글이 든 행: 0


조인에 쓰는 값들을 정한다.

- **좌표**: 시료 파일의 `lat`·`lon` 원값(float32). 관측소 찾기·해외 좌표 맞추기 모두 이 값으로 한다.
- **기상 기간**: 가장 이른 기준일의 110일 전부터 가장 늦은 기준일 당일까지.

In [3]:
IS_DOM = (df['src_route'].astype(str) == 'domestic').to_numpy()
y = (df[TARGET_COL].astype(str) == '검출').to_numpy()
XY = df[['lat', 'lon']]
assert XY.notna().all().all() and (XY.dtypes == 'float32').all()

DATE_MIN = df[DATE_COL].min() - pd.Timedelta(days=LAG_MAX)   # 가장 이른 기준일의 D-110
DATE_MAX = df[DATE_COL].max()                                # 가장 늦은 기준일의 D-0
DATES = pd.date_range(DATE_MIN, DATE_MAX, freq='D')
N_DATES = len(DATES)

print(f'전체 {N_ROWS:,}행 · 검출 {int(y.sum()):,}행')
print(f'국내 {IS_DOM.sum():,}행(검출 {int((IS_DOM & y).sum()):,}) · 해외 {(~IS_DOM).sum():,}행(검출 {int((~IS_DOM & y).sum()):,})')
print('기준일:', df[DATE_COL].min().date(), '~', df[DATE_COL].max().date())
print('기상 기간:', DATE_MIN.date(), '~', DATE_MAX.date(), f'({N_DATES:,}일)')

def pct(m, mask=None):
    """참인 비율(%) — mask 를 주면 그 행들 안에서"""
    m = np.asarray(m, bool)
    return round(float((m if mask is None else m[mask]).mean()) * 100, 2)

전체 676,459행 · 검출 1,969행
국내 628,895행(검출 1,844) · 해외 47,564행(검출 125)
기준일: 2015-04-07 ~ 2026-06-30
기상 기간: 2014-12-18 ~ 2026-06-30 (4,213일)


### 2-1. 기준일과 좌표가 무엇의 날짜·무엇의 주소인가

기상·토양은 **기준일(`mnfctur_de`)** 과 **시료 파일 좌표(`lat`·`lon`)** 에 맞춰 붙인다. 이 둘이 무엇인지는 행마다 다르다.

- **기준일은 제조일만이 아니다.** 칼럼 이름은 `mnfctur_de` 이지만, 어떤 날짜인지는 `date_basis`(기준일 출처)에 있다. 실제 값은 아래 출력처럼 제조일·의뢰일·선적일이다. 테이블정의서의 `date_basis` 설명은 "(제조일 / 의뢰일 / 수거일)"로 적혀 있어 실제 값과 다르다(아래에 정의서 문구를 그대로 찍는다).
- **좌표는 제조 장소만이 아니고, 주소 자체의 위치도 아니다.** 국내 행의 `lat`·`lon` 은 `addr` 이 속한 법정동(읍면동·시군구·시도 단계)의 대표 지점 좌표다(3-3절에서 확인). 그 주소가 어디서 왔는지는 `addr_source`(주소 출처)에 있다. **`addr_source` 의 값 "업소주소"가 누구의 주소인지는 테이블정의서·DB 코멘트에 설명이 없다 — 확인 필요.**
- 시료 파일의 `mnfctur_de` 는 DB 의 `mnfctur_de_or_fallback`(정의서 "기준일. COALESCE(제조일, 의뢰일, 검사시작일)")과 같은 값이다(3-3절에서 확인).
- 그래서 4·5·7절에서 붙이는 값은 "제조일·제조 장소의 값"이 아니라 "**기준일 · 시료 주소가 속한 법정동 대표 지점**의 값"이다.

In [4]:
SPEC_PATH = ROOT / '테이블명세서(모델팀)' / '테이블정의서.xlsx'
spec = pd.read_excel(SPEC_PATH, sheet_name='ready_data', header=0)[['컬럼', '설명']].dropna(subset=['컬럼'])
spec['컬럼'] = spec['컬럼'].astype(str).str.strip()
SPEC = spec.drop_duplicates('컬럼').set_index('컬럼')['설명'].astype(str).str.strip()   # 테이블정의서 ready_data 시트의 칼럼 설명
for c in ('mnfctur_de', 'date_basis', 'addr', 'addr_source', 'mnfctur_cmpny_nm'):
    print(f'정의서 설명 — {c}: {SPEC.get(c, "(이 이름의 칼럼 없음)")}')

def share_table(s):
    """값별 행 수·검출 수와 전체 대비 비율"""
    v = pd.Series(s).astype(str).to_numpy()
    t = pd.DataFrame({'행': pd.Series(v).value_counts(), '검출': pd.Series(y).groupby(v).sum()})
    t['행(%)'] = (t['행'] / N_ROWS * 100).round(2)
    t['검출(%)'] = (t['검출'] / y.sum() * 100).round(2)
    return t.sort_values('행', ascending=False)

print()
print('기준일 출처 date_basis — 기준일이 무슨 날짜인가')
display(share_table(df['date_basis']))
print('주소 출처 addr_source — 좌표가 어느 주소의 위치인가')
display(share_table(df['addr_source']))
print('주소 출처 × 기준일 출처 — 행 수')
display(pd.crosstab(df['addr_source'].astype(str), df['date_basis'].astype(str), margins=True, margins_name='합'))
print('주소 출처 × 기준일 출처 — 검출 수')
display(pd.crosstab(df['addr_source'].astype(str), df['date_basis'].astype(str), values=y.astype(int), aggfunc='sum', margins=True, margins_name='합').fillna(0).astype(int))
up = (df['addr_source'].astype(str) == '업소주소').to_numpy()
req = (df['date_basis'].astype(str) == '의뢰일').to_numpy()
print(f"업소주소 행 {int(up.sum()):,} 중 mnfctur_cmpny_nm(제조회사명)이 비어 있는 행 {int(df.loc[up, 'mnfctur_cmpny_nm'].isna().sum()):,} ({pct(df['mnfctur_cmpny_nm'].isna(), up)}%)")
print(f"주소 출처가 업소주소이면서 기준일이 의뢰일인 행 {int((up & req).sum()):,} · 검출 {int((up & req & y).sum()):,} (전체 검출의 {(up & req & y).sum() / y.sum() * 100:.2f}%)")

정의서 설명 — mnfctur_de: 제조일(원본). 결측이 많아 기준일은 fallback 을 사용
정의서 설명 — date_basis: 기준일 출처 (제조일 / 의뢰일 / 수거일)
정의서 설명 — addr: 최종 확보 주소
정의서 설명 — addr_source: 주소 출처. 제조사소재지 → 인허가대장 → 해외제조업소 → 업소주소 → 검사기관 순. '검사기관'은 시험실 위치
정의서 설명 — mnfctur_cmpny_nm: 제조회사명. s_org 는 해외제조업소명, 농관원은 의뢰기관

기준일 출처 date_basis — 기준일이 무슨 날짜인가


,행,검출,행(%),검출(%)
의뢰일,333425,1249,49.29,63.43
제조일,295973,595,43.75,30.22
선적일,47061,125,6.96,6.35


주소 출처 addr_source — 좌표가 어느 주소의 위치인가


,행,검출,행(%),검출(%)
제조사소재지,327637,486,48.43,24.68
업소주소,292719,1348,43.27,68.46
해외제조업소,47429,125,7.01,6.35
인허가대장,8674,10,1.28,0.51


주소 출처 × 기준일 출처 — 행 수


date_basis,선적일,의뢰일,제조일,합
addr_source,,,,
업소주소,0,200664,92055,292719
인허가대장,0,2570,6104,8674
제조사소재지,0,129823,197814,327637
해외제조업소,47061,368,0,47429
합,47061,333425,295973,676459


주소 출처 × 기준일 출처 — 검출 수


date_basis,선적일,의뢰일,제조일,합
addr_source,,,,
업소주소,0,1008,340,1348
인허가대장,0,2,8,10
제조사소재지,0,239,247,486
해외제조업소,125,0,0,125
합,125,1249,595,1969


업소주소 행 292,719 중 mnfctur_cmpny_nm(제조회사명)이 비어 있는 행 289,121 (98.77%)
주소 출처가 업소주소이면서 기준일이 의뢰일인 행 200,664 · 검출 1,008 (전체 검출의 51.19%)


## 3. 식정원 DB 원자료 받기

- 접속: `리스테리아ML/.env` 의 `DB_HOST`·`DB_PORT`·`DB_NAME`·`DB_USER`·`DB_PASSWORD` (값은 출력하지 않는다). 세션을 **읽기 전용**(`default_transaction_read_only=on`)으로 열고 SELECT 만 한다. 서버에 TEMP 테이블 등 아무것도 만들지 않는다.
- 받은 그대로(값·단위 변환 없음) `raw/raw_<이름>.parquet` 로 저장하고, 조인은 이 파일로 한다. DB 의 numeric 형은 파일에 실수(float64)로 적힌다(값은 같음).

| 파일 | DB 테이블 | 무엇 | 받는 범위 |
|---|---|---|---|
| `raw_kma_asos_daily_info` | `api_data_info.kma_asos_daily_info` | 기상청 종관기상관측(ASOS) 일자료 | 전체 (`SELECT *`) |
| `raw_kma_ground_obs_station_info` | `api_data_info.kma_ground_obs_station_info` | ASOS 지점 정보(위치 이력 포함) | 전체 |
| `raw_nas_agri_weather_daily_info` | `api_data_info.nas_agri_weather_daily_info` | 농촌진흥청 농업기상 기본관측(NAS) 일자료 | 전체 |
| `raw_nas_agri_weather_obs_station` | `api_data_info.nas_agri_weather_obs_station` | NAS 관측지점 정보 | 전체 |
| `raw_keco_airkorea_fine_dust_info` | `api_data_info.keco_airkorea_fine_dust_info` | 에어코리아 대기오염 일평균 | 전체 |
| `raw_keco_airkorea_obs_station` | `api_data_info.keco_airkorea_obs_station` | 에어코리아 측정소 정보 | 전체 |
| `raw_global_factory_weather_listeria` | `api_data_info.global_factory_weather` | 해외 제조업소 좌표 일별 기상 (DB 코멘트 "Open-Meteo Archive(ERA5) 해외 제조업소 좌표 일별 기상") | 해외 시료 좌표 × 기상 기간만 (3-1절) |
| `raw_listeria_m_org` | `ready_data.hazard_preprocessed_data` + `ba_data.m_org_lims_data_info` | 리스테리아 행의 품목제조번호, 시료 파일과 견줄 칼럼(판정·기준일·주소 출처·좌표·좌표 매칭 수준, `db_` 로 시작하는 이름으로 받음 — 3-3절), 식약처 LIMS 원본의 검사 목적·시료 상태·유통 일 칼럼 3개 | `hazard_label = '리스테리아'` 행, 필요한 칼럼만 (8절) |
| `raw_t_cm_cmmn_code_IM02_IM11` | `ba_data.t_cm_cmmn_code` | 공통코드 이름 (검사 목적 IM02 · 시료 상태 IM11) | 두 분류만 |
| `raw_t_prdlst_mnf_ledg_listeria` | `ba_data.t_prdlst_mnf_ledg` | 품목제조대장 | 시료의 품목제조번호에 해당하는 행만 (3-2절) |
| `raw_t_aw_cl` | `ba_data.t_aw_cl` | 품목코드별 수분활성도·식품pH·염도 | 전체 (`SELECT *`) |
| `raw_soil_ph_info` | `api_data_info.soil_ph_info` | 토양검정 결과 (DB 테이블 코멘트 "토양산도 정보") | 전체 (`SELECT *`) |
| `raw_korea_region_code` | `ba_data_info.korea_region_code` | 법정동 코드별 좌표 (DB 테이블 코멘트 "법정동_코드") | 전체 (`SELECT *`) |

DB 밖 자료는 리서치팀 엑셀 `리서치팀문서/통합_수분활성도_수분함량_식품pH_염도_최종.xlsx` 하나다(6절, 파일을 그대로 읽는다).

In [5]:
def connect():
    """읽기 전용 세션으로 접속 — .env 의 DB_* 키만 읽고 값은 출력하지 않는다"""
    e = dotenv_values(ENV_PATH)
    conn = psycopg.connect(host=e['DB_HOST'], port=int(e.get('DB_PORT') or 5432), dbname=e['DB_NAME'],
                           user=e['DB_USER'], password=e.get('DB_PASSWORD') or None,
                           options='-c default_transaction_read_only=on',
                           application_name='listeria_ml2_dataset_build', connect_timeout=15)
    with conn.cursor() as cur:
        cur.execute('show default_transaction_read_only')
        assert cur.fetchone()[0] == 'on', '읽기 전용 세션이 아님'
    return conn


PG_TYPES = {16: pa.bool_(), 20: pa.int64(), 21: pa.int64(), 23: pa.int64(), 700: pa.float64(), 701: pa.float64(), 1700: pa.float64(),
            1082: pa.date32(), 1114: pa.timestamp('us'), 25: pa.large_string(), 1042: pa.large_string(), 1043: pa.large_string()}   # PostgreSQL 자료형 번호 → parquet 자료형

def fetch_to_parquet(conn, sql, params, out_path, chunk=500_000):
    """서버측 커서로 나눠 받아 parquet 로 저장. DB 의 numeric(Decimal)은 float64 로만 바꾸고 값은 그대로 둔다"""
    out_path = Path(out_path)
    tmp = out_path.with_suffix('.parquet.part')
    writer, schema, n = None, None, 0
    with conn.transaction():
        with conn.cursor(name='ml2_fetch') as cur:
            cur.itersize = chunk
            cur.execute(sql, params)
            cols = [d.name for d in cur.description]
            while rows := cur.fetchmany(chunk):
                part = pd.DataFrame(rows, columns=cols)
                for c in part.columns:
                    nn = part[c].dropna()
                    if part[c].dtype == object and len(nn) and isinstance(nn.iloc[0], decimal.Decimal):
                        part[c] = pd.to_numeric(part[c]).astype('float64')
                table = pa.Table.from_pandas(part, preserve_index=False)
                if writer is None:
                    # 첫 묶음에서 값이 전부 빈 칼럼은 자료형을 알 수 없으므로 DB 자료형으로 정한다
                    schema = pa.schema([pa.field(f.name, PG_TYPES.get(d.type_code, pa.string())) if pa.types.is_null(f.type) else pa.field(f.name, f.type)
                                        for f, d in zip(table.schema, cur.description)])
                    writer = pq.ParquetWriter(tmp, schema)
                writer.write_table(table.cast(schema))
                n += len(part)
    writer.close()
    os.replace(tmp, out_path)
    return n


def raw_info(name):
    p = RAW_DIR / f'{name}.parquet'
    return f'{name:38s} {pq.ParquetFile(p).metadata.num_rows:>11,}행  받은 시각 {datetime.fromtimestamp(p.stat().st_mtime):%Y-%m-%d %H:%M}'


RAW_TABLES = {   # 파일 이름 → (SQL, 한 번에 받는 행 수)
    'raw_kma_asos_daily_info':          ('select * from api_data_info.kma_asos_daily_info', 500_000),
    'raw_kma_ground_obs_station_info':  ('select * from api_data_info.kma_ground_obs_station_info', 500_000),
    'raw_nas_agri_weather_daily_info':  ('select * from api_data_info.nas_agri_weather_daily_info', 500_000),
    'raw_nas_agri_weather_obs_station': ('select * from api_data_info.nas_agri_weather_obs_station', 500_000),
    'raw_keco_airkorea_fine_dust_info': ('select * from api_data_info.keco_airkorea_fine_dust_info', 500_000),
    'raw_keco_airkorea_obs_station':    ('select * from api_data_info.keco_airkorea_obs_station', 500_000),
    # 리스테리아 행 + 식약처 LIMS 원본. 값이 드문 칼럼이 있어 한 번에 받는다(나눠 받으면 첫 묶음에 값이 없을 때 자료형을 못 정함)
    'raw_listeria_m_org': ("""
        select h.lims_data_key,
               h.prdlst_mnfctur_sttemnt_no,
               h.judge_result as db_judge_result, h.mnfctur_de as db_mnfctur_de, h.mnfctur_de_or_fallback as db_mnfctur_de_or_fallback,
               h.date_basis as db_date_basis, h.addr_source as db_addr_source, h.train_prdlst_code as db_train_prdlst_code,
               h.lat as db_lat, h.lon as db_lon, h.geo_match_level, h.geocode_status,
               (m.lims_data_key is not null) as has_m_org,
               m.inspct_purps_code, m.splore_sttus_code, m.distb_de
        from ready_data.hazard_preprocessed_data h
        left join ba_data.m_org_lims_data_info m on m.lims_data_key = h.lims_data_key
        where h.hazard_label = '리스테리아'""", 2_000_000),
    'raw_t_cm_cmmn_code_IM02_IM11':     ("select * from ba_data.t_cm_cmmn_code where cmmn_cl_code in ('IM02', 'IM11')", 500_000),
    'raw_t_aw_cl':                      ('select * from ba_data.t_aw_cl', 500_000),
    'raw_korea_region_code':            ('select * from ba_data_info.korea_region_code', 500_000),
    'raw_soil_ph_info':                 ('select * from api_data_info.soil_ph_info', 500_000),
}
todo = [f for f in RAW_TABLES if FORCE_DB or not (RAW_DIR / f'{f}.parquet').exists()]
if todo:
    with connect() as conn:
        for f in todo:
            t0 = time.time()
            sql, chunk = RAW_TABLES[f]
            n = fetch_to_parquet(conn, sql, None, RAW_DIR / f'{f}.parquet', chunk=chunk)
            print(f'받음 {f}: {n:,}행 ({time.time() - t0:.0f}초)')
for f in RAW_TABLES:
    print(raw_info(f))

raw_kma_asos_daily_info                    406,909행  받은 시각 2026-10-06 14:44
raw_kma_ground_obs_station_info                243행  받은 시각 2026-10-06 14:44
raw_nas_agri_weather_daily_info            984,648행  받은 시각 2026-10-06 14:44
raw_nas_agri_weather_obs_station               219행  받은 시각 2026-10-06 14:44
raw_keco_airkorea_fine_dust_info         2,150,630행  받은 시각 2026-10-06 14:45
raw_keco_airkorea_obs_station                  753행  받은 시각 2026-10-06 14:45
raw_listeria_m_org                         765,891행  받은 시각 2026-10-06 17:22
raw_t_cm_cmmn_code_IM02_IM11                    22행  받은 시각 2026-10-06 14:45
raw_t_aw_cl                                  1,316행  받은 시각 2026-10-06 14:54
raw_korea_region_code                       51,884행  받은 시각 2026-10-06 15:07
raw_soil_ph_info                         6,280,656행  받은 시각 2026-10-06 15:09


### 3-1. 해외 제조업소 기상 `api_data_info.global_factory_weather` — 해외 시료 좌표만 받기

DB 코멘트상 이 테이블의 좌표 `lat_g`·`lon_g` 는 "hazard_preprocessed_data 원본 lat 과 동일 값(조인 키)"이다. 그런데 시료 파일의 `lat`·`lon` 은 소수 정밀도가 낮은 float32 로 저장돼 있어 그대로 비교하면 하나도 같지 않다. 그래서 두 단계로 받는다(`3_기상조인_후_학습.ipynb` 3-1절과 같은 방법).

1. **좌표 찾기**: 시료 좌표(float32)마다 "float32 로 바꾸면 그 값이 되는 DB 좌표(double) 범위"를 계산해, 그 범위 안의 DB 좌표를 찾는다.
2. **자료 받기**: 찾은 DB 좌표의 기상 기간 일자료를 `SELECT *` 로 받는다.

이미 받은 파일이 있어도 그 파일에 없는 해외 시료 좌표가 있으면 다시 받는다(해외 시료 좌표가 늘어난 경우).

같은 시료 좌표로 반올림되는 DB 좌표가 2곳 이상이면 둘 다 받고, 4절에서 두 값의 평균을 쓴다(몇 곳인지 4-4절에 출력).

In [6]:
def f32_bounds(v):
    """float32 값 v 로 반올림되는 double 범위 [lo, hi]"""
    v = np.asarray(v, dtype=np.float32)
    up = np.nextafter(v, np.float32(np.inf)).astype(np.float64)
    dn = np.nextafter(v, np.float32(-np.inf)).astype(np.float64)
    v64 = v.astype(np.float64)
    return (v64 + dn) / 2, (v64 + up) / 2

GFW_NAME = 'raw_global_factory_weather_listeria'
GFW_PATH = RAW_DIR / f'{GFW_NAME}.parquet'
OV_XY = XY.loc[~IS_DOM, ['lat', 'lon']].drop_duplicates().reset_index(drop=True)   # 해외 시료 좌표(float32)

SQL_FIND = """
select distinct g.lat_g, g.lon_g
from unnest(%s::float8[], %s::float8[], %s::float8[], %s::float8[]) as b(la_lo, la_hi, lo_lo, lo_hi)
cross join lateral (
    select distinct lat_g, lon_g from api_data_info.global_factory_weather
    where lat_g between b.la_lo and b.la_hi and lon_g between b.lo_lo and b.lo_hi
) g
"""
SQL_DATA = """
select g.*
from unnest(%s::float8[], %s::float8[]) as c(lat, lon)
cross join lateral (
    select * from api_data_info.global_factory_weather
    where lat_g = c.lat and lon_g = c.lon and obsr_de between %s and %s
) g
"""
def gfw_missing():
    """해외 시료 좌표 중 받아 둔 파일에 없는 좌표 수 (DB 좌표를 float32 로 바꿔 견준다)"""
    t = pq.read_table(GFW_PATH, columns=['lat_g', 'lon_g'])
    have = pd.DataFrame({'lat': t.column('lat_g').to_numpy().astype(np.float32), 'lon': t.column('lon_g').to_numpy().astype(np.float32)}).drop_duplicates()
    return int((pd.MultiIndex.from_frame(have).get_indexer(pd.MultiIndex.from_frame(OV_XY)) < 0).sum())

n_miss = gfw_missing() if GFW_PATH.exists() else len(OV_XY)
if GFW_PATH.exists():
    print(f'받아 둔 파일에 없는 해외 시료 좌표 {n_miss}곳' + (' → 다시 받는다' if n_miss else ''))
if FORCE_DB or n_miss:
    la_lo, la_hi = f32_bounds(OV_XY['lat'])
    lo_lo, lo_hi = f32_bounds(OV_XY['lon'])
    with connect() as conn:
        t0 = time.time()
        with conn.cursor() as cur:
            cur.execute(SQL_FIND, (la_lo.tolist(), la_hi.tolist(), lo_lo.tolist(), lo_hi.tolist()))
            grid = pd.DataFrame(cur.fetchall(), columns=['lat_g', 'lon_g'])
        print(f'1단계 좌표 찾기: 시료 좌표 {len(OV_XY):,}곳 → DB 좌표 {len(grid):,}곳 ({time.time() - t0:.0f}초)')
        t0 = time.time()
        n = fetch_to_parquet(conn, SQL_DATA, (grid['lat_g'].tolist(), grid['lon_g'].tolist(),
                                              DATE_MIN.date(), DATE_MAX.date()), GFW_PATH)
        print(f'2단계 자료 받기: {n:,}행 ({time.time() - t0:.0f}초)')
print(f'해외 시료 좌표 {len(OV_XY):,}곳')
print(raw_info(GFW_NAME))

해외 시료 좌표 3,062곳
raw_global_factory_weather_listeria     12,865,736행  받은 시각 2026-10-06 14:47


### 3-2. 품목제조대장 `ba_data.t_prdlst_mnf_ledg` — 시료의 품목제조번호에 해당하는 행만 받기

시료의 품목제조번호(`ready_data.hazard_preprocessed_data.prdlst_mnfctur_sttemnt_no`, 3절에서 받은 `raw_listeria_m_org`)와 `prdlst_report_no`(DB 코멘트 "품목 보고 번호")가 **원문 그대로 같은** 대장 행을 `SELECT *` 로 받는다. 번호에서 글자를 지우는 등의 변환은 하지 않는다.

In [7]:
LEDG_NAME = 'raw_t_prdlst_mnf_ledg_listeria'
LEDG_PATH = RAW_DIR / f'{LEDG_NAME}.parquet'
m_org_keys = pd.read_parquet(RAW_DIR / 'raw_listeria_m_org.parquet', columns=[KEY_COL, 'prdlst_mnfctur_sttemnt_no'])
m_org_keys = m_org_keys[m_org_keys[KEY_COL].isin(set(df[KEY_COL]))]
no = m_org_keys['prdlst_mnfctur_sttemnt_no']
REPORT_NOS = sorted(no[no.notna() & (no.astype(str).str.strip() != '')].unique())
print(f'시료 파일 행 중 품목제조번호가 있는 행 {int((no.notna() & (no.astype(str).str.strip() != "")).sum()):,} · 고유 번호 {len(REPORT_NOS):,}개')
if FORCE_DB or not LEDG_PATH.exists():
    with connect() as conn:
        t0 = time.time()
        n = fetch_to_parquet(conn, 'select * from ba_data.t_prdlst_mnf_ledg where prdlst_report_no = any(%s)',
                             (REPORT_NOS,), LEDG_PATH, chunk=2_000_000)
        print(f'받음 {LEDG_NAME}: {n:,}행 ({time.time() - t0:.0f}초)')
print(raw_info(LEDG_NAME))
del m_org_keys, no

시료 파일 행 중 품목제조번호가 있는 행 204,115 · 고유 번호 35,504개
raw_t_prdlst_mnf_ledg_listeria              35,808행  받은 시각 2026-10-06 14:47


### 3-3. 시료 파일은 무엇을 담고 있나 — 지금 DB 와의 차이, 기준일, 좌표

조인을 시작하기 전에 시료 파일(`listeria_body.parquet`)의 범위와, 기준일·좌표가 무엇인지를 DB 값으로 확인해 둔다. `raw_listeria_m_org` 에 함께 받은 `ready_data.hazard_preprocessed_data` 의 칼럼(`db_` 로 시작하는 이름으로 받음)을 쓴다.

- **범위와 시점**: 시료 파일은 모델팀이 내보낸 표 `staging_data.export_body_listeria` 와 같은 내용이다(작업기록 2026-09-22 확인). 지금 DB 의 리스테리아 행보다 적고, 받은 날도 다르다. 무엇이 빠졌는지와 그 사이 DB 에서 바뀐 값을 아래에 적는다. **이 노트북은 시료 파일의 값을 그대로 쓴다**(DB 의 지금 값으로 바꾸지 않는다).
- **기준일**: 시료 파일 `mnfctur_de` 가 DB 의 어느 칼럼과 같은지 본다.
- **좌표**: 시료 파일 `lat`·`lon` 이 주소 자체의 위치인지 본다.

In [8]:
h = pd.read_parquet(RAW_DIR / 'raw_listeria_m_org.parquet', columns=[KEY_COL, 'db_judge_result', 'db_mnfctur_de', 'db_mnfctur_de_or_fallback', 'db_date_basis',
                                                                   'db_addr_source', 'db_train_prdlst_code', 'db_lat', 'db_lon', 'geo_match_level', 'geocode_status'])
in_file = h[KEY_COL].isin(FILE_KEYS).to_numpy()            # 시료 파일에 있는 행
db_pos = (h['db_judge_result'] == '검출').to_numpy()
print(f"지금 DB(ready_data.hazard_preprocessed_data 의 hazard_label = '리스테리아') {len(h):,}행 · 검출 {int(db_pos.sum()):,}"
      f" (받은 시각 {datetime.fromtimestamp((RAW_DIR / 'raw_listeria_m_org.parquet').stat().st_mtime):%Y-%m-%d %H:%M})")
print(f"시료 파일 {N_ROWS:,}행 · 검출 {int(y.sum()):,} (파일 수정 시각 {datetime.fromtimestamp(BODY_PATH.stat().st_mtime):%Y-%m-%d %H:%M})"
      f" · 시료 파일의 행 키가 모두 DB 에 있는가: {bool(df[KEY_COL].isin(set(h[KEY_COL])).all())}")
out = h[~in_file]
print(f"DB 에만 있고 시료 파일에는 없는 행 {len(out):,} · 그중 검출 {int((out['db_judge_result'] == '검출').sum()):,} — 주소 출처(DB addr_source)별:")
display(pd.DataFrame({'주소 출처': out['db_addr_source'].fillna('(빈 값)').to_numpy(), '검출': (out['db_judge_result'] == '검출').to_numpy(), '좌표 없음': out['db_lat'].isna().to_numpy()})
        .groupby('주소 출처').agg(행=('검출', 'size'), 검출=('검출', 'sum'), 좌표_없음=('좌표 없음', 'sum')).sort_values('행', ascending=False))

hm = h.set_index(KEY_COL).reindex(df[KEY_COL]).reset_index()          # 시료 파일과 같은 행 순서
def to_day(s):
    return pd.to_datetime(s).to_numpy().astype('datetime64[D]')
def diff_mask(a, b):
    """두 열이 다른 행 (둘 다 빈 값이면 같다고 본다)"""
    a, b = pd.Series(a).astype(object).reset_index(drop=True), pd.Series(b).astype(object).reset_index(drop=True)
    return ((a.isna() != b.isna()) | (a.notna() & b.notna() & (a.astype(str) != b.astype(str)))).to_numpy()

base_day = df[DATE_COL].to_numpy().astype('datetime64[D]')
print()
print('기준일 — 시료 파일 mnfctur_de 가 DB 의 어느 칼럼과 같은가')
print(f"  DB mnfctur_de_or_fallback 과 같은 행 {int((to_day(hm['db_mnfctur_de_or_fallback']) == base_day).sum()):,} / {N_ROWS:,} · 정의서 설명: {SPEC.get('mnfctur_de_or_fallback')}")
print(f"  DB mnfctur_de 와 같은 행 {int((to_day(hm['db_mnfctur_de']) == base_day).sum()):,} · DB mnfctur_de 가 비어 있는 행 {int(hm['db_mnfctur_de'].isna().sum()):,} · 정의서 설명: {SPEC.get('mnfctur_de')}")

print()
print('시료 파일 값과 지금 DB 값이 다른 행 수 (같은 행 키끼리)')
for fc, dc in (('judge_result', 'db_judge_result'), ('date_basis', 'db_date_basis'), ('train_prdlst_code', 'db_train_prdlst_code'), ('addr_source', 'db_addr_source')):
    dm = diff_mask(df[fc], hm[dc])
    print(f'  {fc}: {int(dm.sum()):,}행 (그중 검출 {int((dm & y).sum())})')
xy_diff = ~((hm['db_lat'].astype('float32').to_numpy() == XY['lat'].to_numpy()) & (hm['db_lon'].astype('float32').to_numpy() == XY['lon'].to_numpy()))
print(f'  lat·lon (DB 값을 float32 로 맞춰 비교): {int(xy_diff.sum()):,}행')
dm = diff_mask(df['train_prdlst_code'], hm['db_train_prdlst_code'])
print('  train_prdlst_code 가 다른 행에서 많이 나오는 짝 (이 칼럼은 6절 식품 조인의 둘째 키다):')
display(pd.DataFrame({'시료 파일': df['train_prdlst_code'].astype(object).to_numpy()[dm], '지금 DB': hm['db_train_prdlst_code'].astype(object).to_numpy()[dm], '검출': y[dm]})
        .groupby(['시료 파일', '지금 DB'], dropna=False).agg(행=('검출', 'size'), 검출=('검출', 'sum')).sort_values('행', ascending=False).head(6))
dm = diff_mask(df['addr_source'], hm['db_addr_source'])
print('  addr_source 가 다른 행의 짝:', pd.DataFrame({'a': df['addr_source'].astype(object).to_numpy()[dm], 'b': hm['db_addr_source'].astype(object).to_numpy()[dm]}).value_counts().to_dict())

print()
print('좌표 — 주소 자체의 위치인가')
dom0 = IS_DOM                                                         # 국내 행
n_addr_per_xy = df.loc[dom0].groupby(['lat', 'lon'])['addr'].nunique()
print(f"  국내 행 {int(dom0.sum()):,} · 서로 다른 주소 {df.loc[dom0, 'addr'].nunique():,}개 · 서로 다른 좌표 {len(n_addr_per_xy):,}곳"
      f" · 좌표 한 곳에 든 서로 다른 주소 수: 중앙값 {n_addr_per_xy.median():.0f} · 최대 {int(n_addr_per_xy.max())}")
krc0 = pd.read_parquet(RAW_DIR / 'raw_korea_region_code.parquet', columns=['legal_dong_code', 'legal_dong_nm', 'level', 'kw_query', 'lat', 'lon']).rename(columns={'lat': 'k_lat', 'lon': 'k_lon'})
m = df.loc[dom0, ['legal_dong_code', 'lat', 'lon']].astype({'legal_dong_code': str}).merge(krc0, on='legal_dong_code', how='left')
same_pt = ((m['k_lat'].astype('float32') == m['lat']) & (m['k_lon'].astype('float32') == m['lon'])).to_numpy()
print(f"  좌표가 ba_data_info.korea_region_code 의 그 법정동 좌표(lat·lon)와 같은 행 {int(same_pt.sum()):,} ({same_pt.mean() * 100:.2f}%)"
      f" · 그 법정동의 level: {m.loc[same_pt, 'level'].value_counts().to_dict()}")
print('  korea_region_code.kw_query 값의 예:', m.loc[same_pt, 'kw_query'].dropna().drop_duplicates().head(4).tolist())
print(f"  DB geo_match_level (정의서: {SPEC.get('geo_match_level')}) — 국내 행:", hm.loc[dom0, 'geo_match_level'].fillna('(빈 값)').value_counts().to_dict())

sgg = df.loc[dom0, 'legal_dong_code'].astype(str).str[:5].to_numpy()
med = df.loc[dom0, ['lat', 'lon']].astype(float).groupby(sgg).transform('median')
off_km = haversine_km(df.loc[dom0, 'lat'].to_numpy(float), df.loc[dom0, 'lon'].to_numpy(float), med['lat'].to_numpy(), med['lon'].to_numpy())
far = off_km > 50
print(f'  같은 시군구(법정동 코드 앞 5자리) 행들의 좌표 중앙값에서 50km 넘게 떨어진 국내 행: {int(far.sum())}행 (고치지 않고 그대로 둔다)')
# 그 좌표가 korea_region_code 에서 어느 법정동의 좌표인지 함께 본다 — 주소와 다른 지역의 좌표가 찍힌 행을 가려 보려고
kk = krc0.dropna(subset=['k_lat', 'k_lon'])
name_of_xy = pd.Series(kk['legal_dong_nm'].to_numpy(), index=pd.MultiIndex.from_arrays([kk['k_lat'].astype('float32'), kk['k_lon'].astype('float32')]))
name_of_xy = name_of_xy[~name_of_xy.index.duplicated()]
far_xy = pd.MultiIndex.from_arrays([df.loc[dom0, 'lat'].to_numpy()[far], df.loc[dom0, 'lon'].to_numpy()[far]])
display(pd.DataFrame({'addr': df.loc[dom0, 'addr'].str.slice(0, 40).to_numpy()[far], '좌표가 같은 법정동 (korea_region_code.legal_dong_nm)': name_of_xy.reindex(far_xy).fillna('(같은 좌표 없음)').to_numpy(),
                      '시군구 좌표 중앙값까지(km)': off_km[far].round(0)}).value_counts().rename('행').reset_index().head(12))

print('해외 행의 DB geocode_status (정의서: ' + str(SPEC.get('geocode_status')) + ')')
display(pd.DataFrame({'geocode_status': hm.loc[~IS_DOM, 'geocode_status'].fillna('(빈 값)').to_numpy(), '검출': y[~IS_DOM]})
        .groupby('geocode_status').agg(행=('검출', 'size'), 검출=('검출', 'sum')))
del h, hm, out, m, krc0, med

지금 DB(ready_data.hazard_preprocessed_data 의 hazard_label = '리스테리아') 765,891행 · 검출 2,062 (받은 시각 2026-10-06 17:22)


시료 파일 676,537행 → 78행을 지운 뒤 676,459행 · 검출 1,969 (파일 수정 시각 2026-09-28 10:26) · 시료 파일의 행 키가 모두 DB 에 있는가: True
DB 에만 있고 시료 파일에는 없는 행 89,354 · 그중 검출 93 — 주소 출처(DB addr_source)별:


,행,검출,좌표_없음
주소 출처,,,
검사기관,86696,91,0
제조사소재지,1473,0,1473
해외제조업소,1142,2,1142
업소주소,43,0,42



기준일 — 시료 파일 mnfctur_de 가 DB 의 어느 칼럼과 같은가
  DB mnfctur_de_or_fallback 과 같은 행 676,459 / 676,459 · 정의서 설명: 기준일. COALESCE(제조일, 의뢰일, 검사시작일)
  DB mnfctur_de 와 같은 행 295,973 · DB mnfctur_de 가 비어 있는 행 380,486 · 정의서 설명: 제조일(원본). 결측이 많아 기준일은 fallback 을 사용

시료 파일 값과 지금 DB 값이 다른 행 수 (같은 행 키끼리)


  judge_result: 0행 (그중 검출 0)
  date_basis: 0행 (그중 검출 0)


  train_prdlst_code: 9,236행 (그중 검출 6)
  addr_source: 688행 (그중 검출 0)
  lat·lon (DB 값을 float32 로 맞춰 비교): 296행


... (4 outputs left out)

  같은 시군구(법정동 코드 앞 5자리) 행들의 좌표 중앙값에서 50km 넘게 떨어진 국내 행: 44행 (고치지 않고 그대로 둔다)


,addr,좌표가 같은 법정동 (korea_region_code.legal_dong_nm),시군구 좌표 중앙값까지(km),행
0,울산광역시 북구 신천로 57(지상1층 매곡동),전남광주통합특별시 북구 매곡동,228.0,10
1,울산광역시 북구 매곡1로 14(지상1층 매곡동),전남광주통합특별시 북구 매곡동,228.0,10
2,제주특별자치도 제주시 추자면 예초리 308-1,제주특별자치도 제주시 추자면,56.0,5
3,울산광역시 북구 매곡본길 3(지상1층 매곡동),전남광주통합특별시 북구 매곡동,228.0,4
4,인천광역시 옹진군 백령면 백령도 236 롯데리아 백령점,인천광역시 옹진군 백령면,175.0,3
5,울산광역시 북구 매곡로 93(지상1층 매곡동),전남광주통합특별시 북구 매곡동,228.0,3
6,울산광역시 북구 신천로 57(지상1층 매곡동)(더베스트푸드식자재마트),전남광주통합특별시 북구 매곡동,228.0,3
7,울산광역시 북구 매곡1로 49(매곡동),전남광주통합특별시 북구 매곡동,228.0,2
8,제주특별자치도 제주시 추자면 추자로 556,제주특별자치도 제주시 추자면,56.0,2
9,제주특별자치도 제주시 추자면 추자로 2,제주특별자치도 제주시 추자면,56.0,1


해외 행의 DB geocode_status (정의서: 해외 지오코딩 상태. ok / low_precision / manual / failed)


,행,검출
geocode_status,,
low_precision,17936,50
ok,29628,75


위 출력으로 확인되는 것:

- **시료 파일은 지금 DB 의 리스테리아 행 전체가 아니다.** DB 에만 있는 행은 대부분 주소 출처가 '검사기관'인 행이다(내보내는 DB 함수 `staging_data.build_export_qual` 이 이 행들을 뺀다 — 작업기록 2026-09-22). 시료 파일을 받은 뒤 DB 에서 `train_prdlst_code` 등이 바뀐 행이 있다.
- **시료 파일의 `mnfctur_de` 는 DB 의 `mnfctur_de_or_fallback`(기준일)이다.** 이름이 같은 DB 칼럼 `mnfctur_de`(제조일 원본)가 아니다.
- **국내 행의 좌표는 주소 자체의 위치가 아니라, 그 주소가 속한 법정동(읍면동·시군구·시도 단계)의 대표 지점이다.** 서로 다른 주소 여러 개가 같은 좌표를 쓴다. `ba_data_info.korea_region_code` 의 `lat`·`lon` 이 법정동의 어느 지점인지는 DB 코멘트가 없다(`kw_query` 값은 주민센터·구청·시청 이름이다) — 확인 필요.
- 좌표가 같은 시군구의 다른 행들과 50km 넘게 떨어진 행이 있다. 섬 지역처럼 실제로 먼 곳도 있고, **주소와 다른 지역의 법정동 좌표가 찍힌 행**도 있다(표에서 주소의 시·도와 "좌표가 같은 법정동"의 시·도가 다른 줄). 고치지 않았다.
- 해외 행은 DB `geocode_status` 가 `low_precision` 인 행이 있다. 이 값의 뜻은 정의서에 값 이름뿐이다 — 확인 필요.

## 4. 조인 — 기상·대기 (17개 변수 × D-0~D-110)

방법은 `리스테리아ML/EXP/1/3_기상조인_후_학습.ipynb` 4절과 같다. 달라진 점은 표 아래에 적는다.

### 조인 규칙

| 규칙 | 내용 |
|---|---|
| 날짜 | 기준일 = 시료 파일의 `mnfctur_de`(제조일만이 아니다 — 2-1절). 기준일 당일(D-0)부터 110일 전(D-110)까지 하루씩. 칼럼 이름은 `<변수>_d000`(D-0) … `<변수>_d110`(D-110). D-0 값은 기준일이 끝나야 알 수 있는 값이다(아래 "규칙을 정한 근거") |
| 국내 행 | 시료 파일의 `lat`·`lon`(시료 주소가 속한 법정동의 대표 지점 좌표 — 제조 장소도, 주소 자체의 위치도 아니다, 2-1·3-3절)에서 **실제 거리(구면 거리, km)** 로 가까운 관측소 3곳의 **평균**. 관측소는 **그날 그 변수 값이 있는 곳 중에서** 고른다(3곳이 안 되면 있는 곳만 평균). 거리 상한 없음 |
| 쓰지 않는 관측소 | NAS '동방로거테스트'(`561034A001`)는 쓰지 않는다 (사용자 결정 2026-10-06, 이유는 4-2절) |
| 관측소 좌표 | ASOS 는 지점 정보에 위치 이력(운영 시작일 `oper_start_dt`)이 있어 **그날 운영 중이던 위치**를 쓴다. NAS·에어코리아는 좌표가 하나뿐이라 그 좌표 |
| 대체 칼럼 | 국내 기본 칼럼이 그날 비면(그날 값 있는 관측소가 **전국에 한 곳도 없으면**) 대체 칼럼을 같은 방식으로 붙인다. 그래도 없으면 결측 그대로. 이번 데이터셋에서는 이 조건이 한 번도 생기지 않아 대체 칼럼 값이 들어간 칸은 0이다(4-5절) |
| 해외 행 | `api_data_info.global_factory_weather` 에서 시료와 같은 좌표(3-1절)의 그날 값. 날짜 `obsr_de` 는 DB 코멘트 "관측 일자(UTC 기준)"라서 현지 날짜와 어긋날 수 있다(5-2절) |
| 국내·해외 | 같은 칼럼에 넣는다. 해외 자료가 없는 변수는 해외 행이 결측 |
| 값 | 결측 채움·이상값 처리 없음. 단위 변환은 아래 두 가지(해외 풍속·일조)만 한다. ASOS `sum_sshr` 는 DB 에 텍스트로 들어 있어 숫자로만 바꾼다 |

### `3_기상조인_후_학습.ipynb` 와 달라진 점 (사용자 지시 2026-10-06)

| 항목 | 그때 | 이번 |
|---|---|---|
| 날짜 | D-1~D-100 (100칼럼) | D-0~D-110 (111칼럼) |
| 해외 풍속 | 단위가 달라 뺌 | `wind_speed_mean`·`wind_speed_max`(DB 코멘트 "일평균/일최대 지상 10m 풍속(km/h)")를 **3.6 으로 나눠 m/s** 로 맞춰 넣는다 |
| 해외 일조 | 단위가 달라 뺌 | `sunshine_duration`(DB 코멘트 "일조시간(초). 구름에 가리지 않은 실제 일조")을 **3600 으로 나눠 시간**으로 맞춰 넣는다 |
| 토양수분 | 국내만 | 그대로 국내만 — NAS `soil_wt`(DB 코멘트 "토양수분보정값(%)"). 해외 `soil_moisture_mean`(m³/m³)은 넣지 않는다 |
| 풍향 | 뺌 | 5절에서 sin·cos 로 바꿔 넣는다 |
| 입력 | `X.parquet` | 시료 파일 `listeria_body.parquet` |
| 거리 | 위경도 차이를 그대로 쓴 유클리드 거리 √((위도 차)² + (경도 차)²) | **실제 거리(구면 거리, km)**. 위경도 차이를 그대로 쓰면 위도 1도(약 111km)와 경도 1도(북위 37도에서 약 89km)를 같은 길이로 보게 되어 동서쪽 관측소가 실제보다 멀게 계산된다. 두 방식에서 가까운 3곳이 달라지는 좌표 수는 4-3절에 적는다 |
| 동방로거테스트 | 포함 | **제외** (4-2절) |

### 규칙을 정한 근거 — 적혀 있는 것과 적혀 있지 않은 것

| 규칙 | 근거 |
|---|---|
| 가까운 관측소 3곳의 단순 평균 | `3_기상조인_후_학습.ipynb` 의 규칙을 그대로 따랐다. 왜 3곳인지, 왜 거리 가중이 아닌 단순 평균인지는 그 노트북에도 적혀 있지 않다. 가까운 곳과 셋째 곳의 거리 차이는 4-5절에 적는다 |
| 거리 상한 없음 | 그날 값 있는 관측소 중에서 고르므로, 값을 재는 관측소가 적은 변수(지중온도 등)는 먼 관측소가 뽑힌다. 상한을 두지 않은 것도 이전 노트북 규칙 그대로다. 변수별 거리는 4-5절 표에 적는다 |
| 110일 | 사용자 지시(2026-10-06 메모 "d-110일치"). 이전 판은 100일이었다(`리스테리아ML/EXP/1/readme.md` "과거 기상 기후는 최대 100일까지 사용 가능"). 110일로 늘린 이유는 메모에 적혀 있지 않다 |
| 기준일 당일(D-0) 포함 | 사용자 지시(2026-10-06 메모 "당일d-0도 넣어"). 이전 판은 D-1 부터였다. D-0 값(그날의 일평균·일최고·일합계)은 기준일이 끝나야 알 수 있는 값이다. **예측을 기준일의 어느 시점에 하는지는 이 노트북에서 정하지 않았다** |
| 강수량만 농업기상(NAS) `rn` | ASOS `sum_rn`(정의서 "일강수량")은 비가 오지 않은 날이 빈칸이라 "그날 값 있는 관측소" 규칙을 쓸 수 없다(`3_기상조인_후_학습.ipynb`). 테이블정의서의 NAS `rn` 설명은 "일강수량. NAS 는 0 을 명시 기록하므로 NULL 은 진짜 결측"이다 |
| 이상한 값을 지우지 않음 | 사용자 결정(2026-10-06). 농업기상 원자료의 값(강수 300mm 초과, 토양수분 음수·50.0 등 — 4-2절 출력)이 3곳 평균에 그대로 들어간다 |
| 동방로거테스트만 제외 | 사용자 결정(2026-10-06). 다른 관측소의 이상한 값은 위 줄대로 그대로 둔다 |
| 국내·해외 값을 같은 칼럼에 | `3_기상조인_후_학습.ipynb` 규칙 그대로. 두 자료는 단위를 맞춰도 같은 장소·같은 날의 값이 다르다(5-2절) |
| 측정망을 가리지 않음 | 에어코리아 측정소는 도시대기·도로변대기·항만 등 측정망 종류가 섞여 있고(4-2절), 가리지 않고 가까운 3곳을 평균했다. 근거는 따로 없다 |

### 4-0. 변수를 고른 근거 (리서치팀 자료)

붙이는 변수는 리서치팀 자료 두 가지에서 왔다.

1. `리서치팀문서/예측인자 정리 ppt.jpg` — "리스테리아 모노사이토제네스 예측인자 우선순위" 표: 수분활성도 1위 · 온도(기온) 1위 · pH 3위 · 염도 4위 · 토양 온도 5위 · 토양 수분 6위 · 대기조성 7위 · 토양 pH 8위 · 강수 9위. 같은 그림의 "추가 예측인자 제시": 식품 유형 · 저장 및 유통 온도 · 포장 유형 · 일조·일사 · 수질.
2. `리서치팀문서/위해요소별_중요인자_데이터출처.xlsx` 의 '리스테리아 모노사이토제네스' 시트 — `데이터 제공 항목` 칸에서 **빨간 글자**로 적힌 항목. 빨간 글자의 뜻(수집할 주요 인자)은 파일 안에 적혀 있지 않고 사용자 설명(2026-09-10)에 따른 것이다. 아래 셀이 빨간 글자를 그대로 뽑아 보여 준다.

이 가운데 식정원 DB 에 있는 것만 넣었다. 변수별 근거는 아래 두 번째 표와 같다.

**근거 문서에 있지만 넣지 않은 것 · 칼럼은 있지만 근거 문서에 없어 넣지 않은 것**
- **습도**: ASOS `avg_rhm`, NAS `hum`, 해외 `relative_humidity_mean` 세 자료 모두에 칼럼이 있으나, 엑셀에서 빨간 글자가 아니고 PPT 우선순위 표에도 없어 넣지 않았다.
- **PM10·PM2.5·오존**: 에어코리아 행에서 빨간 글자가 아니어서 넣지 않았다. 같은 행의 이산화질소·일산화탄소·아황산가스는 빨간 글자라 넣었다.
- **엑셀 빨간 글자인데 넣지 못한 것** (`리스테리아ML/EXP/1/3_기상조인_후_학습.ipynb` 계획 셀의 "뺀 것과 사유", 2026-09-28~29 DB 확인): 대기조성(이산화탄소), 방재기상(AWS), 기상청 농업기상(AAOS), 농업기상 상세관측·팜맵(높이·깊이별 값), 미래 기후도, 토양수분량(수자원공사·유역별·흙토람), 염분·해상풍, 지하수 오염도 — DB 에 없음. 수질오염원 설치시설 — 시설 위치 목록이라 주변 시설 수 세기 같은 가공이 필요.
- **PPT 추가 예측인자**(식품 유형 · 저장 및 유통 온도 · 포장 유형 · 수질)에 해당하는 변수를 따로 만들지는 않았다. 시료 파일의 품목코드 칼럼과 8절의 시료 상태(실온·냉장·냉동)·유통기한이 들어 있을 뿐이다. "포장 유형"에 가까운 칼럼으로 품목제조대장의 `ba_data.t_prdlst_mnf_ledg.frmlc_mtrqlt`(정의서 "포장_재질")가 있으나 요청에 없어 넣지 않았다(값 있는 행 수는 8-4절).

In [9]:
from openpyxl import load_workbook
from openpyxl.cell.rich_text import CellRichText, TextBlock

RED = 'FFFF0000'
def is_red(font):
    return getattr(getattr(font, 'color', None), 'rgb', None) == RED

ws = load_workbook(ROOT / '리서치팀문서' / '위해요소별_중요인자_데이터출처.xlsx', rich_text=True)['리스테리아 모노사이토제네스']
red_rows = []
for r in range(2, ws.max_row + 1):
    cell = ws.cell(r, 4)                       # D열 = 데이터 제공 항목
    v = cell.value
    if v is None:
        continue
    if isinstance(v, CellRichText):
        red = ''.join(b.text for b in v if isinstance(b, TextBlock) and is_red(b.font))
        rest = ''.join((b.text if isinstance(b, TextBlock) else str(b)) for b in v if not (isinstance(b, TextBlock) and is_red(b.font)))
    else:
        red, rest = (str(v), '') if is_red(cell.font) else ('', str(v))
    red_rows.append({'데이터명': ' '.join(str(ws.cell(r, 2).value).split()), '제공기관': ' '.join(str(ws.cell(r, 3).value).split()),
                     '빨간 글자 항목': ' '.join(red.split()), '빨간 글자가 아닌 항목(앞 100자)': ' '.join(rest.split())[:100]})
print('엑셀 `데이터 제공 항목` 칸의 빨간 글자 (파일에서 그대로 뽑음)')
with pd.option_context('display.max_colwidth', 110, 'display.max_rows', None):
    display(pd.DataFrame(red_rows))

BASIS = pd.DataFrame([
    ('avg_temp · max_temp · min_temp', '평균·최고·최저기온', '온도(기온) 1위', 'ASOS 행 "기온" · 농업기상 기본관측 행 "기온, 최고기온, 최저기온"'),
    ('rain', '강수량', '강수 9위', 'ASOS 행 "강수" · 농업기상 기본관측 행 "강수량"'),
    ('avg_wind · max_wind', '평균·최대풍속', '표에 없음', 'ASOS 행 "바람" · 농업기상 기본관측 행 "풍속" (같은 행의 "최대풍속"은 빨간 글자가 아님)'),
    ('wind_dir_sin · wind_dir_cos', '최다풍향', '표에 없음', 'ASOS 행 "바람" · 농업기상 기본관측 행 "풍향"'),
    ('sunshine', '일조시간', '추가 예측인자 "일조·일사"', 'ASOS 행 "일조" · 농업기상 기본관측 행 "일조시간"'),
    ('solar_rad', '일사량', '추가 예측인자 "일조·일사"', '농업기상 기본관측 행 "일사량" (ASOS 행의 "일사"는 빨간 글자가 아님)'),
    ('soil_t5 · soil_t10 · soil_t20 · soil_t30 · soil_t_nas', '지중온도', '토양 온도 5위', '빨간 글자가 아님 (농업기상 기본관측 행의 "지중온도"는 검은 글자, ASOS 행에는 지중온도 항목이 적혀 있지 않음)'),
    ('soil_moist', '토양수분', '토양 수분 6위', '쓴 자료(농업기상 기본관측)의 "토양수분보정값"은 빨간 글자가 아님. 빨간 글자인 기상청 농업기상(AAOS) 행 "토양수분"·팜맵 행 "토양수분값" 자료는 DB 에 없음'),
    ('no2 · co · so2', '이산화질소·일산화탄소·아황산가스', '표의 "대기조성 7위" — 엑셀의 "대기조성(이산화탄소)" 행 자료는 DB 에 없음', '에어코리아 행 "이산화질소(ppm), 일산화탄소(ppm), 아황산가스(ppm)"'),
    ('aw · ph · na', '수분활성도·식품pH·염도', '수분활성도 1위 · pH 3위 · 염도 4위', '이 엑셀에는 없음 (식품 값은 6절의 두 자료)'),
    ('soil_ph · soil_om · soil_vldpha', '토양 산도·유기물·유효인산', '토양 pH 8위', '토양검정 화학성 상세정보 행 "산도(pH)유효인산(mg/kg" · "유기물(g/kg),"'),
    ('inspct_purps_code · splore_sttus_code · distb_de · pog_daycnt', '검사 목적·시료 상태·유통 일·유통기한', '표에 없음 (추가 예측인자에 "저장 및 유통 온도"가 있음)', '이 엑셀에는 없음 (사용자 지정 2026-10-06)'),
], columns=['데이터셋 칼럼', '변수', 'PPT 우선순위 표·추가 예측인자', '엑셀 빨간 글자'])
print('변수별 근거')
with pd.option_context('display.max_colwidth', 120):
    display(BASIS)

엑셀 `데이터 제공 항목` 칸의 빨간 글자 (파일에서 그대로 뽑음)


,데이터명,제공기관,빨간 글자 항목,빨간 글자가 아닌 항목(앞 100자)
0,종관기상관측(ASOS),기상청,"기온, 강수, 바람,일조","기압, 습도, 일사, , 눈, 구름, 시정, 지면상태, 지면온도, 일기현상, 증발량, 현상번호"
1,방재기상관측(AWS),기상청,"기온, 강수, 바람,","습도, 기압"
2,농업기상 관측 조회(AAOS),기상청,"기온, 풍속토양수분, 일조",", 습도, 증발량, 지면·초상·지중온도, 복사, 조도, 일사,, 지하수위"
3,농업기상 기본관측 조회,농촌진흥청 국립농업과학원,"기온, 최고기온, 최저기온풍향, 풍속 강수량, 일조시간, 일사량","결과메세지(정상), 전체 결과 수, 관측지점코드, 관측일자,, 습도, , 최대풍속,, 결로시간, 초상온도, 지중온도, 토양수분보정값"
4,농업기상 상세관측 조회,농촌진흥청 국립농업과학원,"기온(40-50-150cm기준 최고, 최저),풍향(150-300-400cm, 1000cm), 최대풍속(150-300-1000cm), 강수량일조시간, 일사량지중온도(5-10-20cm)토양수분(...","(기본관측에서 제공하는 데이터 포함) 관측시각, 습도(50-150-400cm), , 증발량, , 결로시간, 수중온도, 초상온도, 지중열전도량, , 토양수분보정값(%), ), 습도("
5,토양검정 화학성 상세정보,농촌진흥청 국립농업과학원,"산도(pH)유효인산(mg/kg유기물(g/kg),","시료채취년도, 토양검정일, 경지구분코드, ), 유효규산(mg/kg), 마그네슘(cmol+/kg), 칼륨(cmol+/kg), 칼슘(cmol+/kg), 전기전도도(dS/m)"
6,농업기상 농업용 미래상세 기후도,농촌진흥청 국립농업과학원,"기상요소(월강수량, 월평균기온, 월최고기온, 월최저기온)",예측자료를 기후도면으로 제공 (기후변화시나리오(SSP/RCP)별 제공)
7,팜맵 기반 농업기상 조회,농림축산식품부 농림수산식품교육문화정보원,"기온(50-150-400cm), 풍향(150-300-1000cm), 풍속(150-300-1000cm),강수량, 토양수분값(10-20-30cm),일사량, 일조시간, 일사량, 일조시간풍향, 풍속...","지번코드, 관측시간, 습도(50-150-400cm), 결로시간, 초상온도, 지중온도(10-20cm), 증발량, 수중온도,지중열전도량, 토양수분보정값(10-20-30cm), 적설량,"
8,팜맵 기반 토양검정 조회,농림축산식품부 농림수산식품교육문화정보원,"산도, 유기물함량, 유효인산농도,","유효규산농도, 실제비료칼리사용량, 실제비료석회사용량, 실제비료마그네슘사용량, 석회소요량, 질산 태질소용량, 염기치환용량, 전기전도도, 암모니아태질소용량"
9,토양 수분량,한국수자원공사,토양수분량 관측단위(%),", - 용담시험유역 6개 지점(주천, 부귀, 상전, 안천, 천천, 계북)의 토심별 일별 수분량을 포함. - 부귀·주천은 10, 20, 40, 60cm 정보를, 나머지 4개소는 80"


변수별 근거


,데이터셋 칼럼,변수,PPT 우선순위 표·추가 예측인자,엑셀 빨간 글자
0,avg_temp · max_temp · min_temp,평균·최고·최저기온,온도(기온) 1위,"ASOS 행 ""기온"" · 농업기상 기본관측 행 ""기온, 최고기온, 최저기온"""
1,rain,강수량,강수 9위,"ASOS 행 ""강수"" · 농업기상 기본관측 행 ""강수량"""
2,avg_wind · max_wind,평균·최대풍속,표에 없음,"ASOS 행 ""바람"" · 농업기상 기본관측 행 ""풍속"" (같은 행의 ""최대풍속""은 빨간 글자가 아님)"
3,wind_dir_sin · wind_dir_cos,최다풍향,표에 없음,"ASOS 행 ""바람"" · 농업기상 기본관측 행 ""풍향"""
4,sunshine,일조시간,"추가 예측인자 ""일조·일사""","ASOS 행 ""일조"" · 농업기상 기본관측 행 ""일조시간"""
5,solar_rad,일사량,"추가 예측인자 ""일조·일사""","농업기상 기본관측 행 ""일사량"" (ASOS 행의 ""일사""는 빨간 글자가 아님)"
6,soil_t5 · soil_t10 · soil_t20 · soil_t30 · soil_t_nas,지중온도,토양 온도 5위,"빨간 글자가 아님 (농업기상 기본관측 행의 ""지중온도""는 검은 글자, ASOS 행에는 지중온도 항목이 적혀 있지 않음)"
7,soil_moist,토양수분,토양 수분 6위,"쓴 자료(농업기상 기본관측)의 ""토양수분보정값""은 빨간 글자가 아님. 빨간 글자인 기상청 농업기상(AAOS) 행 ""토양수분""·팜맵 행 ""토양수분값"" 자료는 DB 에 없음"
8,no2 · co · so2,이산화질소·일산화탄소·아황산가스,"표의 ""대기조성 7위"" — 엑셀의 ""대기조성(이산화탄소)"" 행 자료는 DB 에 없음","에어코리아 행 ""이산화질소(ppm), 일산화탄소(ppm), 아황산가스(ppm)"""
9,aw · ph · na,수분활성도·식품pH·염도,수분활성도 1위 · pH 3위 · 염도 4위,이 엑셀에는 없음 (식품 값은 6절의 두 자료)


### 4-1. 변수 정의

단위의 출처는 자료마다 다르다. **NAS**(`api_data_info.nas_agri_weather_daily_info`)와 **해외**(`api_data_info.global_factory_weather`)는 DB 코멘트에 단위가 적혀 있다. **ASOS**(`api_data_info.kma_asos_daily_info`)는 DB 코멘트가 없고 정의서에 풍속·일사량·일조의 단위가 없어, `3_기상조인_후_학습.ipynb` 에서 값 범위로 판단한 단위(풍속 m/s · 일사량 MJ/m² · 일조 시간)를 그대로 쓴다. **에어코리아** 세 변수의 ppm 은 테이블정의서("이산화질소농도"·"일산화탄소농도"·"아황산가스농도")에 없고 리서치팀 엑셀의 표기("이산화질소(ppm), 일산화탄소(ppm), 아황산가스(ppm)")를 따른 것이다 — 확인 필요. 이 셀 아래 출력에 이번 원자료의 값 범위를 다시 찍는다.

In [10]:
# 칼럼 이름 · 한글 이름 · 단위 · 국내 기본 (관측망, DB 칼럼) · 국내 대체 또는 None · 해외 (DB 칼럼, 나누는 수) 또는 None
VARIABLES = [
    dict(var='avg_temp',   ko='평균기온',      unit='°C',    prim=('ASOS', 'avg_ta'),       fb=('NAS', 'temp'),        ov=('temperature_mean', 1)),
    dict(var='max_temp',   ko='최고기온',      unit='°C',    prim=('ASOS', 'max_ta'),       fb=('NAS', 'hghst_artmp'), ov=('temperature_max', 1)),
    dict(var='min_temp',   ko='최저기온',      unit='°C',    prim=('ASOS', 'min_ta'),       fb=('NAS', 'lowst_artmp'), ov=('temperature_min', 1)),
    dict(var='rain',       ko='강수량',        unit='mm',    prim=('NAS', 'rn'),            fb=None,                   ov=('precipitation_sum', 1)),
    dict(var='avg_wind',   ko='평균풍속',      unit='m/s',   prim=('ASOS', 'avg_ws'),       fb=('NAS', 'wind'),        ov=('wind_speed_mean', 3.6)),
    dict(var='max_wind',   ko='최대풍속',      unit='m/s',   prim=('ASOS', 'max_ws'),       fb=('NAS', 'max_wind'),    ov=('wind_speed_max', 3.6)),
    dict(var='sunshine',   ko='일조시간',      unit='시간',  prim=('ASOS', 'sum_sshr'),     fb=None,                   ov=('sunshine_duration', 3600)),
    dict(var='solar_rad',  ko='일사량',        unit='MJ/m²', prim=('ASOS', 'sum_gsr'),      fb=('NAS', 'srqty'),       ov=('radiation_sum', 1)),
    dict(var='soil_t5',    ko='지중온도 5cm',  unit='°C',    prim=('ASOS', 'avg_cm5_te'),   fb=None,                   ov=None),
    dict(var='soil_t10',   ko='지중온도 10cm', unit='°C',    prim=('ASOS', 'avg_cm10_te'),  fb=None,                   ov=None),
    dict(var='soil_t20',   ko='지중온도 20cm', unit='°C',    prim=('ASOS', 'avg_cm20_tes'), fb=None,                   ov=None),
    dict(var='soil_t30',   ko='지중온도 30cm', unit='°C',    prim=('ASOS', 'avg_cm30_te'),  fb=None,                   ov=None),
    dict(var='soil_t_nas', ko='지중온도(NAS, 깊이 설명 없음)', unit='°C', prim=('NAS', 'soil_temp'), fb=None,          ov=None),
    dict(var='soil_moist', ko='토양수분(국내만)', unit='%',   prim=('NAS', 'soil_wt'),       fb=None,                   ov=None),
    dict(var='no2',        ko='이산화질소',    unit='ppm',   prim=('AIR', 'no2_value'),     fb=None,                   ov=None),
    dict(var='co',         ko='일산화탄소',    unit='ppm',   prim=('AIR', 'co_value'),      fb=None,                   ov=None),
    dict(var='so2',        ko='아황산가스',    unit='ppm',   prim=('AIR', 'so2_value'),     fb=None,                   ov=None),
]
# 5절 풍향 — ASOS max_ws1(정의서 "최다풍향", 도)을 sin·cos 로 바꾼 값. 해외 자료 없음
WIND_DIR_VARIABLES = [
    dict(var='wind_dir_sin', ko='최다풍향 sin', unit='(−1~1)', prim=('ASOS', 'max_ws1_sin'), fb=None, ov=None),
    dict(var='wind_dir_cos', ko='최다풍향 cos', unit='(−1~1)', prim=('ASOS', 'max_ws1_cos'), fb=None, ov=None),
]
ALL_WX = VARIABLES + WIND_DIR_VARIABLES
WX_VARS = [v['var'] for v in ALL_WX]

def wx_cols(var):
    return [f'{var}_d{k:03d}' for k in LAGS]

WX_COLS = [c for v in WX_VARS for c in wx_cols(v)]
NET_TABLE = {'ASOS': 'api_data_info.kma_asos_daily_info', 'NAS': 'api_data_info.nas_agri_weather_daily_info',
             'AIR': 'api_data_info.keco_airkorea_fine_dust_info'}
display(pd.DataFrame([{
    '칼럼': f"{v['var']}_d000~d{LAG_MAX:03d}", '변수': v['ko'], '단위': v['unit'],
    '국내 기본': f"{v['prim'][0]} `{v['prim'][1]}`", '국내 대체': f"{v['fb'][0]} `{v['fb'][1]}`" if v['fb'] else '없음',
    '해외': (f"`{v['ov'][0]}`" + (f" ÷ {v['ov'][1]}" if v['ov'][1] != 1 else '')) if v['ov'] else '없음'} for v in ALL_WX]))
print(f'기상·대기·풍향 변수 {len(WX_VARS)}개 × {len(LAGS)}일 = {len(WX_COLS):,}칼럼')

,칼럼,변수,단위,국내 기본,국내 대체,해외
0,avg_temp_d000~d110,평균기온,°C,ASOS `avg_ta`,NAS `temp`,`temperature_mean`
1,max_temp_d000~d110,최고기온,°C,ASOS `max_ta`,NAS `hghst_artmp`,`temperature_max`
2,min_temp_d000~d110,최저기온,°C,ASOS `min_ta`,NAS `lowst_artmp`,`temperature_min`
3,rain_d000~d110,강수량,mm,NAS `rn`,없음,`precipitation_sum`
4,avg_wind_d000~d110,평균풍속,m/s,ASOS `avg_ws`,NAS `wind`,`wind_speed_mean` ÷ 3.6
5,max_wind_d000~d110,최대풍속,m/s,ASOS `max_ws`,NAS `max_wind`,`wind_speed_max` ÷ 3.6
6,sunshine_d000~d110,일조시간,시간,ASOS `sum_sshr`,없음,`sunshine_duration` ÷ 3600
7,solar_rad_d000~d110,일사량,MJ/m²,ASOS `sum_gsr`,NAS `srqty`,`radiation_sum`
8,soil_t5_d000~d110,지중온도 5cm,°C,ASOS `avg_cm5_te`,없음,없음
9,soil_t10_d000~d110,지중온도 10cm,°C,ASOS `avg_cm10_te`,없음,없음


기상·대기·풍향 변수 19개 × 111일 = 2,109칼럼


### 4-2. 관측망별 일자료·관측소 좌표

관측망마다 (날짜 × 관측소) 표를 만든다. 칸 값은 그날 그 관측소의 원자료 값이고, 자료 행이 없거나 값이 비면 NaN 이다.

- **ASOS**: 좌표는 `api_data_info.kma_ground_obs_station_info` 의 `lat`·`lon`. 한 지점에 위치 행이 여러 개(운영 시작일 `oper_start_dt` 별)라, 날짜마다 "운영 시작일이 그날 이전인 행 중 가장 최근 행"의 좌표를 쓴다(그날 이전 행이 없으면 가장 이른 행).
- **NAS**: 좌표는 `api_data_info.nas_agri_weather_obs_station` 의 `instl_la`(위도)·`instl_lo`(경도), 관측소당 1행.
- **에어코리아**: 일자료는 측정소 이름 `station_nm` 으로만 구분된다. 좌표는 `api_data_info.keco_airkorea_obs_station` 의 `dm_x`·`dm_y`(DB 코멘트 "WGS84 기반 X/Y 좌표". 값을 보면 `dm_x` 가 위도 범위, `dm_y` 가 경도 범위라 그렇게 쓴다 — 아래 출력으로 확인). 같은 이름이 두 행인 측정소는 좌표가 같을 때만 1행으로 쓴다(다르면 멈춘다). 측정소 정보에 없는 측정소는 좌표를 몰라 뺀다(아래 출력).
- 같은 관측소·같은 날 행이 두 개 이상이면 어느 값을 쓸지 정해야 하므로 멈춘다(아래에서 0건 확인).
- **이상한 값은 지우지 않는다**(사용자 결정 2026-10-06). 농업기상(NAS) 원자료에는 강수량 300mm 초과, 토양수분보정값 음수·정확히 50.0 같은 값이 있고(아래 출력), 3곳 평균에 그대로 들어간다.
- **에어코리아 측정소는 측정망 종류(`mang_name`: 도시대기·도로변대기·항만 등)를 가리지 않고 쓴다**(아래 출력).
- **NAS '동방로거테스트'(`561034A001`)는 쓰지 않는다**(사용자 결정 2026-10-06). 빼는 관측소는 이 한 곳뿐이다. 관측지점 정보에서 `use_yn` 이 0 인 곳이 여기 하나뿐이고, 이름에 '테스트'가 들어 있으며, 설치 주소가 업체 주소다. 일자료도 1~3월 평균기온이 30.1 로 찍힌 날이 있는 등 날씨로 보기 어려운 값이 있다(아래 출력). DB 에 "시험용"이라는 설명은 없다 — 이름·주소·`use_yn`·값으로 판단한 것이므로 식정원 확인이 필요하다.

In [11]:
def didx(ts):
    """날짜 → 기상 기간 첫날(DATE_MIN)부터 며칠째 (0부터)"""
    return (pd.Timestamp(ts) - DATE_MIN).days

def to_date(s):
    return pd.to_datetime(s).dt.normalize()

def daily_table(d, sid_col, date_col, value_cols, sids):
    """(날짜 × 관측소) float32 표 — 기상 기간(DATES) 전체, 관측소 순서 sids"""
    d = d[[sid_col, date_col] + value_cols].copy()
    d[date_col] = to_date(d[date_col])
    assert not d.duplicated([sid_col, date_col]).any(), '같은 관측소·같은 날 행이 두 개 이상 있음'
    d = d[(d[date_col] >= DATE_MIN) & (d[date_col] <= DATE_MAX)]
    sid_pos = {s: j for j, s in enumerate(sids)}
    d = d[d[sid_col].isin(sid_pos)]
    ri = (d[date_col] - DATE_MIN).dt.days.to_numpy()
    ci = d[sid_col].map(sid_pos).to_numpy()
    out = {}
    for c in value_cols:
        V = np.full((N_DATES, len(sids)), np.nan, dtype=np.float32)
        V[ri, ci] = pd.to_numeric(d[c]).to_numpy(np.float32)
        out[c] = V
    return out

def net_cols(net):
    return sorted({v['prim'][1] for v in VARIABLES if v['prim'][0] == net} | {v['fb'][1] for v in VARIABLES if v['fb'] and v['fb'][0] == net})

NETS = {}   # 관측망 → {'sids', 'names', 'periods': [(시작 날짜번호, 끝 날짜번호, 좌표 배열)], 'values': {칼럼: 표}}

# ASOS — 날짜별 위치
asos_cols = net_cols('ASOS') + ['max_ws1']   # max_ws1 = 최다풍향 (5절)
asos = pd.read_parquet(RAW_DIR / 'raw_kma_asos_daily_info.parquet', columns=['stn_id', 'meas_date'] + asos_cols)
asos_st = pd.read_parquet(RAW_DIR / 'raw_kma_ground_obs_station_info.parquet')
asos['stn_id'] = asos['stn_id'].astype(str); asos_st['stn_id'] = asos_st['stn_id'].astype(str)
print('ASOS 일자료 기간:', asos['meas_date'].min(), '~', asos['meas_date'].max(), f'· {len(asos):,}행')
print('  sum_sshr(텍스트) 중 숫자로 안 바뀌는 값:', int((asos['sum_sshr'].notna() & pd.to_numeric(asos['sum_sshr'], errors='coerce').isna()).sum()), '건')
sids = sorted(asos['stn_id'].unique())
st = asos_st[asos_st['stn_id'].isin(sids)].copy()
st['start'] = to_date(st['oper_start_dt'])
assert set(st['stn_id']) == set(sids), 'ASOS 좌표가 없는 관측소가 있음'
changes = sorted(d for d in st['start'].unique() if DATE_MIN < d <= DATE_MAX)
bounds = [DATE_MIN] + changes + [DATE_MAX + pd.Timedelta(days=1)]
periods = []
for a, b in zip(bounds[:-1], bounds[1:]):
    xy = []
    for s in sids:
        g = st[st['stn_id'] == s].sort_values('start')
        ok = g[g['start'] <= a]
        r = ok.iloc[-1] if len(ok) else g.iloc[0]
        xy.append((float(r['lat']), float(r['lon'])))
    periods.append((didx(a), min(didx(b), N_DATES), np.array(xy)))
NETS['ASOS'] = {'sids': sids, 'names': st.drop_duplicates('stn_id').set_index('stn_id')['stn_ko'].reindex(sids).tolist(),
                'periods': periods, 'values': daily_table(asos, 'stn_id', 'meas_date', asos_cols, sids)}
print(f'ASOS: 관측소 {len(sids)}곳 · 위치가 바뀌는 날 {len(changes)}번 → 기간 {len(periods)}개 · 칼럼 {asos_cols}')

# NAS — 관측소당 좌표 1개
nas_cols = net_cols('NAS')
nas = pd.read_parquet(RAW_DIR / 'raw_nas_agri_weather_daily_info.parquet', columns=['obsr_spot_code', 'meas_date'] + nas_cols)
nas_st = pd.read_parquet(RAW_DIR / 'raw_nas_agri_weather_obs_station.parquet').set_index('obsr_spot_code')
print('NAS 일자료 기간:', nas['meas_date'].min(), '~', nas['meas_date'].max(), f'· {len(nas):,}행')
EXCLUDE_NAS = ['561034A001']   # 동방로거테스트 — 쓰지 않는다
ex = nas_st.loc[EXCLUDE_NAS]
print('  쓰지 않는 관측소:')
display(ex[['obsr_spot_nm', 'use_yn', 'instl_adres', 'obsr_begin_datetm', 'obsr_end_datetm']])
print('  관측지점 정보의 use_yn 값별 지점 수:', nas_st['use_yn'].value_counts(dropna=False).to_dict(), "· 이름에 '테스트'가 든 지점:", nas_st.index[nas_st['obsr_spot_nm'].str.contains('테스트')].tolist())
ex_day = nas[nas['obsr_spot_code'].isin(EXCLUDE_NAS)]
ex_month = pd.to_datetime(ex_day['meas_date']).dt.month
print(f"  이 관측소 일자료 {len(ex_day):,}행 ({ex_day['meas_date'].min()} ~ {ex_day['meas_date'].max()}) · 값 있는 행: " + ', '.join(f'{c} {int(ex_day[c].notna().sum())}' for c in nas_cols))
print(f"  1~3월 평균기온(temp) 값: {ex_day.loc[ex_month.isin([1, 2, 3]), 'temp'].value_counts().to_dict()} · 평균기온이 30.1 인 날 {int((ex_day['temp'] == 30.1).sum())}일 · 최고기온(hghst_artmp) 최솟값 {ex_day['hghst_artmp'].min()}")
sids = [s_ for s_ in sorted(nas['obsr_spot_code'].unique()) if s_ not in EXCLUDE_NAS]
assert nas_st.index.is_unique and set(sids) <= set(nas_st.index), 'NAS 좌표가 없는 관측소가 있음'
xy = nas_st.loc[sids, ['instl_la', 'instl_lo']].to_numpy(float)
NETS['NAS'] = {'sids': sids, 'names': nas_st.loc[sids, 'obsr_spot_nm'].tolist(), 'periods': [(0, N_DATES, xy)],
               'values': daily_table(nas, 'obsr_spot_code', 'meas_date', nas_cols, sids)}
print(f'NAS: 관측소 {len(sids)}곳 (동방로거테스트 561034A001 을 뺀 수 · 포함 여부: {"561034A001" in sids}) · 칼럼 {nas_cols}')

# 에어코리아 — 측정소 이름으로 좌표 연결
air_cols = net_cols('AIR')
air = pd.read_parquet(RAW_DIR / 'raw_keco_airkorea_fine_dust_info.parquet', columns=['station_nm', 'msur_dt'] + air_cols)
air_st = pd.read_parquet(RAW_DIR / 'raw_keco_airkorea_obs_station.parquet')
print('에어코리아 일자료 기간:', air['msur_dt'].min(), '~', air['msur_dt'].max(), f'· {len(air):,}행')
print(f"  에어코리아 dm_x 범위 {air_st['dm_x'].min():.2f}~{air_st['dm_x'].max():.2f} (위도) · dm_y 범위 {air_st['dm_y'].min():.2f}~{air_st['dm_y'].max():.2f} (경도)")
assert air_st['dm_x'].between(33, 39).all() and air_st['dm_y'].between(124, 132).all()
dup = air_st.groupby('station_nm')[['dm_x', 'dm_y']].nunique()
assert (dup <= 1).all().all(), '같은 이름인데 좌표가 다른 측정소가 있음'
air_xy = air_st.drop_duplicates('station_nm').set_index('station_nm')[['dm_x', 'dm_y']]
all_air = sorted(air['station_nm'].unique())
sids = [s for s in all_air if s in air_xy.index]
no_xy = [s for s in all_air if s not in air_xy.index]
NETS['AIR'] = {'sids': sids, 'names': sids, 'periods': [(0, N_DATES, air_xy.loc[sids].to_numpy(float))],
               'values': daily_table(air, 'station_nm', 'msur_dt', air_cols, sids)}
print(f'에어코리아: 일자료 측정소 {len(all_air)}곳 중 좌표 있는 {len(sids)}곳 사용, 좌표 없어 뺀 {len(no_xy)}곳: {no_xy}')

# 농업기상(NAS) 원자료에 그대로 있는 값 — 지우지 않는다(사용자 결정 2026-10-06). 기상 기간 안, 동방로거테스트 제외
V_rn, V_sw = NETS['NAS']['values']['rn'], NETS['NAS']['values']['soil_wt']
big = V_rn > 300
print(f"NAS rn(강수량) 300mm 초과: {int(big.sum())}칸 · 관측소 {int(big.any(axis=0).sum())}곳 · 큰 값 8개: {np.sort(V_rn[big])[::-1][:8].tolist()}")
print(f"NAS soil_wt(토양수분보정값) 음수: {int((V_sw < 0).sum()):,}칸 · 관측소 {int((V_sw < 0).any(axis=0).sum())}곳 · 최솟값 {float(np.nanmin(V_sw)):.1f}"
      f" / 정확히 50.0 인 칸 {int((V_sw == 50.0).sum()):,} · 정확히 5.0 인 칸 {int((V_sw == 5.0).sum()):,} (왜 이 값에 몰리는지 설명 없음 — 확인 필요)")
# 에어코리아 측정망 종류 (keco_airkorea_obs_station.mang_name)
AIR_MANG = air_st.drop_duplicates('station_nm').set_index('station_nm')['mang_name'].reindex(NETS['AIR']['sids']).to_numpy()
print('쓴 에어코리아 측정소의 측정망 종류(mang_name):', pd.Series(AIR_MANG).value_counts(dropna=False).to_dict())

# 단위 판단에 쓴 값 범위 (원자료 그대로, 기상 기간 안)
rng_rows = []
for v in VARIABLES:
    for role, spec in (('기본', v['prim']), ('대체', v['fb'])):
        if spec:
            V = NETS[spec[0]]['values'][spec[1]]
            rng_rows.append({'변수': v['ko'], '구분': role, '테이블.칼럼': f'{NET_TABLE[spec[0]]}.{spec[1]}', '단위': v['unit'],
                             '값 있는 칸': int((~np.isnan(V)).sum()), '최솟값': float(np.nanmin(V)), '중앙값': float(np.nanmedian(V)), '최댓값': float(np.nanmax(V)),
                             '값 있는 관측소 수': int((~np.isnan(V)).any(axis=0).sum())})
display(pd.DataFrame(rng_rows))
del asos, nas, air; gc.collect();

ASOS 일자료 기간: 2015-01-01 ~ 2026-09-03 · 406,909행


  sum_sshr(텍스트) 중 숫자로 안 바뀌는 값: 0 건


ASOS: 관측소 99곳 · 위치가 바뀌는 날 20번 → 기간 21개 · 칼럼 ['avg_cm10_te', 'avg_cm20_tes', 'avg_cm30_te', 'avg_cm5_te', 'avg_ta', 'avg_ws', 'max_ta', 'max_ws', 'min_ta', 'sum_gsr', 'sum_sshr', 'max_ws1']
NAS 일자료 기간: 2008-01-01 ~ 2026-09-03 · 984,648행
  쓰지 않는 관측소:


,obsr_spot_nm,use_yn,instl_adres,obsr_begin_datetm,obsr_end_datetm
obsr_spot_code,,,,,
561034A001,동방로거테스트,0,"전라븍도 정읍시 북면 북면공단2길 87, (유)동방이노베이션",2021-09-07,2026-07-15


  관측지점 정보의 use_yn 값별 지점 수: {'1': 218, '0': 1} · 이름에 '테스트'가 든 지점: ['561034A001']
  이 관측소 일자료 444행 (2021-09-08 ~ 2025-12-31) · 값 있는 행: hghst_artmp 197, lowst_artmp 243, max_wind 199, rn 331, soil_temp 0, soil_wt 0, srqty 274, temp 176, wind 253
  1~3월 평균기온(temp) 값: {30.1: 75} · 평균기온이 30.1 인 날 84일 · 최고기온(hghst_artmp) 최솟값 -40.0


NAS: 관측소 218곳 (동방로거테스트 561034A001 을 뺀 수 · 포함 여부: False) · 칼럼 ['hghst_artmp', 'lowst_artmp', 'max_wind', 'rn', 'soil_temp', 'soil_wt', 'srqty', 'temp', 'wind']
에어코리아 일자료 기간: 2015-01-01 ~ 2026-08-31 · 2,150,630행
  에어코리아 dm_x 범위 33.23~38.56 (위도) · dm_y 범위 124.65~130.90 (경도)


에어코리아: 일자료 측정소 691곳 중 좌표 있는 672곳 사용, 좌표 없어 뺀 19곳: ['난지도리(폐쇄)', '남산동(폐쇄)', '남중동', '노원동(폐쇄)', '동대문(폐쇄)', '동탄기흥로', '부곡동', '석사동', '송정1동(폐쇄)', '송정동 대기환경측정소', '송정동(봉명동)', '신흥동', '야음동', '원시동', '정곡리(폐쇄)', '중동(유해+중금속)', '지품면(폐쇄)', '진상면(폐쇄)', '호매실동(폐쇄)']
NAS rn(강수량) 300mm 초과: 124칸 · 관측소 44곳 · 큰 값 8개: [500.0, 499.5, 499.0, 499.0, 498.5, 498.5, 498.5, 498.0]
NAS soil_wt(토양수분보정값) 음수: 8,820칸 · 관측소 43곳 · 최솟값 -36.2 / 정확히 50.0 인 칸 24,568 · 정확히 5.0 인 칸 11,514 (왜 이 값에 몰리는지 설명 없음 — 확인 필요)
쓴 에어코리아 측정소의 측정망 종류(mang_name): {'도시대기': 532, '도로변대기': 68, '항만': 34, '교외대기': 27, '국가배경농도(도서)': 11}


,변수,구분,테이블.칼럼,단위,값 있는 칸,최솟값,중앙값,최댓값,값 있는 관측소 수
0,평균기온,기본,api_data_info.kma_asos_daily_info.avg_ta,°C,400042,-19.700001,14.200000,34.099998,99
1,평균기온,대체,api_data_info.nas_agri_weather_daily_info.temp,°C,726637,-49.400002,13.500000,49.500000,194
2,최고기온,기본,api_data_info.kma_asos_daily_info.max_ta,°C,400435,-15.200000,20.100000,41.000000,99
3,최고기온,대체,api_data_info.nas_agri_weather_daily_info.hghst_artmp,°C,726414,-50.000000,20.200001,50.000000,194
4,최저기온,기본,api_data_info.kma_asos_daily_info.min_ta,°C,400428,-25.200001,8.900000,30.600000,99
5,최저기온,대체,api_data_info.nas_agri_weather_daily_info.lowst_artmp,°C,726412,-50.000000,7.300000,50.000000,195
6,강수량,기본,api_data_info.nas_agri_weather_daily_info.rn,mm,761240,0.000000,0.000000,500.000000,202
7,평균풍속,기본,api_data_info.kma_asos_daily_info.avg_ws,m/s,399971,0.000000,1.600000,22.299999,99
8,평균풍속,대체,api_data_info.nas_agri_weather_daily_info.wind,m/s,658941,0.000000,0.800000,37.299999,181
9,최대풍속,기본,api_data_info.kma_asos_daily_info.max_ws,m/s,400122,0.000000,4.300000,49.000000,99


### 4-3. 국내 행: 날짜마다 가까운 관측소 3곳 평균

국내 시료의 고유 좌표마다, 기상 기간의 날짜마다 아래를 계산해 (좌표 × 날짜) 표를 만든다.

1. 그 좌표에서 모든 관측소까지 **실제 거리(구면 거리, km)** — 1절의 `haversine_km`, 지구 반지름 6,371.0088km
2. 그날 그 변수 값이 있는 관측소만 남겨 가까운 순으로 3곳을 고른다
3. 세 값의 평균 (3곳이 안 되면 있는 곳만 평균, 한 곳도 없으면 NaN)

점검용으로 고른 관측소까지의 평균 거리(km)도 같이 구한다. 그다음 기본 칼럼이 NaN 인 칸에만 대체 칼럼 값을 넣는다.

In [12]:
DOM_XY = XY.loc[IS_DOM, ['lat', 'lon']].drop_duplicates().reset_index(drop=True)   # 국내 시료 좌표(시료 파일 float32)
DOM_LOC = pd.MultiIndex.from_frame(DOM_XY)
print(f'국내 시료 고유 좌표 {len(DOM_XY):,}곳')

def knn_mean(V, periods, loc_xy, k=K_NEAREST, top=TOP_CANDIDATES, block=32):
    """V: (날짜 × 관측소) 값 표. 반환: 평균값 (좌표 × 날짜), 고른 관측소까지 평균 거리 km (좌표 × 날짜)"""
    have = ~np.isnan(V).all(axis=0)              # 기상 기간 중 값이 한 번이라도 있는 관측소만 후보
    Vh = V[:, have]
    n_loc = len(loc_xy)
    M = np.full((n_loc, N_DATES), np.nan, dtype=np.float32)
    Dm = np.full((n_loc, N_DATES), np.nan, dtype=np.float32)
    n_redo = 0
    for a, b, st_xy in periods:
        xy = st_xy[have]
        dist = haversine_km(loc_xy[:, None, 0], loc_xy[:, None, 1], xy[None, :, 0], xy[None, :, 1])   # (좌표, 관측소) 구면 거리 km
        order_all = np.argsort(dist, axis=1, kind='stable')
        t = min(top, xy.shape[0])
        order = order_all[:, :t]
        dsort = np.take_along_axis(dist, order, axis=1)
        for s0 in range(a, b, block):
            s1 = min(s0 + block, b)
            vals = Vh[s0:s1][:, order]                        # (날짜, 좌표, 후보)
            ok = ~np.isnan(vals)
            sel = ok & (np.cumsum(ok, axis=2) <= k)           # 값 있는 곳 중 가까운 k 곳
            cnt = sel.sum(axis=2)
            with np.errstate(invalid='ignore', divide='ignore'):
                mean = np.where(sel, vals, 0).sum(axis=2, dtype=np.float64) / cnt
                dmean = (sel * dsort[None]).sum(axis=2) / cnt
            # 후보 60곳 안에 값 있는 곳이 k 곳보다 적은데, 전체 관측소에는 더 있는 칸 → 전체에서 다시 고른다
            n_day = (~np.isnan(Vh[s0:s1])).sum(axis=1)
            for bi, li in zip(*np.nonzero(cnt < np.minimum(k, n_day)[:, None])):
                o = order_all[li]
                v = Vh[s0 + bi, o]
                idx = np.nonzero(~np.isnan(v))[0][:k]
                mean[bi, li] = v[idx].astype(np.float64).mean()
                dmean[bi, li] = dist[li, o[idx]].mean()
                n_redo += 1
            M[:, s0:s1] = mean.T
            Dm[:, s0:s1] = dmean.T
    return M, Dm, n_redo

국내 시료 고유 좌표 3,599곳


#### 거리 방식에 따라 가까운 3곳이 달라지는 좌표 수 (참고)

위경도 차이를 그대로 쓴 거리(`3_기상조인_후_학습.ipynb` 방식)와 실제 거리(km)로 각각 가까운 ASOS 관측소 3곳을 골라 비교한다. **규모를 보려는 계산**이라 관측소는 가장 최근 위치를 쓰고 그날 값이 있는지는 따지지 않는다(실제 조인은 날짜별 위치·그날 값 있는 관측소로 한다).

In [13]:
st_xy = NETS['ASOS']['periods'][-1][2]                     # ASOS 관측소의 가장 최근 위치
lx = DOM_XY.to_numpy(float)
d_deg = np.sqrt(((lx[:, None, :] - st_xy[None, :, :]) ** 2).sum(-1))
d_km = haversine_km(lx[:, None, 0], lx[:, None, 1], st_xy[None, :, 0], st_xy[None, :, 1])
top_deg = np.sort(np.argsort(d_deg, axis=1, kind='stable')[:, :K_NEAREST], axis=1)
top_km = np.sort(np.argsort(d_km, axis=1, kind='stable')[:, :K_NEAREST], axis=1)
changed = (top_deg != top_km).any(axis=1)
loc_of_row = DOM_LOC.get_indexer(pd.MultiIndex.from_frame(XY.loc[IS_DOM, ['lat', 'lon']]))
n_per_loc = np.bincount(loc_of_row, minlength=len(DOM_XY))
print(f'가까운 ASOS 3곳이 두 방식에서 다른 국내 좌표: {int(changed.sum()):,} / {len(DOM_XY):,}곳 ({changed.mean() * 100:.1f}%)'
      f' · 그 좌표의 시료 행 {int(n_per_loc[changed].sum()):,} / {int(IS_DOM.sum()):,} ({n_per_loc[changed].sum() / IS_DOM.sum() * 100:.1f}%)')
i = int(np.argmax(np.where(changed, n_per_loc, -1)))        # 달라지는 좌표 중 시료 행이 가장 많은 곳
ex = pd.DataFrame({'관측소': NETS['ASOS']['names'], '위도 차(도)': st_xy[:, 0] - lx[i, 0], '경도 차(도)': st_xy[:, 1] - lx[i, 1],
                   '위경도 차이 거리': d_deg[i], '실제 거리(km)': d_km[i]})
ex['위경도 차이 순위'] = ex['위경도 차이 거리'].rank(method='first').astype(int)
ex['실제 거리 순위'] = ex['실제 거리(km)'].rank(method='first').astype(int)
print(f'예 — 달라지는 좌표 중 시료 행이 가장 많은 곳 ({lx[i, 0]:.4f}, {lx[i, 1]:.4f}), 시료 {int(n_per_loc[i]):,}행')
display(ex.sort_values('실제 거리(km)').head(5).round(4).reset_index(drop=True))
del d_deg, d_km

가까운 ASOS 3곳이 두 방식에서 다른 국내 좌표: 717 / 3,599곳 (19.9%) · 그 좌표의 시료 행 124,950 / 628,895 (19.9%)
예 — 달라지는 좌표 중 시료 행이 가장 많은 곳 (37.5396, 127.0540), 시료 4,462행


,관측소,위도 차(도),경도 차(도),위경도 차이 거리,실제 거리(km),위경도 차이 순위,실제 거리 순위
0,서울,0.0318,-0.0882,0.0938,8.5465,1,1
1,수원,-0.2821,-0.0710,0.2909,31.9847,2,2
2,인천,-0.0619,-0.4291,0.4336,38.4727,4,3
3,양평,-0.0510,0.4405,0.4434,39.2598,5,4
4,동두천,0.3623,0.0067,0.3624,40.2953,3,5


### 4-4. 해외 행: 같은 좌표의 그날 값 (풍속·일조는 국내 단위로 맞춤)

`raw_global_factory_weather_listeria` 의 DB 좌표를 float32 로 바꿔 시료 좌표와 맞춘다. 한 시료 좌표에 DB 좌표가 2곳 이상 걸리면 그 값들의 평균을 쓴다.

단위 변환은 조인할 때 한다(원자료 파일은 km/h·초 그대로): 풍속 ÷ 3.6 → m/s, 일조 ÷ 3600 → 시간.

In [14]:
OV_LOC = pd.MultiIndex.from_frame(OV_XY)
OV_DB_COLS = sorted({v['ov'][0] for v in VARIABLES if v['ov']})

t = pq.read_table(GFW_PATH, columns=['lat_g', 'lon_g', 'obsr_de'])
lat_g = t.column('lat_g').to_numpy(); lon_g = t.column('lon_g').to_numpy()
day_no = t.column('obsr_de').cast(pa.int32()).to_numpy().astype(np.int64) - np.datetime64(DATE_MIN.date(), 'D').astype(np.int64)
print('해외 원자료 기간:', pc.min(t.column('obsr_de')).as_py(), '~', pc.max(t.column('obsr_de')).as_py(), f'· {t.num_rows:,}행')
del t

# DB 좌표(double) → 시료 좌표(float32) 번호
g_all = pd.DataFrame({'lat_g': lat_g, 'lon_g': lon_g})
gid = g_all.groupby(['lat_g', 'lon_g'], sort=False).ngroup().to_numpy()   # 행마다 DB 좌표 번호 (처음 나온 순서)
grid = g_all.drop_duplicates().reset_index(drop=True)                     # DB 좌표 목록 (같은 순서)
assert len(grid) == gid.max() + 1 and (grid['lat_g'].to_numpy()[gid] == lat_g).all() and (grid['lon_g'].to_numpy()[gid] == lon_g).all()
del g_all, lat_g, lon_g
grid['lat'] = grid['lat_g'].astype(np.float32); grid['lon'] = grid['lon_g'].astype(np.float32)
grid['loc'] = OV_LOC.get_indexer(pd.MultiIndex.from_frame(grid[['lat', 'lon']]))
n_grid = grid[grid['loc'] >= 0].groupby('loc').size()
print(f'원자료의 DB 좌표 {len(grid):,}곳 · 해외 시료 좌표 {len(OV_XY):,}곳 중 DB 좌표를 찾은 곳 {len(n_grid):,}곳 · 못 찾은 곳 {len(OV_XY) - len(n_grid):,}곳')
print(f'DB 좌표가 2곳 이상 걸린 시료 좌표 {(n_grid > 1).sum()}곳 → 그 값들의 평균을 쓴다')
row_loc = grid['loc'].to_numpy()[gid]
del gid
ok_row = (row_loc >= 0) & (day_no >= 0) & (day_no < N_DATES)
flat = row_loc[ok_row] * N_DATES + day_no[ok_row]

OV_VALUES = {}   # 해외 DB 칼럼 → (해외 좌표 × 날짜) 표, 원자료 단위 그대로
for c in OV_DB_COLS:
    v = pq.read_table(GFW_PATH, columns=[c]).column(0).to_numpy(zero_copy_only=False).astype(np.float64)[ok_row]
    has = ~np.isnan(v)
    s = np.bincount(flat[has], weights=v[has], minlength=len(OV_XY) * N_DATES)
    n = np.bincount(flat[has], minlength=len(OV_XY) * N_DATES)
    with np.errstate(invalid='ignore', divide='ignore'):
        OV_VALUES[c] = (s / n).astype(np.float32).reshape(len(OV_XY), N_DATES)   # 값이 없는 칸은 0/0 = NaN
    del v, s, n
del flat, ok_row, row_loc, day_no; gc.collect()

display(pd.DataFrame([{
    '해외 DB 칼럼': c, '값 있는 칸(%)': round(float((~np.isnan(T)).mean() * 100), 2),
    '원자료 중앙값': float(np.nanmedian(T)), '원자료 최댓값': float(np.nanmax(T)),
    '나누는 수': next(v['ov'][1] for v in VARIABLES if v['ov'] and v['ov'][0] == c),
    '바꾼 뒤 중앙값': round(float(np.nanmedian(T)) / next(v['ov'][1] for v in VARIABLES if v['ov'] and v['ov'][0] == c), 3),
    '바꾼 뒤 최댓값': round(float(np.nanmax(T)) / next(v['ov'][1] for v in VARIABLES if v['ov'] and v['ov'][0] == c), 3),
} for c, T in OV_VALUES.items()]))

해외 원자료 기간: 2015-01-01 ~ 2026-06-30 · 12,865,736행


원자료의 DB 좌표 3,064곳 · 해외 시료 좌표 3,062곳 중 DB 좌표를 찾은 곳 3,062곳 · 못 찾은 곳 0곳
DB 좌표가 2곳 이상 걸린 시료 좌표 2곳 → 그 값들의 평균을 쓴다


,해외 DB 칼럼,값 있는 칸(%),원자료 중앙값,원자료 최댓값,나누는 수,바꾼 뒤 중앙값,바꾼 뒤 최댓값
0,precipitation_sum,99.67,0.200000,541.599976,1.0,0.200,541.600
1,radiation_sum,99.67,14.960000,35.299999,1.0,14.960,35.300
2,sunshine_duration,99.67,35317.398438,84589.500000,3600.0,9.810,23.497
3,temperature_max,99.67,19.000000,49.000000,1.0,19.000,49.000
4,temperature_mean,99.67,14.500000,41.000000,1.0,14.500,41.000
5,temperature_min,99.67,10.600000,38.099998,1.0,10.600,38.100
6,wind_speed_max,99.67,16.799999,120.599998,3.6,4.667,33.500
7,wind_speed_mean,99.67,10.200000,79.800003,3.6,2.833,22.167


### 4-5. D-0~D-110 칼럼 만들기와 저장

행마다 (좌표, 기준일)을 알면 위 표에서 기준일 0~110일 전 값을 꺼내 칼럼 111개를 만든다. 변수 하나씩 계산해 `중간/wx_<변수>.parquet`(행 키 `lims_data_key` + 111칼럼, 시료 파일과 같은 행 순서)로 저장한다. 점검 표(국내·해외 값 있는 비율, 대체 칼럼을 쓴 비율, 고른 관측소까지 거리)는 `중간/join_summary.csv` 로 함께 저장한다.

이미 파일이 있고 행 수가 지금 시료 행 수와 같으면(`FORCE_JOIN = False`) 다시 계산하지 않고 점검 표만 읽는다. 행 수가 다르면(예: 시료 행 수가 바뀐 뒤) 다시 계산한다.

In [15]:
SUMMARY_PATH = MID_DIR / 'join_summary.csv'
KEYS = df[KEY_COL].astype(str).to_numpy()
row_di = (df[DATE_COL] - DATE_MIN).dt.days.to_numpy()
row_dom_loc = DOM_LOC.get_indexer(pd.MultiIndex.from_frame(XY.loc[IS_DOM, ['lat', 'lon']]))
row_ov_loc = OV_LOC.get_indexer(pd.MultiIndex.from_frame(XY.loc[~IS_DOM, ['lat', 'lon']]))
assert (row_dom_loc >= 0).all() and (row_ov_loc >= 0).all()
dom_cells = row_di[IS_DOM][:, None] - LAGS[None, :]   # (국내 행, 111) 날짜 번호
ov_cells = row_di[~IS_DOM][:, None] - LAGS[None, :]
assert dom_cells.min() >= 0 and ov_cells.min() >= 0 and dom_cells.max() < N_DATES and ov_cells.max() < N_DATES
N_DOM, N_OV = int(IS_DOM.sum()), int((~IS_DOM).sum())

def lagged(T, loc, cells):
    return T[loc[:, None], cells]

def join_variable(spec):
    """변수 하나를 조인해 중간 파일로 저장하고 점검 값을 돌려준다"""
    var, prim, fb, ov = spec['var'], spec['prim'], spec['fb'], spec['ov']
    t0 = time.time()
    net = NETS[prim[0]]
    Mp, Dp, redo_p = knn_mean(net['values'][prim[1]], net['periods'], DOM_XY.to_numpy(float))
    dom = lagged(Mp, row_dom_loc, dom_cells)
    dom_d = lagged(Dp, row_dom_loc, dom_cells)
    del Mp, Dp
    n_fb = 0
    if fb:
        netf = NETS[fb[0]]
        Mf, Df, _ = knn_mean(netf['values'][fb[1]], netf['periods'], DOM_XY.to_numpy(float))
        dom_f = lagged(Mf, row_dom_loc, dom_cells)
        use_fb = np.isnan(dom) & ~np.isnan(dom_f)
        n_fb = int(use_fb.sum())
        if n_fb:
            dom[use_fb] = dom_f[use_fb]
            dom_d[use_fb] = lagged(Df, row_dom_loc, dom_cells)[use_fb]
        del Mf, Df, dom_f, use_fb
    out = np.full((N_ROWS, len(LAGS)), np.nan, dtype=np.float32)
    out[IS_DOM] = dom
    if ov:
        out[~IS_DOM] = lagged(OV_VALUES[ov[0]], row_ov_loc, ov_cells) / np.float32(ov[1])
    table = pa.table({KEY_COL: pa.array(KEYS, type=pa.large_string()),
                      **{c: pa.array(out[:, j], from_pandas=True) for j, c in enumerate(wx_cols(var))}})   # 결측(NaN)은 null 로 적는다
    pq.write_table(table, MID_DIR / f'wx_{var}.parquet', row_group_size=ROW_GROUP)
    has_dom = ~np.isnan(dom)
    row = {
        '칼럼': var, '변수': spec['ko'], '단위': spec['unit'],
        '국내 값 있음(%)': round(float(has_dom.mean()) * 100, 3),
        '국내 중 대체 칼럼 값(%)': round(n_fb / (N_DOM * len(LAGS)) * 100, 3),
        '해외 값 있음(%)': round(float((~np.isnan(out[~IS_DOM])).mean()) * 100, 3) if ov else 0.0,
        '국내 값이 하나도 없는 행': int((~has_dom).all(axis=1).sum()),
        '고른 관측소까지 평균 거리 중앙값(km)': round(float(np.nanmedian(dom_d)), 2),
        '고른 관측소까지 평균 거리 90%(km)': round(float(np.nanpercentile(dom_d, 90)), 2),
        '평균 거리 50km 넘는 칸(%)': round(float((dom_d > 50).sum() / max(int(has_dom.sum()), 1) * 100), 2),
        '평균 거리 100km 넘는 칸(%)': round(float((dom_d > 100).sum() / max(int(has_dom.sum()), 1) * 100), 2),
        '평균 거리 최댓값(km)': round(float(np.nanmax(dom_d)), 1),
        '후보 60곳 밖에서 다시 고른 칸': redo_p,
        '초': round(time.time() - t0),
    }
    del out, dom, dom_d, table, has_dom; gc.collect()
    return row

def run_join(specs):
    """specs 중 중간 파일이 없는 변수만 조인하고, 점검 표(join_summary.csv)를 갱신해 돌려준다"""
    def stale(s):   # 중간 파일이 없거나, 행 수가 지금 시료 행 수와 다르면 다시 만든다
        p = MID_DIR / f"wx_{s['var']}.parquet"
        return not p.exists() or pq.ParquetFile(p).metadata.num_rows != N_ROWS
    todo = [s for s in specs if FORCE_JOIN or stale(s)]
    rows = []
    for s in todo:
        rows.append(join_variable(s))
        r = rows[-1]
        print(f"{s['var']:13s} 저장 ({r['초']}초) — 국내 {r['국내 값 있음(%)']}% · 해외 {r['해외 값 있음(%)']}%")
    old = pd.read_csv(SUMMARY_PATH) if SUMMARY_PATH.exists() else pd.DataFrame()
    if rows:
        new = pd.DataFrame(rows)
        old = pd.concat([old[~old['칼럼'].isin(new['칼럼'])] if len(old) else old, new], ignore_index=True)
        order = {v: i for i, v in enumerate(WX_VARS)}
        old = old.sort_values('칼럼', key=lambda s: s.map(order)).reset_index(drop=True)
        old.to_csv(SUMMARY_PATH, index=False, encoding='utf-8-sig')
    else:
        print('조인 결과가 이미 있고 행 수가 같아 다시 계산하지 않음 (FORCE_JOIN=False)')
    names = [s['var'] for s in specs]
    return old[old['칼럼'].isin(names)].reset_index(drop=True)

JOIN_SUMMARY_WX = run_join(VARIABLES)
display(JOIN_SUMMARY_WX.drop(columns=['초']))

avg_temp      저장 (28초) — 국내 100.0% · 해외 100.0%


max_temp      저장 (27초) — 국내 100.0% · 해외 100.0%


min_temp      저장 (26초) — 국내 100.0% · 해외 100.0%


rain          저장 (15초) — 국내 99.934% · 해외 100.0%


avg_wind      저장 (26초) — 국내 100.0% · 해외 100.0%


max_wind      저장 (27초) — 국내 100.0% · 해외 100.0%


... (8 outputs left out)

no2           저장 (15초) — 국내 100.0% · 해외 0.0%


co            저장 (15초) — 국내 100.0% · 해외 0.0%


so2           저장 (15초) — 국내 100.0% · 해외 0.0%


,칼럼,변수,단위,국내 값 있음(%),국내 중 대체 칼럼 값(%),해외 값 있음(%),국내 값이 하나도 없는 행,고른 관측소까지 평균 거리 중앙값(km),고른 관측소까지 평균 거리 90%(km),평균 거리 50km 넘는 칸(%),평균 거리 100km 넘는 칸(%),평균 거리 최댓값(km),후보 60곳 밖에서 다시 고른 칸
0,avg_temp,평균기온,°C,100.000,0.0,100.0,0,24.67,32.83,0.01,0.00,150.7,0
1,max_temp,최고기온,°C,100.000,0.0,100.0,0,24.67,32.83,0.01,0.00,113.0,0
2,min_temp,최저기온,°C,100.000,0.0,100.0,0,24.67,32.83,0.01,0.00,113.0,0
3,rain,강수량,mm,99.934,0.0,100.0,0,15.53,21.32,0.01,0.01,145.7,0
4,avg_wind,평균풍속,m/s,100.000,0.0,100.0,0,24.67,32.83,0.01,0.00,150.7,0
5,max_wind,최대풍속,m/s,100.000,0.0,100.0,0,24.67,32.83,0.01,0.00,113.0,0
6,sunshine,일조시간,시간,100.000,0.0,100.0,0,24.67,32.83,0.01,0.00,113.0,0
7,solar_rad,일사량,MJ/m²,100.000,0.0,100.0,0,33.44,46.47,5.54,0.00,194.9,0
8,soil_t5,지중온도 5cm,°C,100.000,0.0,0.0,0,49.93,76.45,49.96,0.93,214.6,0
9,soil_t10,지중온도 10cm,°C,100.000,0.0,0.0,0,49.93,76.45,49.94,0.92,214.6,0


- **국내 값 있음**: 국내 행 × 111일 칸 중 값이 있는 비율. 그날 전국에 값 있는 관측소가 한 곳도 없을 때만 빈다.
- **해외 값 있음**: 해외 행 × 111일. 해외 자료가 없는 변수는 0%.
- **대체 칼럼 값**: 국내 칸 중 기본 칼럼이 없어 대체 칼럼 값을 넣은 비율.
- **고른 관측소까지 거리**: 고른 관측소(최대 3곳)까지 평균 거리(구면 거리, km). 거리 상한이 없어, 값을 재는 관측소가 적은 변수는 먼 관측소가 뽑힌다(표의 "50km·100km 넘는 칸"과 "최댓값").
- ASOS·에어코리아·해외 자료는 2015-01-01 부터 있다. 농업기상(NAS) 표는 2008년부터 있지만 **2014년 행이 한 건도 없다**(아래 출력). 그래서 110일 전이 2015-01-01 보다 앞인 행은 그 날짜 칸이 모든 변수에서 빈다. 해당 행 수를 아래에 적는다.
- **대체 칼럼 규칙은 실제로는 한 칸도 쓰이지 않았다**(표의 "국내 중 대체 칼럼 값"이 모두 0). 2015년부터는 ASOS 기본 칼럼에 값 있는 관측소가 하루도 0곳이 된 적이 없고(아래 출력), 그 전 날짜에는 대체 자료(NAS)도 없기 때문이다.
- 농업기상(NAS)이 기본인 강수량·지중온도(NAS)·토양수분의 결측 칸은 거의 전부 **2016·2020·2024년 2월 29일** 칸이다. `api_data_info.nas_agri_weather_daily_info` 에 이 세 날짜의 값이 전 관측소에서 없다(아래 출력) — 왜 없는지는 확인 필요.

In [16]:
early = df[DATE_COL] - pd.Timedelta(days=LAG_MAX) < pd.Timestamp('2015-01-01')
print(f'D-{LAG_MAX} 이 2015-01-01 보다 앞인 행: {int(early.sum())}행')
display(df.loc[early, [KEY_COL, DATE_COL, 'src_route', TARGET_COL]].assign(
    **{'2015-01-01 보다 앞인 날짜 칸 수': ((pd.Timestamp('2015-01-01') - (df.loc[early, DATE_COL] - pd.Timedelta(days=LAG_MAX))).dt.days)}))

D-110 이 2015-01-01 보다 앞인 행: 1행


,lims_data_key,mnfctur_de,src_route,judge_result,2015-01-01 보다 앞인 날짜 칸 수
627023,O000043_190101350_1_5,2015-04-07,domestic,불검출,14


#### 결측 칸이 생기는 날, 관측소 수의 변화, 관측소까지 거리

In [17]:
def n_station_per_day(net, col):
    """날마다 그 칼럼 값이 있는 관측소 수"""
    return (~np.isnan(NETS[net]['values'][col])).sum(axis=1)

i2015 = didx('2015-01-01')
print('1) ASOS 기본 칼럼 — 2015-01-01 부터 날마다 값 있는 관측소 수의 최솟값:',
      {c: int(n_station_per_day('ASOS', c)[i2015:].min()) for c in ('avg_ta', 'max_ta', 'min_ta', 'avg_ws', 'max_ws', 'sum_sshr', 'sum_gsr', 'avg_cm5_te')})
print("   대체 칼럼 값을 넣은 칸의 비율(%) — 변수 전체 합:", float(JOIN_SUMMARY_WX['국내 중 대체 칼럼 값(%)'].sum()))
nas_year = pd.to_datetime(pd.read_parquet(RAW_DIR / 'raw_nas_agri_weather_daily_info.parquet', columns=['meas_date'])['meas_date']).dt.year.value_counts().sort_index()
print('   NAS 일자료의 연도별 행 수(2012~2016):', {int(k): int(v) for k, v in nas_year.loc[2012:2016].items()}, '← 2014년 없음')
n_rn = n_station_per_day('NAS', 'rn')
zero = n_rn == 0
feb29 = np.array([d.month == 2 and d.day == 29 for d in DATES])
print(f"2) NAS rn(강수량) 값 있는 관측소가 0곳인 날 {int(zero.sum())}일: 2월 29일 {[d.strftime('%Y-%m-%d') for d in DATES[zero & feb29]]}"
      f" · 그 밖 {DATES[zero & ~feb29].min().date()} ~ {DATES[zero & ~feb29].max().date()} ({int((zero & ~feb29).sum())}일)")
print(f'   국내 행 × {len(LAGS)}일 칸 중 그런 날에 걸린 칸: 2월 29일 {int((zero & feb29)[dom_cells].sum()):,}칸 · 그 밖 {int((zero & ~feb29)[dom_cells].sum()):,}칸')

print('3) 해마다 값 있는 관측소 수가 다르다 — 날마다 값 있는 관측소 수의 연도별 중앙값')
yr = DATES.year.to_numpy()
cnt = {label: pd.Series(n_station_per_day(net, col)).groupby(yr).median() for label, net, col in (
    ('평균기온 (ASOS avg_ta)', 'ASOS', 'avg_ta'), ('일사량 (ASOS sum_gsr)', 'ASOS', 'sum_gsr'), ('지중온도 5cm (ASOS avg_cm5_te)', 'ASOS', 'avg_cm5_te'),
    ('강수량 (NAS rn)', 'NAS', 'rn'), ('지중온도 (NAS soil_temp)', 'NAS', 'soil_temp'), ('토양수분 (NAS soil_wt)', 'NAS', 'soil_wt'),
    ('이산화질소 (에어코리아 no2_value)', 'AIR', 'no2_value'))}
display(pd.DataFrame(cnt).loc[2015:].T.astype(int))

print('4) 가까운 3곳이 서로 얼마나 떨어져 있나 (어림 — 관측소는 가장 최근 위치, 그날 값이 있는지는 따지지 않음)')
lx = DOM_XY.to_numpy(float)
n_loc_rows = np.bincount(row_dom_loc, minlength=len(DOM_XY))
st_xy = NETS['ASOS']['periods'][-1][2]
d_sorted = np.sort(haversine_km(lx[:, None, 0], lx[:, None, 1], st_xy[None, :, 0], st_xy[None, :, 1]), axis=1)
m = (d_sorted[:, 0] <= 10) & (d_sorted[:, 2] > 30)
print(f'   ASOS: 가장 가까운 관측소는 10km 이내인데 셋째로 가까운 관측소는 30km 넘는 국내 행 {int(n_loc_rows[m].sum()):,} ({n_loc_rows[m].sum() / N_DOM * 100:.1f}%)')
air_xy = NETS['AIR']['periods'][0][2]
near_air = np.argsort(haversine_km(lx[:, None, 0], lx[:, None, 1], air_xy[None, :, 0], air_xy[None, :, 1]), axis=1, kind='stable')[:, :K_NEAREST]
road = (AIR_MANG[near_air] == '도로변대기').any(axis=1)
road_port = np.isin(AIR_MANG[near_air], ['도로변대기', '항만']).any(axis=1)
print(f'   에어코리아: 가까운 3곳에 도로변대기 측정소가 1곳 이상인 국내 행 {int(n_loc_rows[road].sum()):,} ({n_loc_rows[road].sum() / N_DOM * 100:.1f}%)'
      f' · 도로변대기 또는 항만이 1곳 이상인 행 {int(n_loc_rows[road_port].sum()):,} ({n_loc_rows[road_port].sum() / N_DOM * 100:.1f}%)')
del d_sorted, near_air

1) ASOS 기본 칼럼 — 2015-01-01 부터 날마다 값 있는 관측소 수의 최솟값: {'avg_ta': 91, 'max_ta': 92, 'min_ta': 92, 'avg_ws': 81, 'max_ws': 92, 'sum_sshr': 91, 'sum_gsr': 36, 'avg_cm5_te': 23}
   대체 칼럼 값을 넣은 칸의 비율(%) — 변수 전체 합: 0.0


   NAS 일자료의 연도별 행 수(2012~2016): {2012: 32480, 2013: 42036, 2015: 54891, 2016: 61923} ← 2014년 없음
2) NAS rn(강수량) 값 있는 관측소가 0곳인 날 17일: 2월 29일 ['2016-02-29', '2020-02-29', '2024-02-29'] · 그 밖 2014-12-18 ~ 2014-12-31 (14일)


   국내 행 × 111일 칸 중 그런 날에 걸린 칸: 2월 29일 45,992칸 · 그 밖 14칸
3) 해마다 값 있는 관측소 수가 다르다 — 날마다 값 있는 관측소 수의 연도별 중앙값


,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
평균기온 (ASOS avg_ta),94,94,95,95,95,95,95,95,97,97,97,97
일사량 (ASOS sum_gsr),38,39,42,42,47,47,47,48,53,60,65,65
지중온도 5cm (ASOS avg_cm5_te),26,26,27,28,28,28,28,28,29,29,29,29
강수량 (NAS rn),140,155,170,178,185,188,189,191,191,193,199,200
지중온도 (NAS soil_temp),21,22,23,23,24,23,23,24,24,26,30,39
토양수분 (NAS soil_wt),87,73,61,59,127,126,126,123,123,124,129,128
이산화질소 (에어코리아 no2_value),301,303,308,351,422,498,570,606,628,643,651,661


4) 가까운 3곳이 서로 얼마나 떨어져 있나 (어림 — 관측소는 가장 최근 위치, 그날 값이 있는지는 따지지 않음)
   ASOS: 가장 가까운 관측소는 10km 이내인데 셋째로 가까운 관측소는 30km 넘는 국내 행 128,317 (20.4%)
   에어코리아: 가까운 3곳에 도로변대기 측정소가 1곳 이상인 국내 행 148,574 (23.6%) · 도로변대기 또는 항만이 1곳 이상인 행 173,713 (27.6%)


## 5. 조인 — 풍향 (최다풍향을 sin·cos 로 바꿔 가까운 3곳 평균)

- 원자료: `api_data_info.kma_asos_daily_info.max_ws1` — 테이블정의서 설명 "최다풍향"(DB 코멘트 없음). 값은 아래 출력처럼 20·50·70… 의 16가지다. 이 노트북은 이 값을 각도(도)로 보고 sin·cos 로 바꾼다. 값 사이 간격이 20도·30도로 고르지 않아, 정확한 각도인지 16방위를 10도 단위로 적은 값인지는 정의서에 설명이 없다 — 확인 필요.
- 각도는 그대로 평균 낼 수 없다(350°와 10°의 평균이 180°가 됨). 그래서 **관측소·날짜마다 sin(각도)·cos(각도) 두 값으로 바꾼 뒤**, 4절과 똑같이 "그날 값 있는 관측소 중 가까운 3곳 평균"을 sin·cos 각각 낸다(사용자 지시 2026-10-06).
- 칼럼: `wind_dir_sin_d000~d110`, `wind_dir_cos_d000~d110`. 북쪽(360°)은 sin 0 · cos 1, 동쪽(90°)은 sin 1 · cos 0.
- 평균 낸 뒤 다시 길이 1로 맞추지 않는다. 세 관측소의 풍향이 서로 다르면 (sin, cos)의 길이가 1보다 짧아진다.
- 해외: `api_data_info.global_factory_weather` 에 풍향 칼럼이 없어 해외 행은 결측. NAS `widdir`("풍향")은 지시에 없어 쓰지 않는다.

In [18]:
V_deg = NETS['ASOS']['values']['max_ws1']
vals, cnts = np.unique(V_deg[~np.isnan(V_deg)], return_counts=True)
print('max_ws1 값 종류(도):', vals.tolist())
print('값별 칸 수:', dict(zip(vals.astype(int).tolist(), cnts.tolist())))
print(f'값 있는 칸 {int((~np.isnan(V_deg)).sum()):,} / {V_deg.size:,} · 값 있는 관측소 {int((~np.isnan(V_deg)).any(axis=0).sum())}곳')
rad = np.deg2rad(V_deg.astype(np.float64))
NETS['ASOS']['values']['max_ws1_sin'] = np.sin(rad).astype(np.float32)   # NaN 은 NaN 그대로
NETS['ASOS']['values']['max_ws1_cos'] = np.cos(rad).astype(np.float32)
del rad

JOIN_SUMMARY_WD = run_join(WIND_DIR_VARIABLES)
display(JOIN_SUMMARY_WD.drop(columns=['초']))

max_ws1 값 종류(도): [20.0, 50.0, 70.0, 90.0, 110.0, 140.0, 160.0, 180.0, 200.0, 230.0, 250.0, 270.0, 290.0, 320.0, 340.0, 360.0]
값별 칸 수: {20: 27108, 50: 29345, 70: 20993, 90: 19434, 110: 18905, 140: 18070, 160: 18021, 180: 19500, 200: 20918, 230: 25760, 250: 28675, 270: 39788, 290: 41663, 320: 33525, 340: 20886, 360: 16559}
값 있는 칸 399,150 / 417,087 · 값 있는 관측소 99곳


wind_dir_sin  저장 (15초) — 국내 100.0% · 해외 0.0%


wind_dir_cos  저장 (15초) — 국내 100.0% · 해외 0.0%


,칼럼,변수,단위,국내 값 있음(%),국내 중 대체 칼럼 값(%),해외 값 있음(%),국내 값이 하나도 없는 행,고른 관측소까지 평균 거리 중앙값(km),고른 관측소까지 평균 거리 90%(km),평균 거리 50km 넘는 칸(%),평균 거리 100km 넘는 칸(%),평균 거리 최댓값(km),후보 60곳 밖에서 다시 고른 칸
0,wind_dir_sin,최다풍향 sin,(−1~1),100.0,0.0,0.0,0,24.67,32.83,0.01,0.0,150.7,0
1,wind_dir_cos,최다풍향 cos,(−1~1),100.0,0.0,0.0,0,24.67,32.83,0.01,0.0,150.7,0


### 5-1. 기상·대기·풍향 조인 점검 — 원자료로 다시 계산해 대조 (중간 파일 기준)

4·5절 계산(표로 한꺼번에 계산)이 규칙대로 됐는지, **`raw/` 원자료 파일에서 한 칸씩 직접 다시 계산**해 저장된 값과 비교한다. 4-2절에서 만든 (날짜 × 관측소) 표를 거치지 않고 원자료 행에서 바로 계산한다.

- 무작위 시료 50행(국내 45 · 해외 5, seed 42) × 변수 19개 × 날짜 4개(D-0·D-1·D-55·D-110) 칸
- 국내: 그날 원자료에서 값 있는 관측소를 실제 거리(km)순으로 정렬 → 앞 3곳 평균 (ASOS 는 그날 위치, 동방로거테스트 제외, 풍향은 관측소 값마다 sin·cos 로 바꾼 뒤 평균) → 기본이 없으면 대체 칼럼
- 해외: 원자료에서 같은 좌표·그날 행의 값 (풍속 ÷ 3.6, 일조 ÷ 3600)
- 여기서는 `중간/` 파일과 대조한다. 최종 `데이터셋.parquet` 와의 대조는 11절에서 같은 함수로 다시 한다.

In [19]:
CHECK_LAGS = (0, 1, 55, 110)
rng = np.random.default_rng(SEED)
CHECK_ROWS = np.concatenate([rng.choice(np.nonzero(IS_DOM)[0], 45, replace=False),
                             rng.choice(np.nonzero(~IS_DOM)[0], 5, replace=False)])
CHECK_DATES = sorted({df.at[r, DATE_COL] - pd.Timedelta(days=int(k)) for r in CHECK_ROWS for k in CHECK_LAGS})

# 원자료를 대조할 날짜만 남겨 날짜별로 나눠 둔다
RAW_NET = {'ASOS': ('raw_kma_asos_daily_info', 'stn_id', 'meas_date'),
           'NAS': ('raw_nas_agri_weather_daily_info', 'obsr_spot_code', 'meas_date'),
           'AIR': ('raw_keco_airkorea_fine_dust_info', 'station_nm', 'msur_dt')}
RAW_BY_DATE = {}
for net, (fname, sid_col, date_col) in RAW_NET.items():
    cols = sorted({s[k][1] for s in VARIABLES for k in ('prim', 'fb') if s[k] and s[k][0] == net} | ({'max_ws1'} if net == 'ASOS' else set()))
    r = pd.read_parquet(RAW_DIR / f'{fname}.parquet', columns=[sid_col, date_col] + cols)
    r['day_no'] = (to_date(r[date_col]) - DATE_MIN).dt.days          # 기상 기간 첫날부터 며칠째
    r = r[r['day_no'].isin({didx(d) for d in CHECK_DATES})].rename(columns={sid_col: 'sid'})
    r['sid'] = r['sid'].astype(str)
    if net == 'NAS':
        r = r[~r['sid'].isin(EXCLUDE_NAS)]          # 동방로거테스트는 쓰지 않는다
    RAW_BY_DATE[net] = {int(d): g.drop(columns=[date_col, 'day_no']) for d, g in r.groupby('day_no')}
del r

ASOS_ST_RAW = pd.read_parquet(RAW_DIR / 'raw_kma_ground_obs_station_info.parquet')
ASOS_ST_RAW['stn_id'] = ASOS_ST_RAW['stn_id'].astype(str)
ASOS_ST_RAW['start'] = to_date(ASOS_ST_RAW['oper_start_dt'])
NAS_ST_RAW = pd.read_parquet(RAW_DIR / 'raw_nas_agri_weather_obs_station.parquet')
AIR_ST_RAW = pd.read_parquet(RAW_DIR / 'raw_keco_airkorea_obs_station.parquet').drop_duplicates('station_nm')

_ST_CACHE = {}
def raw_station_xy(net, date):
    """원자료 관측소 표에서 그날의 관측소 좌표 (sid, 이름, 위도, 경도)"""
    key = (net, didx(date) if net == 'ASOS' else 0)
    if key not in _ST_CACHE:
        if net == 'ASOS':
            out = []
            for s, g in ASOS_ST_RAW.groupby('stn_id'):
                g = g.sort_values('start')
                ok = g[g['start'] <= date]
                r = ok.iloc[-1] if len(ok) else g.iloc[0]
                out.append((s, r['stn_ko'], float(r['lat']), float(r['lon'])))
            _ST_CACHE[key] = pd.DataFrame(out, columns=['sid', '이름', 'la', 'lo'])
        elif net == 'NAS':
            _ST_CACHE[key] = NAS_ST_RAW.rename(columns={'obsr_spot_code': 'sid', 'obsr_spot_nm': '이름', 'instl_la': 'la', 'instl_lo': 'lo'})[['sid', '이름', 'la', 'lo']]
        else:
            _ST_CACHE[key] = pd.DataFrame({'sid': AIR_ST_RAW['station_nm'], '이름': AIR_ST_RAW['station_nm'], 'la': AIR_ST_RAW['dm_x'], 'lo': AIR_ST_RAW['dm_y']})
    return _ST_CACHE[key]

def recompute_dom(lat, lon, date, net, col, k=K_NEAREST):
    """국내 한 칸을 원자료에서 직접 계산 — (평균, 고른 관측소 표)"""
    day = RAW_BY_DATE[net].get(didx(date))
    if day is None:
        return np.nan, pd.DataFrame()
    src = col.removesuffix('_sin').removesuffix('_cos')
    t = day[['sid', src]].copy()
    t['원자료 값'] = pd.to_numeric(t[src])
    t = t.dropna(subset=['원자료 값'])
    if col.endswith('_sin'):
        t['값'] = np.sin(np.deg2rad(t['원자료 값'].to_numpy(np.float64))).astype(np.float32)
    elif col.endswith('_cos'):
        t['값'] = np.cos(np.deg2rad(t['원자료 값'].to_numpy(np.float64))).astype(np.float32)
    else:
        t['값'] = t['원자료 값'].astype(np.float32)
    t = t.merge(raw_station_xy(net, pd.Timestamp(date)), on='sid', how='inner')   # 좌표 없는 측정소는 빠진다
    t['거리(km)'] = haversine_km(np.float64(lat), np.float64(lon), t['la'].to_numpy(float), t['lo'].to_numpy(float))
    t = t.sort_values('sid').sort_values('거리(km)', kind='stable').head(k)
    t.insert(0, '관측망', net)
    return (float(t['값'].astype(np.float64).mean()) if len(t) else np.nan), t[['관측망', 'sid', '이름', '거리(km)', '원자료 값', '값']]

# 해외: 대조할 해외 행의 좌표만 원자료에서 읽는다
OV_CHECK_RAW = {}
for r in CHECK_ROWS[~IS_DOM[CHECK_ROWS]]:
    la, lo = XY.at[r, 'lat'], XY.at[r, 'lon']
    (la_lo, la_hi), (lo_lo, lo_hi) = f32_bounds([la]), f32_bounds([lo])
    g = pq.read_table(GFW_PATH, filters=[('lat_g', '>=', la_lo[0]), ('lat_g', '<=', la_hi[0]),
                                         ('lon_g', '>=', lo_lo[0]), ('lon_g', '<=', lo_hi[0])]).to_pandas()
    g['date'] = to_date(g['obsr_de'])
    OV_CHECK_RAW[int(r)] = g

def recompute_ov(r, date, ov):
    g = OV_CHECK_RAW[int(r)]
    v = g.loc[g['date'] == pd.Timestamp(date), ov[0]].astype(np.float64)
    return float(v.mean()) / ov[1] if v.notna().any() else np.nan

def verify_weather(read_saved, label):
    """read_saved(var, cols) → 저장된 값 표(시료 파일과 같은 행 순서). 다시 계산한 값과 대조한다"""
    n_cell = n_same = n_nan = 0
    bad = []
    for spec in ALL_WX:
        var = spec['var']
        cols = [f'{var}_d{k:03d}' for k in CHECK_LAGS]
        saved = read_saved(var, cols)
        for r in CHECK_ROWS:
            lat, lon, base = XY.at[r, 'lat'], XY.at[r, 'lon'], df.at[r, DATE_COL]
            for k, c in zip(CHECK_LAGS, cols):
                date = base - pd.Timedelta(days=k)
                if IS_DOM[r]:
                    v, _ = recompute_dom(lat, lon, date, *spec['prim'])
                    if np.isnan(v) and spec['fb']:
                        v, _ = recompute_dom(lat, lon, date, *spec['fb'])
                else:
                    v = recompute_ov(r, date, spec['ov']) if spec['ov'] else np.nan
                s = float(saved[c][r])
                n_cell += 1
                same = (np.isnan(v) and np.isnan(s)) or (not np.isnan(v) and not np.isnan(s) and abs(np.float32(v) - s) <= 1e-5 * max(1, abs(v)))
                n_same += same
                n_nan += bool(np.isnan(s))
                if not same:
                    bad.append((var, int(r), k, v, s))
    print(f'[{label}] 대조한 칸 {n_cell:,}개 중 같은 칸 {n_same:,}개, 다른 칸 {len(bad)}개 (대조한 칸 중 결측 {n_nan:,}개 — 결측끼리도 같아야 같은 칸)')
    assert not bad, bad[:10]

def read_mid(var, cols):
    t = pq.read_table(MID_DIR / f'wx_{var}.parquet', columns=[KEY_COL] + cols)
    assert (t.column(KEY_COL).to_numpy(zero_copy_only=False) == KEYS).all(), f'{var}: 행 순서가 시료 파일과 다름'
    return {c: t.column(c).to_numpy() for c in cols}

verify_weather(read_mid, '중간 파일')

# 예시 1: 국내 시료 1건의 D-0 평균기온
r = int(CHECK_ROWS[0])
date = df.at[r, DATE_COL]
v, t = recompute_dom(XY.at[r, 'lat'], XY.at[r, 'lon'], date, 'ASOS', 'avg_ta')
print(f"\n예시 1 — {KEY_COL} {df.at[r, KEY_COL]} · 좌표 ({XY.at[r, 'lat']:.4f}, {XY.at[r, 'lon']:.4f}) · 기준일 {date.date()} (D-0) 평균기온")
display(t.assign(**{'거리(km)': t['거리(km)'].round(2)}))
print(f"세 관측소 평균 = {v:.4f} → 저장된 avg_temp_d000 = {read_mid('avg_temp', ['avg_temp_d000'])['avg_temp_d000'][r]}")

# 예시 2: 같은 시료의 D-0 풍향
vs, ts = recompute_dom(XY.at[r, 'lat'], XY.at[r, 'lon'], date, 'ASOS', 'max_ws1_sin')
vc, tc = recompute_dom(XY.at[r, 'lat'], XY.at[r, 'lon'], date, 'ASOS', 'max_ws1_cos')
print('\n예시 2 — 같은 시료의 D-0 최다풍향 (원자료 값 = 도)')
display(ts.rename(columns={'값': 'sin'}).assign(cos=tc['값'].to_numpy(), **{'거리(km)': ts['거리(km)'].round(2)}))
print(f"sin 평균 = {vs:.4f} → 저장된 wind_dir_sin_d000 = {read_mid('wind_dir_sin', ['wind_dir_sin_d000'])['wind_dir_sin_d000'][r]}"
      f" · cos 평균 = {vc:.4f} → 저장된 wind_dir_cos_d000 = {read_mid('wind_dir_cos', ['wind_dir_cos_d000'])['wind_dir_cos_d000'][r]}")

# 예시 3: 해외 시료 1건의 D-1 풍속·일조 (단위 변환)
r = int(CHECK_ROWS[~IS_DOM[CHECK_ROWS]][0])
date = df.at[r, DATE_COL] - pd.Timedelta(days=1)
g = OV_CHECK_RAW[r]
raw_row = g.loc[g['date'] == date, ['lat_g', 'lon_g', 'obsr_de', 'wind_speed_mean', 'wind_speed_max', 'sunshine_duration']]
print(f"\n예시 3 — 해외 {KEY_COL} {df.at[r, KEY_COL]} · 좌표 ({XY.at[r, 'lat']:.4f}, {XY.at[r, 'lon']:.4f}) · 기준일 {df.at[r, DATE_COL].date()} 의 D-1 = {date.date()}")
display(raw_row)
for var, col, div in (('avg_wind', 'wind_speed_mean', 3.6), ('max_wind', 'wind_speed_max', 3.6), ('sunshine', 'sunshine_duration', 3600)):
    print(f"  {col} {float(raw_row[col].iloc[0]):.3f} ÷ {div} = {float(raw_row[col].iloc[0]) / div:.4f} → 저장된 {var}_d001 = {read_mid(var, [f'{var}_d001'])[f'{var}_d001'][r]}")

[중간 파일] 대조한 칸 3,800개 중 같은 칸 3,800개, 다른 칸 0개 (대조한 칸 중 결측 220개 — 결측끼리도 같아야 같은 칸)

예시 1 — lims_data_key O000084_240101587_1_3 · 좌표 (37.4641, 126.4505) · 기준일 2024-05-08 (D-0) 평균기온


,관측망,sid,이름,거리(km),원자료 값,값
12,ASOS,112,인천,15.46,14.2,14.2
47,ASOS,201,강화,27.05,13.7,13.7
11,ASOS,108,서울,46.98,14.7,14.7


세 관측소 평균 = 14.2000 → 저장된 avg_temp_d000 = 14.199999809265137

예시 2 — 같은 시료의 D-0 최다풍향 (원자료 값 = 도)


,관측망,sid,이름,거리(km),원자료 값,sin,cos
12,ASOS,112,인천,15.46,50.0,0.766044,0.642788
47,ASOS,201,강화,27.05,50.0,0.766044,0.642788
11,ASOS,108,서울,46.98,50.0,0.766044,0.642788


sin 평균 = 0.7660 → 저장된 wind_dir_sin_d000 = 0.7660444378852844 · cos 평균 = 0.6428 → 저장된 wind_dir_cos_d000 = 0.6427876353263855

예시 3 — 해외 lims_data_key 1471111_220701807_1_6 · 좌표 (35.5641, 116.6518) · 기준일 2022-11-15 의 D-1 = 2022-11-14


,lat_g,lon_g,obsr_de,wind_speed_mean,wind_speed_max,sunshine_duration
2874,35.56411,116.65182,2022-11-14,8.1,10.7,34704.35


  wind_speed_mean 8.100 ÷ 3.6 = 2.2500 → 저장된 avg_wind_d001 = 2.250000238418579


  wind_speed_max 10.700 ÷ 3.6 = 2.9722 → 저장된 max_wind_d001 = 2.972222328186035
  sunshine_duration 34704.350 ÷ 3600 = 9.6401 → 저장된 sunshine_d001 = 9.640097618103027


### 5-2. 국내 값과 해외 값은 같은 칼럼에 있어도 성격이 다르다 · 해외 날짜는 UTC 기준

국내 행은 관측소 3곳의 관측값 평균이고, 해외 행은 `api_data_info.global_factory_weather`(DB 코멘트 "Open-Meteo Archive(ERA5) 해외 제조업소 좌표 일별 기상")의 그 좌표 값이다. 단위는 맞췄지만 자료의 종류가 달라, 같은 장소·같은 날에도 값이 다를 수 있다. 이를 재려고 **overseas 행 중 한국 주소가 로마자로 적힌 행**을 쓴다(`addr` 에 KOREA, 도 이름, 광역시 이름, `-EUP`·`-MYEON` 이 들어 있는 overseas 행 — 아래 출력에 행 수). 이 행들은 좌표가 한국 안에 있으면서 해외용 자료의 값이 붙어 있어, 같은 (좌표, 날짜)에서 두 값을 나란히 볼 수 있다.

- 견주는 칸: 이 행들 × 111일. 국내 값은 이 좌표에 4절의 국내 규칙(그날 값 있는 관측소 중 가까운 3곳 평균, 기본 칼럼)을 여기서 따로 적용해 구한 값(비교용 — 데이터셋에 넣지 않는다), 해외용 값은 4절에서 이 행들에 붙인 값(데이터셋에 들어가는 값. 풍속 ÷ 3.6, 일조 ÷ 3600).
- 해외 자료의 날짜 `obsr_de` 는 DB 코멘트가 "관측 일자(UTC 기준)"이다. 현지 시각이 UTC 보다 빠른 곳(경도가 큰 동쪽)은 그 날짜 칸에 현지 날짜로 다음 날 아침이 섞인다. 아래에서 최저기온으로 확인한다: 해외용 값이 같은 날짜의 국내 값과 더 맞는지, 다음 날짜의 국내 값과 더 맞는지.
- 이 차이를 보정하지는 않았다(사용자 결정 2026-10-06 — 규칙 유지).

In [20]:
# overseas 행 중 한국 주소가 로마자로 적힌 행 — 좌표가 한국 안이라 국내 규칙 값과 견줄 수 있다 (이 비교에만 쓴다. 데이터셋의 값은 바꾸지 않는다)
KR_WORDS = (r"KOREA|GYEONGGI|GYEONGSANG|JEOLLA|CHUNGCHEONG|GANGWON|JEJU"
            r"|SEOUL|BUSAN|INCHEON|DAEGU|DAEJEON|GWANGJU|ULSAN|-EUP\b|-MYEON\b")
kr_rows = np.nonzero(~IS_DOM & df['addr'].astype('string').str.upper().str.contains(KR_WORDS, regex=True).fillna(False).to_numpy())[0]
KR_XY = XY.iloc[kr_rows].drop_duplicates().reset_index(drop=True)
kr_loc = pd.MultiIndex.from_frame(KR_XY).get_indexer(pd.MultiIndex.from_frame(XY.iloc[kr_rows]))
kr_cells = (df[DATE_COL].iloc[kr_rows] - DATE_MIN).dt.days.to_numpy()[:, None] - LAGS[None, :]      # (행, 111일) 날짜 번호
kr_in = (kr_cells >= 0) & (kr_cells < N_DATES)                                          # 기상 기간 안의 칸
print(f'overseas 행 중 한국 주소가 로마자로 적힌 행 {len(kr_rows)} (검출 {int(y[kr_rows].sum())}) · 좌표 {len(KR_XY)}곳'
      f' · {len(kr_rows)}행 × {len(LAGS)}일 = {kr_cells.size:,}칸 중 기상 기간 밖이라 견주지 못하는 칸 {int((~kr_in).sum())}')
PRIM_OF = {v['var']: v['prim'] for v in VARIABLES}
PAIRS = [(v['var'], v['ko'], v['unit'], v['ov'][0], v['ov'][1]) for v in VARIABLES if v['ov']]

def corr_ok(a, b):
    ok = ~np.isnan(a) & ~np.isnan(b)
    return float(np.corrcoef(a[ok], b[ok])[0, 1])

cmp_rows, d0_rows, kept = [], [], {}
for var, ko, unit, col, div in PAIRS:
    t = pq.read_table(MID_DIR / f'wx_{var}.parquet', columns=wx_cols(var))
    d0 = t.column(f'{var}_d000').to_numpy()
    net = NETS[PRIM_OF[var][0]]
    M, _, _ = knn_mean(net['values'][PRIM_OF[var][1]], net['periods'], KR_XY.to_numpy(float))       # 이 행들의 좌표에 국내 규칙(기본 칼럼) 적용
    dom_v = np.where(kr_in, M[kr_loc[:, None], np.clip(kr_cells, 0, N_DATES - 1)], np.nan).astype(np.float64)   # (행, 111일) 국내 규칙 값
    del M
    ov_v = np.column_stack([t.column(c).to_numpy()[kr_rows] for c in wx_cols(var)]).astype(np.float64)   # (행, 111일) 4절에서 이 행들에 붙인 해외용 자료 값
    del t
    ok = ~np.isnan(dom_v) & ~np.isnan(ov_v)
    cmp_rows.append({'변수': ko, '단위': unit, '견준 칸': int(ok.sum()), '국내 규칙 값 평균': round(float(dom_v[ok].mean()), 3), '해외용 자료 값 평균': round(float(ov_v[ok].mean()), 3),
                     '해외용 ÷ 국내': round(float(ov_v[ok].mean() / dom_v[ok].mean()), 2), '상관계수': round(corr_ok(dom_v, ov_v), 3)})
    d0_rows.append({'변수': ko, '단위': unit, '국내 행 D-0 중앙값': round(float(np.nanmedian(d0[IS_DOM])), 3), '해외 행 D-0 중앙값': round(float(np.nanmedian(d0[~IS_DOM])), 3)})
    kept[var] = (dom_v, ov_v)
print(f'1) 같은 한국 좌표 {len(KR_XY)}곳(overseas 행 중 한국 주소가 로마자로 적힌 {len(kr_rows)}행의 좌표·기준일) · 같은 날짜에서 두 자료의 값')
display(pd.DataFrame(cmp_rows))
print('2) 데이터셋의 기준일 당일(D-0) 값 — 국내 행과 해외 행 (장소가 다르므로 참고)')
display(pd.DataFrame(d0_rows))

dom_v, ov_v = kept['min_temp']
print('3) 최저기온 — 해외용 자료의 날짜 t 값과 국내 값의 상관계수: '
      f'국내 같은 날짜 t {corr_ok(ov_v[:, 1:], dom_v[:, 1:]):.3f} · 국내 다음 날짜 t+1 {corr_ok(ov_v[:, 1:], dom_v[:, :-1]):.3f}')
lon_ov = XY['lon'].to_numpy()[~IS_DOM].astype(float)
off = lon_ov / 15                                                     # 경도로 어림한 UTC 와의 시차(시간)
y_ov = y[~IS_DOM]
print(f'4) 해외 행 {len(lon_ov):,}행(검출 {int(y_ov.sum())}) 중 경도로 어림한 시차가 +6시간 이상인 행 {int((off >= 6).sum()):,} (검출 {int(((off >= 6) & y_ov).sum())})'
      f' · −4시간 이하인 행 {int((off <= -4).sum()):,} (검출 {int(((off <= -4) & y_ov).sum())})')
del kept, NETS, OV_VALUES; gc.collect();

지운 78행 × 111일 = 8,658칸 중 기상 기간 밖이라 견주지 못하는 칸 0


1) 같은 한국 좌표 9곳(2절에서 지운 78행의 좌표·기준일) · 같은 날짜에서 두 자료의 값


,변수,단위,견준 칸,국내 규칙 값 평균,해외용 자료 값 평균,해외용 ÷ 국내,상관계수
0,평균기온,°C,8658,12.266,11.201,0.91,0.991
1,최고기온,°C,8658,17.914,16.084,0.90,0.991
2,최저기온,°C,8658,7.499,7.121,0.95,0.964
3,강수량,mm,8652,3.097,3.410,1.10,0.715
4,평균풍속,m/s,8658,1.754,2.565,1.46,0.767
5,최대풍속,m/s,8658,4.314,4.315,1.00,0.773
6,일조시간,시간,8658,6.413,8.350,1.30,0.853
7,일사량,MJ/m²,8658,13.992,14.355,1.03,0.957


2) 데이터셋의 기준일 당일(D-0) 값 — 국내 행과 해외 행 (장소가 다르므로 참고)


,변수,단위,국내 행 D-0 중앙값,해외 행 D-0 중앙값
0,평균기온,°C,13.933,13.700
1,최고기온,°C,20.167,18.300
2,최저기온,°C,8.333,9.600
3,강수량,mm,0.000,0.100
4,평균풍속,m/s,1.600,2.722
5,최대풍속,m/s,4.100,4.528
6,일조시간,시간,7.433,9.713
7,일사량,MJ/m²,14.057,14.460


3) 최저기온 — 해외용 자료의 날짜 t 값과 국내 값의 상관계수: 국내 같은 날짜 t 0.964 · 국내 다음 날짜 t+1 0.986
4) 해외 행 47,564행(검출 125) 중 경도로 어림한 시차가 +6시간 이상인 행 14,088 (검출 56) · −4시간 이하인 행 9,286 (검출 11)


## 6. 조인 — 식품 (수분활성도·식품pH·염도)

품목코드로 붙인다. 자료는 두 가지다.

| 자료 | 내용 |
|---|---|
| `ba_data.t_aw_cl` (`raw_t_aw_cl`) | 품목코드 `prdlst_cd` 별 `aw`(DB 코멘트 "수분활성도")·`ph`("식품pH")·`na`("염도"). 값은 텍스트 |
| 리서치팀 엑셀 `리서치팀문서/통합_수분활성도_수분함량_식품pH_염도_최종.xlsx` 의 `값` 시트 | 품목코드 `PRDLST_CD` 별 `수분활성도`·`식품 pH`·`염도`. `출처` 시트에 값마다 문헌이 적혀 있다 |

### 조인 규칙

| 규칙 | 내용 |
|---|---|
| 조인 키 순서 (사용자 결정 2026-10-06) | 시료의 `prdlst_code`(원본 식품공전 품목코드) 정확 일치 → `prdlst_code` 앞 14자리(20자리 코드만 해당 — 앞 14자리를 쓰는 근거는 6-2절의 "20자리 품목코드" 항목) → `train_prdlst_code`(학습용 품목코드, 상위 품목 단계) → 그래도 없으면 결측. 값마다 따로 적용. 구체적인 코드부터 본다 |
| 두 자료 중 어느 값 (사용자 결정 2026-10-06) | 품목코드마다 **엑셀에 값이 있으면 엑셀 값**, 없으면 `ba_data.t_aw_cl` 값. 어떤 값이 엑셀에서 왔는지는 6-3절에 적는다 |
| 값 바꾸기 (기존 규칙 그대로) | 숫자는 그대로, **범위**(`0.93~0.98`)는 **가운데 값**, **부등호**(`<0.98`, `≥0.98`)는 **경계값**, 그 밖(`-` 등)은 결측 |
| 값 표기 칼럼 (기존 규칙 그대로) | `aw_label`·`ph_label`·`na_label`: **단일값** · **중앙**(범위의 가운데 값) · **이상**(`≥`) · **이하**(`≤`) · **초과**(`>`) · **미만**(`<`). 값이 없으면 비어 있다 |
| 엑셀에만 있는 표기 | 범위 기호 `–`·`∼` 는 `~` 와 같은 범위로, `0.98 이상`·`0.65 이하` 는 부등호로 읽는다 (같은 규칙에 표기만 추가) |
| 결측 | 채우지 않는다 |
| 값이 어디서 왔는지 (2026-10-06 추가) | 데이터셋에 `aw_key`·`ph_key`·`na_key`(어느 조인 키로 붙었는지), `aw_src`·`ph_src`·`na_src`(엑셀 / t_aw_cl), `aw_source`(값을 `ba_data.t_aw_cl` 에서 가져온 행의 `t_aw_cl.aw_source` 원값)를 함께 넣는다 |

- 세 값의 단위는 `ba_data.t_aw_cl` 의 DB 코멘트와 엑셀 머리글에 없다 — 확인 필요. (`예측인자 정리 ppt.jpg` 의 생장 한계 표에는 염도가 "≤16%"로 적혀 있다.)
- `ba_data.t_aw_cl.aw_source` 는 DB 코멘트가 없다 — 뜻 확인 필요. 값은 `original`·`recomputed(...)`·`derived_median(...)`·`mapped_from:...`·`assigned_...` 꼴이다(6-4절).
- 값은 **품목코드 단위 값**이다. 그 시료를 잰 값이 아니다. `train_prdlst_code` 로 붙은 행은 상위 품목 단계의 값이다.

In [21]:
FOOD_PRIORITY = 'excel'      # 'excel' = 엑셀에 값이 있으면 엑셀, 없으면 t_aw_cl / 't_aw_cl' = t_aw_cl 에 값이 있으면 t_aw_cl, 없으면 엑셀
FOOD_KEY_ORDER = ['prdlst_code', 'prdlst_code 앞 14자리', 'train_prdlst_code']   # 조인 키를 보는 순서 (구체적인 코드부터)
XLSX_PATH = ROOT / '리서치팀문서' / '통합_수분활성도_수분함량_식품pH_염도_최종.xlsx'
FOOD_COLS = ['aw', 'ph', 'na']
FOOD_KO = {'aw': '수분활성도', 'ph': '식품pH', 'na': '염도'}
FOOD_LABELS = ['단일값', '중앙', '이상', '이하', '초과', '미만']

NUM = r'-?\d+(?:\.\d+)?'
RANGE_SEP = r'[~～∼\-–]'
SIGN_LABEL = {'<': '미만', '>': '초과', '≤': '이하', '≥': '이상', '<=': '이하', '>=': '이상'}
SIGN_FLIP = {'미만': '초과', '초과': '미만', '이하': '이상', '이상': '이하'}   # 부등호가 숫자 뒤에 올 때 (예: `0.98<` = 0.98 초과)

def parse_food(s):
    """원문 → (숫자, 적용 규칙, 값 표기). 값 표기: 단일값 · 중앙 · 이상 · 이하 · 초과 · 미만. 값이 없으면 ''"""
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return np.nan, '빈칸', ''
    if isinstance(s, (int, float, np.integer, np.floating)) and not isinstance(s, bool):
        return float(s), '숫자 그대로', '단일값'       # 엑셀의 숫자 칸
    t = str(s).strip()
    if t == '':
        return np.nan, '빈칸', ''
    if re.fullmatch(NUM, t):
        return float(t), '숫자 그대로', '단일값'
    m = re.fullmatch(rf'({NUM})\s*{RANGE_SEP}\s*({NUM})', t)
    if m:
        return (float(m[1]) + float(m[2])) / 2, '범위 → 가운데 값', '중앙'
    m = re.fullmatch(rf'([<>≤≥]=?)\s*({NUM})', t)
    if m:
        return float(m[2]), '부등호 → 경계값', SIGN_LABEL[m[1]]
    m = re.fullmatch(rf'({NUM})\s*([<>≤≥]=?)', t)
    if m:
        return float(m[1]), '부등호 → 경계값', SIGN_FLIP[SIGN_LABEL[m[2]]]
    m = re.fullmatch(rf'({NUM})\s*(이상|이하|초과|미만)', t)
    if m:
        return float(m[1]), '부등호 → 경계값', m[2]
    return np.nan, '해석 못 함 → 결측', ''

def raw_text(s):
    """원문을 보여 줄 때 쓰는 글자 (빈칸은 None)"""
    return None if s is None or (isinstance(s, float) and np.isnan(s)) or str(s).strip() == '' else str(s).strip()

aw_cl = pd.read_parquet(RAW_DIR / 'raw_t_aw_cl.parquet')
assert aw_cl['prdlst_cd'].is_unique
aw_cl = aw_cl.set_index('prdlst_cd')
xl = pd.read_excel(XLSX_PATH, sheet_name='값', dtype=object).rename(
    columns={'PRDLST_CD': 'prdlst_cd', 'KOR_NM': 'kor_nm', 'LV': 'lv', '수분활성도': 'aw', '수분함량': 'mc', '식품 pH': 'ph', '염도': 'na'})
assert xl['prdlst_cd'].is_unique
xl = xl.set_index('prdlst_cd')
print(f'ba_data.t_aw_cl: 품목코드 {len(aw_cl):,}개 · 엑셀 `값` 시트: 품목코드 {len(xl):,}개')
print(f'엑셀 코드 중 t_aw_cl 에도 있는 코드 {int(xl.index.isin(aw_cl.index).sum()):,}개 · 엑셀에만 있는 코드 {int((~xl.index.isin(aw_cl.index)).sum())}개 · t_aw_cl 에만 있는 코드 {int((~aw_cl.index.isin(xl.index)).sum()):,}개')
print('코드 첫 글자별 개수 — t_aw_cl:', pd.Series(aw_cl.index.str[0]).value_counts().to_dict(), '· 엑셀:', pd.Series(xl.index.str[0]).value_counts().to_dict())

ba_data.t_aw_cl: 품목코드 1,316개 · 엑셀 `값` 시트: 품목코드 653개


엑셀 코드 중 t_aw_cl 에도 있는 코드 653개 · 엑셀에만 있는 코드 0개 · t_aw_cl 에만 있는 코드 663개


코드 첫 글자별 개수 — t_aw_cl: {'A': 909, 'C': 309, 'B': 66, 'E': 11, 'Z': 9, 'D': 8, 'G': 2, 'X': 2} · 엑셀: {'A': 314, 'C': 275, 'B': 64}


### 6-1. 품목코드별 값 표 — 두 자료의 값을 숫자로 바꾸고 비교

품목코드마다 두 자료의 원문을 위 규칙으로 숫자·값 표기로 바꾼다. 그다음 두 자료를 비교해 (가) 숫자가 다른 코드, (나) 숫자는 같고 값 표기가 다른 코드(부등호 방향·등호 유무), (다) 한쪽에만 값이 있는 코드를 센다.

In [22]:
k1 = df['prdlst_code'].astype(str)
k14 = k1.str[:14]
k2 = df['train_prdlst_code'].astype(str)
n_k1 = k1.value_counts(); n_k2 = k2.value_counts()
print('시료 prdlst_code 길이별 행 수:', k1.str.len().value_counts().to_dict())

CODE_TABLE = {}   # 값 → 품목코드별 표
parse_rows, diff_rows = [], []
for c in FOOD_COLS:
    T = pd.DataFrame(index=aw_cl.index.union(xl.index))
    for src, frame in (('db', aw_cl), ('xl', xl)):
        p = frame[c].map(parse_food)
        T[f'{src}_raw'] = frame[c].map(raw_text)
        T[f'{src}_num'] = p.str[0]
        T[f'{src}_rule'] = p.str[1]
        T[f'{src}_label'] = p.str[2]
        for rule, n in p.str[1].value_counts().items():
            parse_rows.append({'값': FOOD_KO[c], '자료': 't_aw_cl' if src == 'db' else '엑셀', '규칙': rule, '품목코드 수': n})
    T['kor_nm'] = aw_cl['kor_nm'].reindex(T.index).fillna(xl['kor_nm'].reindex(T.index))
    T['db_source'] = aw_cl['aw_source'].reindex(T.index)          # t_aw_cl.aw_source 원값 (수분활성도 값의 출처 표시)
    both = T['db_num'].notna() & T['xl_num'].notna()
    T['비교'] = np.select(
        [both & ~np.isclose(T['db_num'].fillna(0), T['xl_num'].fillna(0)), both & (T['db_label'] != T['xl_label']), both,
         T['db_num'].notna(), T['xl_num'].notna()],
        ['(가) 숫자가 다름', '(나) 숫자 같고 값 표기가 다름', '같음', '(다) t_aw_cl 에만 값', '(다) 엑셀에만 값'], '둘 다 값 없음')
    T['시료 행(prdlst_code)'] = n_k1.reindex(T.index).fillna(0).astype(int)
    T['시료 행(train_prdlst_code)'] = n_k2.reindex(T.index).fillna(0).astype(int)
    CODE_TABLE[c] = T
    for name, g in T.groupby('비교'):
        diff_rows.append({'값': FOOD_KO[c], '비교': name, '품목코드 수': len(g), '시료 행(prdlst_code 일치)': int(g['시료 행(prdlst_code)'].sum()),
                          '시료 행(train_prdlst_code 일치)': int(g['시료 행(train_prdlst_code)'].sum())})
print('원문 → 숫자 규칙별 품목코드 수')
display(pd.DataFrame(parse_rows).pivot_table(index='규칙', columns=['값', '자료'], values='품목코드 수', aggfunc='sum', fill_value=0))
bad = pd.concat([T.loc[T[f'{s}_rule'] == '해석 못 함 → 결측', [f'{s}_raw']].rename(columns={f'{s}_raw': '원문'}).assign(값=FOOD_KO[c], 자료=s)
                 for c, T in CODE_TABLE.items() for s in ('db', 'xl')])
print('해석 못 해 결측으로 둔 원문:', bad.groupby(['값', '자료', '원문']).size().to_dict())
print('\n두 자료 비교 (품목코드 기준)')
display(pd.DataFrame(diff_rows).set_index(['값', '비교']))

show = ['kor_nm', 'db_raw', 'xl_raw', 'db_num', 'xl_num', 'db_label', 'xl_label', '시료 행(prdlst_code)', '시료 행(train_prdlst_code)']
ren = {'kor_nm': '품목 이름', 'db_raw': 't_aw_cl 원문', 'xl_raw': '엑셀 원문', 'db_num': 't_aw_cl 숫자', 'xl_num': '엑셀 숫자', 'db_label': 't_aw_cl 표기', 'xl_label': '엑셀 표기'}
A = CODE_TABLE['aw']
print('\n수분활성도 — (가) 숫자가 다른 코드')
display(A.loc[A['비교'] == '(가) 숫자가 다름', show].rename(columns=ren).sort_values('시료 행(prdlst_code)', ascending=False))
print('수분활성도 — (나) 숫자는 같고 값 표기가 다른 코드 (원문 짝별 코드 수·시료 행)')
g = A[A['비교'] == '(나) 숫자 같고 값 표기가 다름']
display(g.groupby(['db_raw', 'xl_raw', 'db_label', 'xl_label']).agg(품목코드=('kor_nm', 'size'), 예=('kor_nm', lambda s: ', '.join(s.head(4))),
        **{'시료 행(prdlst_code)': ('시료 행(prdlst_code)', 'sum'), '시료 행(train_prdlst_code)': ('시료 행(train_prdlst_code)', 'sum')})
        .rename_axis(['t_aw_cl 원문', '엑셀 원문', 't_aw_cl 표기', '엑셀 표기']).sort_values('시료 행(prdlst_code)', ascending=False))
for c in ('ph', 'na'):
    T = CODE_TABLE[c]
    print(f"{FOOD_KO[c]} — 두 자료에 다 값이 있는 코드 {int((T['db_num'].notna() & T['xl_num'].notna()).sum())}개 중 숫자 또는 값 표기가 다른 코드 {int(T['비교'].isin(['(가) 숫자가 다름', '(나) 숫자 같고 값 표기가 다름']).sum())}개")
odd = A[(A['db_num'] < 0) | (A['db_num'] > 1) | (A['xl_num'] < 0) | (A['xl_num'] > 1)]
print(f'\n수분활성도 숫자가 0~1 밖인 코드 {len(odd)}개 (고치지 않고 그대로 둔다):')
display(odd[show].rename(columns=ren))

시료 prdlst_code 길이별 행 수: {14: 642474, 20: 33985}


원문 → 숫자 규칙별 품목코드 수


값             수분활성도         식품pH           염도     
자료          t_aw_cl   엑셀 t_aw_cl   엑셀 t_aw_cl   엑셀
규칙                                                
범위 → 가운데 값       91   95     212  212     106  106
부등호 → 경계값       140  139       2    2       0    0
빈칸               45    1     934    1     816    0
숫자 그대로          980  230     167  167     394  394
해석 못 함 → 결측      60  188       1  271       0  153

해석 못 해 결측으로 둔 원문: {('수분활성도', 'db', '-'): 60, ('수분활성도', 'xl', '-'): 188, ('식품pH', 'db', '6..00~6.70'): 1, ('식품pH', 'xl', '-'): 270, ('식품pH', 'xl', '6..00~6.70'): 1, ('염도', 'xl', '-'): 153}

두 자료 비교 (품목코드 기준)


품목코드 수  시료 행(prdlst_code 일치)  시료 행(train_prdlst_code 일치)
값     비교                                                                          
수분활성도 (가) 숫자가 다름               8                  6323                           0
      (나) 숫자 같고 값 표기가 다름      49                 44836                        6398
      (다) t_aw_cl 에만 값       747                 43159                      294113
      같음                     407                152540                       44724
      둘 다 값 없음               105                  3547                      330929
식품pH  같음                     381                212104                      113661
      둘 다 값 없음               935                 38301                      562503
염도    같음                     500                222821                      136602
      둘 다 값 없음               816                 27584                      539562


수분활성도 — (가) 숫자가 다른 코드


,품목 이름,t_aw_cl 원문,엑셀 원문,t_aw_cl 숫자,엑셀 숫자,t_aw_cl 표기,엑셀 표기,시료 행(prdlst_code),시료 행(train_prdlst_code)
prdlst_cd,,,,,,,,,
C0316040000000,건조저장육류,0.15,0.85-0.93,0.150,0.890,단일값,중앙,5852,0
C0319030100000,조미건어포,0.15,0.6~0.85,0.150,0.725,단일값,중앙,291,0
C0308030000000,건면,0.15,0.51,0.150,0.510,단일값,단일값,70,0
A0600100200000,양배추,0.997,0.975,0.997,0.975,단일값,단일값,53,0
C0319030200000,건어포,0.15,0.6~0.85,0.150,0.725,단일값,중앙,44,0
C0319030300000,기타 건포류,0.15,0.48~0.66,0.150,0.570,단일값,중앙,11,0
B0206010500000,소라,>0.98,≥0.95,0.980,0.950,초과,이상,2,0
A0600202000000,치커리(잎),0.992,0.986,0.992,0.986,단일값,단일값,0,0


수분활성도 — (나) 숫자는 같고 값 표기가 다른 코드 (원문 짝별 코드 수·시료 행)


품목코드                            예  시료 행(prdlst_code)  시료 행(train_prdlst_code)
t_aw_cl 원문 엑셀 원문   t_aw_cl 표기 엑셀 표기                                                                               
<0.98      ≤0.98   미만         이하        1                       즉석섭취식품              33526                        0
           0.98<   미만         초과        6       우유, 환원유, 강화우유, 유산균첨가우유              10952                        0
<0.65      0.65 이하 미만         이하        1                          침출차                182                        0
>0.98      ≥0.98   초과         이상       39  기타포유류고기, 가금류, 오리고기, 기타가금류고기                176                        0
>0.92      ≥0.92   초과         이상        1                         로열젤리                  0                        0
>0.98      0.98 이상 초과         이상        1                          음료류                  0                     6398

식품pH — 두 자료에 다 값이 있는 코드 381개 중 숫자 또는 값 표기가 다른 코드 0개
염도 — 두 자료에 다 값이 있는 코드 500개 중 숫자 또는 값 표기가 다른 코드 0개

수분활성도 숫자가 0~1 밖인 코드 1개 (고치지 않고 그대로 둔다):


,품목 이름,t_aw_cl 원문,엑셀 원문,t_aw_cl 숫자,엑셀 숫자,t_aw_cl 표기,엑셀 표기,시료 행(prdlst_code),시료 행(train_prdlst_code)
prdlst_cd,,,,,,,,,
C0315020200000,영양강화 밀가루,-0.85,-,-0.85,NaN,단일값,,8,0


### 6-2. 시료 행에 붙이기

`food_join(우선 자료, 조인 키 순서)` 가 규칙대로 붙인다. 정한 규칙대로 붙인 결과를 저장하고, 규칙이 달랐다면 달라졌을 행 수도 참고로 적는다.

In [23]:
KEY_SERIES = {'prdlst_code': k1, 'train_prdlst_code': k2, 'prdlst_code 앞 14자리': k14}

def food_join(priority, key_order):
    """시료 행 순서의 표: 값·값 표기 + 어느 키·어느 품목코드·어느 자료·어떤 원문에서 왔는지"""
    levels = [(name, KEY_SERIES[name]) for name in key_order]
    out = pd.DataFrame(index=df.index)
    for c in FOOD_COLS:
        T = CODE_TABLE[c]
        first, second = ('xl', 'db') if priority == 'excel' else ('db', 'xl')
        use_first = T[f'{first}_num'].notna()
        code_num = T[f'{first}_num'].where(use_first, T[f'{second}_num'])
        code_label = T[f'{first}_label'].where(use_first, T[f'{second}_label'])
        code_raw = T[f'{first}_raw'].where(use_first, T[f'{second}_raw'])
        code_rule = T[f'{first}_rule'].where(use_first, T[f'{second}_rule'])
        code_src = pd.Series(np.where(use_first, first, second), index=T.index).map({'db': 't_aw_cl', 'xl': '엑셀'})
        code_ok = code_num.notna()
        hit_code = pd.Series(None, index=df.index, dtype=object)
        hit_key = pd.Series('', index=df.index, dtype=object)
        for name, k in levels:
            take = hit_code.isna() & k.isin(code_ok.index[code_ok]).to_numpy()
            hit_code[take] = k[take]
            hit_key[take] = name
        out[c] = hit_code.map(code_num).astype('float32')
        out[f'{c}_label'] = hit_code.map(code_label).fillna('')
        out[f'{c}_key'] = hit_key
        out[f'{c}_code'] = hit_code
        out[f'{c}_src'] = hit_code.map(code_src).fillna('')
        out[f'{c}_raw'] = hit_code.map(code_raw)
        out[f'{c}_rule'] = hit_code.map(code_rule).fillna('')
        if c == 'aw':   # 값을 t_aw_cl 에서 가져온 행에만 t_aw_cl.aw_source 를 남긴다
            out['aw_source'] = hit_code.map(T['db_source']).where(out['aw_src'] == 't_aw_cl')
    return out

food = food_join(FOOD_PRIORITY, FOOD_KEY_ORDER)
assert len(food) == N_ROWS

FOOD_SUMMARY = pd.DataFrame([{
    '칼럼': c, '값': FOOD_KO[c], '값 있음(%)': pct(food[c].notna()),
    'prdlst_code 로(%)': pct(food[f'{c}_key'] == 'prdlst_code'),
    '앞 14자리로(%)': pct(food[f'{c}_key'] == 'prdlst_code 앞 14자리'),
    'train_prdlst_code 로(%)': pct(food[f'{c}_key'] == 'train_prdlst_code'),
    't_aw_cl 값(%)': pct(food[f'{c}_src'] == 't_aw_cl'), '엑셀 값(%)': pct(food[f'{c}_src'] == '엑셀'),
    '검출 행 값 있음(%)': pct(food[c].notna(), y), '불검출 행 값 있음(%)': pct(food[c].notna(), ~y),
    '국내 값 있음(%)': pct(food[c].notna(), IS_DOM), '해외 값 있음(%)': pct(food[c].notna(), ~IS_DOM),
} for c in FOOD_COLS])
print(f"규칙: 우선 자료 = {'엑셀' if FOOD_PRIORITY == 'excel' else 't_aw_cl'} · 조인 키 순서 = {' → '.join(FOOD_KEY_ORDER)}")
display(FOOD_SUMMARY)

print('값 표기별 시료 행 수')
display(pd.DataFrame({f'{c}_label': food[f'{c}_label'].replace('', '(값 없음)').value_counts() for c in FOOD_COLS})
        .reindex(FOOD_LABELS + ['(값 없음)']).fillna(0).astype(int))
print('적용 규칙별 시료 행 수')
display(pd.DataFrame({FOOD_KO[c]: food[f'{c}_rule'].replace('', '(값 없음)').value_counts() for c in FOOD_COLS}).fillna(0).astype(int))

# (참고) 규칙이 달랐다면 달라졌을 행 수
cmp_rows = []
for label, alt in (("t_aw_cl 값을 먼저 썼다면", food_join('t_aw_cl', FOOD_KEY_ORDER)),
                   ("앞 14자리 단계가 없었다면", food_join(FOOD_PRIORITY, ['prdlst_code', 'train_prdlst_code'])),
                   ("train_prdlst_code 를 앞 14자리보다 먼저 봤다면 (처음 정했던 순서)", food_join(FOOD_PRIORITY, ['prdlst_code', 'train_prdlst_code', 'prdlst_code 앞 14자리']))):
    for c in FOOD_COLS:
        a, b = food[c], alt[c]
        dv = ~((a.isna() & b.isna()) | np.isclose(a.fillna(0), b.fillna(0)) & a.notna() & b.notna())
        dl = (food[f'{c}_label'] != alt[f'{c}_label']).to_numpy() & ~dv.to_numpy()
        cmp_rows.append({'달랐다면': label, '값': FOOD_KO[c], '값 있음(%) (지금 → 달랐다면)': f"{pct(a.notna())} → {pct(b.notna())}",
                         '숫자가 달라지는 행': int(dv.sum()), '그중 검출': int((dv.to_numpy() & y).sum()),
                         '숫자는 같고 값 표기만 달라지는 행': int(dl.sum())})
    del alt
display(pd.DataFrame(cmp_rows).set_index(['달랐다면', '값']))

print('값이 없는 행 — prdlst_code 첫 글자별 (행 수, 괄호 = 검출)')
first = k1.str[0]
display(pd.DataFrame({'행': first.value_counts(), '검출': pd.Series(y).groupby(first.to_numpy()).sum(),
                      **{f'{FOOD_KO[c]} 결측': pd.Series(food[c].isna().to_numpy()).groupby(first.to_numpy()).sum() for c in FOOD_COLS},
                      **{f'{FOOD_KO[c]} 결측 중 검출': pd.Series(food[c].isna().to_numpy() & y).groupby(first.to_numpy()).sum() for c in FOOD_COLS}}).sort_index())

규칙: 우선 자료 = 엑셀 · 조인 키 순서 = prdlst_code → prdlst_code 앞 14자리 → train_prdlst_code


,칼럼,값,값 있음(%),prdlst_code 로(%),앞 14자리로(%),train_prdlst_code 로(%),t_aw_cl 값(%),엑셀 값(%),검출 행 값 있음(%),불검출 행 값 있음(%),국내 값 있음(%),해외 값 있음(%)
0,aw,수분활성도,51.04,36.49,2.84,11.70,17.05,33.98,27.32,51.10,47.34,99.96
1,ph,식품pH,40.51,31.36,2.46,6.70,0.00,40.51,15.49,40.59,37.54,79.79
2,na,염도,41.67,32.94,2.74,5.99,0.00,41.67,15.95,41.74,38.74,80.46


값 표기별 시료 행 수


,aw_label,ph_label,na_label
단일값,180550,45102,112719
중앙,106314,228952,169160
이상,9662,2,0
이하,33708,0,0
초과,14716,0,0
미만,286,0,0
(값 없음),331223,402403,394580


적용 규칙별 시료 행 수


,수분활성도,식품pH,염도
(값 없음),331223,402403,394580
범위 → 가운데 값,106314,228952,169160
부등호 → 경계값,58372,2,0
숫자 그대로,180550,45102,112719


값 있음(%) (지금 → 달랐다면)  숫자가 달라지는 행  그중 검출  숫자는 같고 값 표기만 달라지는 행
달랐다면                                            값                                                                
t_aw_cl 값을 먼저 썼다면                               수분활성도       51.04 → 51.04        6371      5                46997
                                                식품pH        40.51 → 40.51           0      0                    0
                                                염도          41.67 → 41.67           0      0                    0
앞 14자리 단계가 없었다면                                 수분활성도       51.04 → 51.04       18991     40                    0
                                                식품pH        40.51 → 39.51       16623     42                    0
                                                염도          41.67 → 40.92       16522     42                    0
train_prdlst_code 를 앞 14자리보다 먼저 봤다면 (처음 정했던 순서) 수분활성도       51.04 → 51.04       18991     40                    0
                                                식품pH        40.51 → 40.51        9816     17                    0
                                                염도          41.67 → 41.67       11417     17                    0

값이 없는 행 — prdlst_code 첫 글자별 (행 수, 괄호 = 검출)


,행,검출,수분활성도 결측,식품pH 결측,염도 결측,수분활성도 결측 중 검출,식품pH 결측 중 검출,염도 결측 중 검출
A,2124,15,26,825,646,0,3,2
B,10191,74,0,8123,8136,0,47,47
C,333002,449,221,62313,54656,0,183,175
D,139,0,139,139,139,0,0,0
E,448,0,285,448,448,0,0,0
G,9640,0,9640,9640,9640,0,0,0
X,365,0,362,365,365,0,0,0
Z,320550,1431,320550,320550,320550,1431,1431,1431


#### 값이 없는 행은 어떤 행인가 — 품목코드가 Z 로 시작하는 행

예측인자 우선순위 1·3·4위(수분활성도·pH·염도)가 검출 행 대부분에서 비어 있다. 이유는 **검출의 대부분이 품목코드가 Z(기준규격외)로 시작하는 행**이고, 이 코드들은 `ba_data.t_aw_cl`·엑셀 어디에도 값이 없기 때문이다. 값이 없다는 사실 자체가 검출 여부와 함께 움직인다는 점을 아래 수치로 적어 둔다.

- `Z2700400000000` 의 품목명(`train_prdlst_nm`)은 "정액,수정란,종란"인데 제품명은 팽이버섯·작업장 표면 등이다(아래 출력). 이 코드·품목명이 맞는 것인지, 바닥·벽 같은 환경 검체가 예측 대상에 드는지는 **식정원 확인 필요**.

In [24]:
z_row = k1.str.startswith('Z').to_numpy()
no_food = food[FOOD_COLS].isna().all(axis=1).to_numpy()
print(f"prdlst_code 가 Z 로 시작하는 행 {int(z_row.sum()):,} ({pct(z_row)}%) · 검출 {int((z_row & y).sum()):,} (전체 검출의 {(z_row & y).sum() / y.sum() * 100:.1f}%)"
      f" · 그중 수분활성도·식품pH·염도가 모두 빈 행 {int((z_row & no_food).sum()):,}")
display(pd.DataFrame({'prdlst_code': k1.to_numpy()[z_row], 'train_prdlst_nm': df['train_prdlst_nm'].astype(str).to_numpy()[z_row], '검출': y[z_row]})
        .groupby(['prdlst_code', 'train_prdlst_nm']).agg(행=('검출', 'size'), 검출=('검출', 'sum')).sort_values('행', ascending=False))
has_aw_ = food['aw'].notna().to_numpy()
print(f'수분활성도 값이 없는 행의 검출 비율 {y[~has_aw_].mean() * 100:.3f}% ({int(y[~has_aw_].sum()):,} / {int((~has_aw_).sum()):,})'
      f' · 값이 있는 행의 검출 비율 {y[has_aw_].mean() * 100:.3f}% ({int(y[has_aw_].sum()):,} / {int(has_aw_.sum()):,})')
z27 = (k1 == 'Z2700400000000').to_numpy()
nm27 = df.loc[z27, 'prduct_nm'].astype('string')
print(f"Z2700400000000 — {int(z27.sum()):,}행 · 검출 {int((z27 & y).sum()):,} · 제품명에 '팽이'가 든 행 {int(nm27.str.contains('팽이').fillna(False).sum()):,}"
      f"(검출 {int((nm27.str.contains('팽이').fillna(False).to_numpy() & y[z27]).sum())}) · '표면'이 든 행 {int(nm27.str.contains('표면').fillna(False).sum()):,} · 많이 나오는 제품명:")
display(nm27.value_counts().head(8).rename('행').to_frame())

prdlst_code 가 Z 로 시작하는 행 320,550 (47.39%) · 검출 1,431 (전체 검출의 72.7%) · 그중 수분활성도·식품pH·염도가 모두 빈 행 320,550


,,행,검출
prdlst_code,train_prdlst_nm,,
Z3800100000000,기타기준규격외,291813,791
Z2700400000000,"정액,수정란,종란",28651,640
Z3000200000000,수산가공,40,0
Z4100000000000,환경검체,35,0
Z0105500000000,농산물 종자류,8,0
Z3700100000000,인삼.홍삼제품류,3,0


수분활성도 값이 없는 행의 검출 비율 0.432% (1,431 / 331,223) · 값이 있는 행의 검출 비율 0.156% (538 / 345,236)
Z2700400000000 — 28,651행 · 검출 640 · 제품명에 '팽이'가 든 행 4,249(검출 254) · '표면'이 든 행 748 · 많이 나오는 제품명:


,행
prduct_nm,
수출용 팽이버섯 (2-1번),101
수출용 팽이버섯 (2-2번),101
수출용 팽이버섯 (2번),100
수출용 팽이버섯 (1번),100
수출용 팽이버섯 (3-2번),96
수출용 팽이버섯 (3-1번),96
ENOKI MUSHROOM,67
권지,48


#### 20자리 품목코드(대부분 수입 행)는 어느 키로 붙었나 — 조인 키 순서를 바꾼 이유

- 20자리 `prdlst_code` 는 두 자료의 코드(14자리)와 정확히 일치할 수 없다. 그래서 앞 14자리를 그 품목의 코드로 보고 맞춘다.
- **앞 14자리를 쓰는 근거** — 문서에 적힌 규칙은 없다. 테이블정의서의 `prdlst_code` 설명은 "원본 식품공전 품목코드"뿐이고, 20자리 코드가 어떻게 이루어졌는지는 정의서·DB 코멘트에 없다. 아래는 2026-10-06 에 값을 대조해 확인한 것이다.
  - 모델팀 전처리 표의 `ready_data.hazard_preprocessed_data.mapped_prdlst_code`(정의서 설명 "구→신 코드 치환 결과")가, 리스테리아의 20자리 코드 행 34,380행 전부에서 `prdlst_code` 의 앞 14자리와 같다(DB 조회). 모델팀도 20자리 코드를 앞 14자리로 줄여 쓴다.
  - 시료 파일의 20자리 코드 33,992행 중 28,172행(82.9%)은 뒤 6자리가 `000000` 이다. 식품코드 표 `ba_data.t_prdlst_cl` 의 코드는 14자리이고, 20자리 전체가 이 표에 있는 행은 0행, 앞 14자리가 이 표에 있는 행은 27,135행(79.8%)이다.
  - 뒤 6자리가 무슨 뜻인지, 앞 14자리로 줄여 써도 되는지에 대한 식정원·식약처의 설명은 확인하지 못했다 — 확인 필요.
- 처음에는 순서를 정확 일치 → `train_prdlst_code` → 앞 14자리로 정했다. 그 순서에서는 20자리 행에 **자기 품목(앞 14자리) 값이 있어도 상위 품목 단계(`train_prdlst_code`)의 값이 먼저 붙었다**(수분활성도는 20자리 행 대부분이 상위 품목 값 0.839 하나가 됨 — 2026-10-06 검수에서 지적).
- 그래서 **정확 일치 → 앞 14자리 → `train_prdlst_code`** 로 바꿨다(사용자 결정 2026-10-06). 아래는 바꾼 뒤의 결과와, 처음 순서였다면 달라졌을 행이다.

In [25]:
long20 = (k1.str.len() == 20).to_numpy()
key_rows = []
for c in FOOD_COLS:
    key_c = food[f'{c}_key'].to_numpy()
    key_rows.append({'값': FOOD_KO[c], '20자리 코드 행': int(long20.sum()), '앞 14자리 값이 붙은 행': int((long20 & (key_c == 'prdlst_code 앞 14자리')).sum()),
                     'train_prdlst_code 값이 붙은 행': int((long20 & (key_c == 'train_prdlst_code')).sum()), '값 없는 행': int((long20 & food[c].isna().to_numpy()).sum())})
print(f"20자리 prdlst_code {int(long20.sum()):,}행 · 그중 해외 {int((long20 & ~IS_DOM).sum()):,}행 (해외 행의 {(long20 & ~IS_DOM).sum() / (~IS_DOM).sum() * 100:.1f}%)")
display(pd.DataFrame(key_rows))
print('20자리 행의 수분활성도 값 상위:', {round(float(k), 4): int(v) for k, v in pd.Series(food['aw'].to_numpy()[long20]).round(4).value_counts().head(5).items()})

old = food_join(FOOD_PRIORITY, ['prdlst_code', 'train_prdlst_code', 'prdlst_code 앞 14자리'])      # 처음 정했던 순서
chg = long20 & ~np.isclose(food['aw'].fillna(-9).to_numpy(), old['aw'].fillna(-9).to_numpy())
print(f'처음 순서였다면 수분활성도 값이 달랐을 20자리 행: {int(chg.sum()):,} (검출 {int((chg & y).sum())}) — 행이 많은 품목 10개:')
display(pd.DataFrame({'앞 14자리': k14.to_numpy()[chg], '품목 이름': k14.map(CODE_TABLE['aw']['kor_nm']).to_numpy()[chg], '지금 붙은 값(자기 품목)의 원문': food['aw_raw'].to_numpy()[chg],
                      '지금 붙은 값': food['aw'].to_numpy()[chg].round(4), '처음 순서였다면(상위 품목 값)': old['aw'].to_numpy()[chg].round(4), '검출': y[chg]})
        .groupby(['앞 14자리', '품목 이름', '지금 붙은 값(자기 품목)의 원문', '지금 붙은 값', '처음 순서였다면(상위 품목 값)']).agg(행=('검출', 'size'), 검출=('검출', 'sum'))
        .sort_values('행', ascending=False).head(10))
del old

20자리 prdlst_code 33,985행 · 그중 해외 33,970행 (해외 행의 71.4%)


,값,20자리 코드 행,앞 14자리 값이 붙은 행,train_prdlst_code 값이 붙은 행,값 없는 행
0,수분활성도,33985,19234,14747,4
1,식품pH,33985,16623,11104,6258
2,염도,33985,18516,9510,5959


20자리 행의 수분활성도 값 상위: {0.839: 9317, 0.98: 5647, 0.945: 4296, 0.955: 3258, 0.975: 2343}


처음 순서였다면 수분활성도 값이 달랐을 20자리 행: 18,991 (검출 40) — 행이 많은 품목 10개:


,,,,,행,검출
앞 14자리,품목 이름,지금 붙은 값(자기 품목)의 원문,지금 붙은 값,처음 순서였다면(상위 품목 값),,
C0318090300000,치즈,0.91~0.98,0.9450,0.839,4296,17
C0318090200000,가공치즈,0.93~0.98,0.9550,0.839,2501,0
C0302010100000,아이스크림,0.900~0.999,0.9495,0.839,1551,0
C0318080100000,버터,0.80~0.99,0.8950,0.839,1384,0
C0316010100000,햄,0.976,0.9760,0.839,971,4
C0318140000000,유함유가공품,0.37,0.3700,0.839,915,0
C0316020100000,소시지,0.975,0.9750,0.839,721,2
C0316050100000,양념육,0.93-0.98,0.9550,0.839,719,5
C0318070200000,가공유크림,0.97,0.9700,0.839,654,0


### 6-3. 어떤 값이 엑셀에서 왔는지 (사용자 지시 2026-10-06)

시료에 실제로 붙은 값이 두 자료 중 어디에서 왔는지 적는다.

1. 값별·자료별 시료 행 수
2. 엑셀 값이 `ba_data.t_aw_cl` 값을 **대신한** 품목코드 — 두 자료에 다 값이 있는데 숫자나 값 표기가 달라 엑셀 값을 쓴 코드 (6-1절의 (가)·(나))
3. 시료에 붙은 품목코드 전체 목록 — 코드마다 어느 자료의 어떤 원문을 썼는지

행마다의 기록은 데이터셋의 `aw_src`·`ph_src`·`na_src`(자료)·`aw_key`·`ph_key`·`na_key`(어느 키로 붙었는지)·`aw_source` 칼럼과, `중간/food.parquet` 의 `aw_code`(값을 가져온 품목코드)·`aw_raw`(원문) 칼럼에 있다.

In [26]:
src_rows = []
for c in FOOD_COLS:
    for src in ('엑셀', 't_aw_cl'):
        m = (food[f'{c}_src'] == src).to_numpy()
        src_rows.append({'값': FOOD_KO[c], '자료': src, '시료 행': int(m.sum()), '전체 중(%)': pct(m), '검출 행': int((m & y).sum()),
                         '품목코드 수': int(food.loc[m, f'{c}_code'].nunique())})
    m = food[c].isna().to_numpy()
    src_rows.append({'값': FOOD_KO[c], '자료': '(값 없음)', '시료 행': int(m.sum()), '전체 중(%)': pct(m), '검출 행': int((m & y).sum()), '품목코드 수': 0})
print('1) 값별·자료별 시료 행 수')
display(pd.DataFrame(src_rows).set_index(['값', '자료']))

print('2) 엑셀 값이 t_aw_cl 값을 대신한 품목코드 (시료에 붙은 코드만)')
rep_rows = []
for c in FOOD_COLS:
    T = CODE_TABLE[c]
    used = food[f'{c}_code'].value_counts()
    rep = T[T['비교'].isin(['(가) 숫자가 다름', '(나) 숫자 같고 값 표기가 다름']) & T.index.isin(used.index)]
    for code, r in rep.iterrows():
        rep_rows.append({'값': FOOD_KO[c], '품목코드': code, '품목 이름': r['kor_nm'], '무엇이 다른가': r['비교'], 't_aw_cl 원문': r['db_raw'], '엑셀 원문 (쓴 값)': r['xl_raw'],
                         't_aw_cl 숫자·표기': f"{r['db_num']:g} · {r['db_label']}", '엑셀 숫자·표기 (쓴 값)': f"{r['xl_num']:g} · {r['xl_label']}", '붙은 시료 행': int(used[code])})
REPLACED = pd.DataFrame(rep_rows).sort_values(['값', '무엇이 다른가', '붙은 시료 행'], ascending=[True, True, False]).reset_index(drop=True)
with pd.option_context('display.max_rows', None):
    display(REPLACED)
print(f"엑셀 값이 t_aw_cl 값을 대신한 품목코드 {len(REPLACED)}개 · 시료 {int(REPLACED['붙은 시료 행'].sum()):,}행"
      f" (숫자가 달라진 시료 {int(REPLACED.loc[REPLACED['무엇이 다른가'] == '(가) 숫자가 다름', '붙은 시료 행'].sum()):,}행)")

print('3) 시료에 붙은 품목코드 전체 목록')
for c in FOOD_COLS:
    g = (food[food[c].notna()].groupby([f'{c}_code', f'{c}_src', f'{c}_raw', f'{c}_label'])
         .agg(숫자=(c, 'first'), 시료_행=(c, 'size')).reset_index()
         .rename(columns={f'{c}_code': '품목코드', f'{c}_src': '자료', f'{c}_raw': '원문', f'{c}_label': '값 표기', '시료_행': '붙은 시료 행'}))
    g.insert(1, '품목 이름', g['품목코드'].map(CODE_TABLE[c]['kor_nm']))
    g = g.sort_values(['자료', '붙은 시료 행'], ascending=[True, False]).reset_index(drop=True)
    print(f"[{FOOD_KO[c]}] 품목코드 {len(g)}개 — 엑셀 값 {int((g['자료'] == '엑셀').sum())}개 · t_aw_cl 값 {int((g['자료'] == 't_aw_cl').sum())}개")
    with pd.option_context('display.max_rows', None):
        display(g)

1) 값별·자료별 시료 행 수


시료 행  전체 중(%)  검출 행  품목코드 수
값     자료                                    
수분활성도 엑셀       229879    33.98   312     259
      t_aw_cl  115357    17.05   226     105
      (값 없음)   331223    48.96  1431       0
식품pH  엑셀       274056    40.51   305     241
      t_aw_cl       0     0.00     0       0
      (값 없음)   402403    59.49  1664       0
염도    엑셀       281879    41.67   314     296
      t_aw_cl       0     0.00     0       0
      (값 없음)   394580    58.33  1655       0

2) 엑셀 값이 t_aw_cl 값을 대신한 품목코드 (시료에 붙은 코드만)


,값,품목코드,품목 이름,무엇이 다른가,t_aw_cl 원문,엑셀 원문 (쓴 값),t_aw_cl 숫자·표기,엑셀 숫자·표기 (쓴 값),붙은 시료 행
0,수분활성도,C0316040000000,건조저장육류,(가) 숫자가 다름,0.15,0.85-0.93,0.15 · 단일값,0.89 · 중앙,5894
1,수분활성도,C0319030100000,조미건어포,(가) 숫자가 다름,0.15,0.6~0.85,0.15 · 단일값,0.725 · 중앙,291
2,수분활성도,C0308030000000,건면,(가) 숫자가 다름,0.15,0.51,0.15 · 단일값,0.51 · 단일값,70
3,수분활성도,A0600100200000,양배추,(가) 숫자가 다름,0.997,0.975,0.997 · 단일값,0.975 · 단일값,53
4,수분활성도,C0319030200000,건어포,(가) 숫자가 다름,0.15,0.6~0.85,0.15 · 단일값,0.725 · 중앙,44
5,수분활성도,C0319030300000,기타 건포류,(가) 숫자가 다름,0.15,0.48~0.66,0.15 · 단일값,0.57 · 중앙,11
6,수분활성도,B0206010500000,소라,(가) 숫자가 다름,>0.98,≥0.95,0.98 · 초과,0.95 · 이상,8
7,수분활성도,C0322020100000,즉석섭취식품,(나) 숫자 같고 값 표기가 다름,<0.98,≤0.98,0.98 · 미만,0.98 · 이하,33526
8,수분활성도,C0318010100000,우유,(나) 숫자 같고 값 표기가 다름,<0.98,0.98<,0.98 · 미만,0.98 · 초과,8644
9,수분활성도,C0309000000000,음료류,(나) 숫자 같고 값 표기가 다름,>0.98,0.98 이상,0.98 · 초과,0.98 · 이상,1654


엑셀 값이 t_aw_cl 값을 대신한 품목코드 34개 · 시료 53,368행 (숫자가 달라진 시료 6,371행)
3) 시료에 붙은 품목코드 전체 목록
[수분활성도] 품목코드 364개 — 엑셀 값 259개 · t_aw_cl 값 105개


,품목코드,품목 이름,자료,원문,값 표기,숫자,붙은 시료 행
0,C0318000000000,유가공품류,t_aw_cl,0.839,단일값,0.8390,34997
1,C0317010700000,알가열제품,t_aw_cl,0.91,단일값,0.9100,19818
2,C0316000000000,식육가공품 및 포장육,t_aw_cl,0.839,단일값,0.8390,16923
3,C0323000000000,기타식품류,t_aw_cl,0.4,단일값,0.4000,5394
4,C0302010400000,샤베트,t_aw_cl,0.975,단일값,0.9750,4022
5,C0319000000000,수산가공식품류,t_aw_cl,0.815,단일값,0.8150,3385
6,C0318140000000,유함유가공품,t_aw_cl,0.37,단일값,0.3700,2727
7,C0317000000000,알가공품류,t_aw_cl,0.839,단일값,0.8390,2589
8,C0310000000000,특수영양식품,t_aw_cl,0.839,단일값,0.8390,2140
9,C0316010200000,생햄,t_aw_cl,0.975,단일값,0.9750,2116


[식품pH] 품목코드 241개 — 엑셀 값 241개 · t_aw_cl 값 0개


,품목코드,품목 이름,자료,원문,값 표기,숫자,붙은 시료 행
0,C0318000000000,유가공품류,엑셀,3.25~7.44,중앙,5.345,43506
1,C0322020100000,즉석섭취식품,엑셀,4.4~5.9,중앙,5.150,33526
2,C0316050100000,양념육,엑셀,6.5~7.0,중앙,6.750,22990
3,C0316010100000,햄,엑셀,6.03~6.54,중앙,6.285,14494
4,C0318040200000,농후발효유,엑셀,3.80~4.56,중앙,4.180,14443
5,C0316020100000,소시지,엑셀,5.47,단일값,5.470,13716
6,C0318090200000,가공치즈,엑셀,5.6~6.0,중앙,5.800,10798
7,C0318010100000,우유,엑셀,6.55~6.83,중앙,6.690,8644
8,C0318090300000,치즈,엑셀,4.88~7.44,중앙,6.160,7839
9,C0316010300000,프레스햄,엑셀,6.33~6.42,중앙,6.375,7561


[염도] 품목코드 296개 — 엑셀 값 296개 · t_aw_cl 값 0개


,품목코드,품목 이름,자료,원문,값 표기,숫자,붙은 시료 행
0,C0318000000000,유가공품류,엑셀,0.035~0.216,중앙,0.125500,38454
1,C0322020100000,즉석섭취식품,엑셀,1.03,단일값,1.030000,33526
2,C0316050100000,양념육,엑셀,0.10~2.06,중앙,1.080000,22990
3,C0316010100000,햄,엑셀,0.006176,단일값,0.006176,14494
4,C0318040200000,농후발효유,엑셀,0.064~0.216,중앙,0.140000,14443
5,C0316020100000,소시지,엑셀,0.0054102,단일값,0.005410,13716
6,C0318090200000,가공치즈,엑셀,0.03~8.86,중앙,4.445000,10798
7,C0318010100000,우유,엑셀,0.127,단일값,0.127000,8644
8,C0318090300000,치즈,엑셀,0.4~2.4,중앙,1.400000,7839
9,C0316010300000,프레스햄,엑셀,1.59512,단일값,1.595120,7561


### 6-4. 식품 값에서 설명이 필요한 점 (2026-10-06 추가)

값을 고치지는 않는다. 보고서에 쓸 때 설명이 필요한 점을 수치로 적어 둔다.

1. **같은 수분활성도 값이 여러 품목코드에 똑같이 들어 있다** (`ba_data.t_aw_cl`). 많이 겹치는 값과, 그 코드들에 엑셀 값이 있는지, 시료에 얼마나 붙었는지를 적는다. 이 값들이 어디서 온 것인지는 DB 코멘트·테이블정의서에 설명이 없다 — 확인 필요.
2. **`t_aw_cl.aw_source`** — 값 출처를 적은 것으로 보이는 칼럼(DB 코멘트 없음). 값 종류와, 시료에 붙은 수분활성도 값이 어느 종류에서 왔는지를 적는다.
3. **검출 행의 수분활성도가 어느 키로 붙었는지** — `train_prdlst_code` 로 붙은 값은 그 품목의 값이 아니라 상위 품목 단계의 값이다.
4. **범위를 벗어난 값** — 수분활성도가 0~1 밖인 행, 염도 최댓값.

In [27]:
A = CODE_TABLE['aw']
print('1) t_aw_cl 에서 같은 수분활성도 원문이 5개 이상의 품목코드에 들어 있는 경우')
rep_rows = []
for raw, n in aw_cl['aw'].value_counts().items():
    if n < 5 or raw == '-':
        continue
    codes = aw_cl.index[aw_cl['aw'] == raw]
    T = A.loc[codes]
    m = (food['aw_src'] == 't_aw_cl').to_numpy() & (food['aw_raw'] == raw).to_numpy()
    rep_rows.append({'t_aw_cl 원문': raw, '품목코드 수': n, '단계(lv)별 코드 수': aw_cl.loc[codes, 'lv'].astype(int).value_counts().sort_index().to_dict(),
                     'aw_source': aw_cl.loc[codes, 'aw_source'].fillna('(빈 값)').str.replace(r'[(:].*', '', regex=True).str.strip().value_counts().to_dict(),
                     '엑셀에 코드가 있는 수': int(codes.isin(xl.index).sum()), '그중 엑셀 값이 있는 수': int(T['xl_num'].notna().sum()),
                     '엑셀 값이 다른 수': int((T['비교'] == '(가) 숫자가 다름').sum()),
                     '이 값(t_aw_cl)이 붙은 시료 행': int(m.sum()), '그중 검출': int((m & y).sum())})
with pd.option_context('display.max_colwidth', 60):
    display(pd.DataFrame(rep_rows))
for raw in ('0.839', '0.15'):
    codes = aw_cl.index[aw_cl['aw'] == raw]
    T = A.loc[codes, ['kor_nm', 'db_raw', 'xl_raw', 'db_source', '시료 행(prdlst_code)', '시료 행(train_prdlst_code)']].rename(
        columns={'kor_nm': '품목 이름', 'db_raw': 't_aw_cl 원문', 'xl_raw': '엑셀 원문(없으면 빈칸)', 'db_source': 'aw_source'})
    T.insert(1, 'lv', aw_cl.loc[codes, 'lv'].astype(int))
    T.insert(3, '엑셀에 코드가 있는가', codes.isin(xl.index))
    show = T if len(T) <= 12 else T[T['엑셀 원문(없으면 빈칸)'].notna() | (T['시료 행(prdlst_code)'] + T['시료 행(train_prdlst_code)'] > 500)]
    print(f"t_aw_cl 원문이 '{raw}' 인 품목코드 {len(T)}개" + ('' if len(T) <= 12 else f' — 엑셀 값이 있거나 시료 500행 넘게 걸리는 {len(show)}개만 표시'))
    display(show.sort_values('시료 행(train_prdlst_code)', ascending=False))

print('2) t_aw_cl.aw_source 값 종류 (괄호·쌍점 뒤는 줄임) — 품목코드 수')
src_kind = aw_cl['aw_source'].fillna('(빈 값)').str.replace(r'[(:].*', '', regex=True).str.strip()
display(src_kind.value_counts().rename('품목코드 수').to_frame().T)
used_kind = food['aw_source'].fillna('').astype(str).str.replace(r'[(:].*', '', regex=True).str.strip()
t = pd.DataFrame({'자료': food['aw_src'].replace('', '(값 없음)').to_numpy(), 'aw_source 종류': used_kind.to_numpy(), '검출': y})
display(t.groupby(['자료', 'aw_source 종류']).agg(시료_행=('검출', 'size'), 검출=('검출', 'sum')).rename(columns={'시료_행': '시료 행'}))

print('3) 수분활성도 값이 있는 행 — 어느 키로 붙었는지')
has_aw = food['aw'].notna().to_numpy()
t = pd.DataFrame({'조인 키': food['aw_key'].to_numpy(), '자료': food['aw_src'].to_numpy(), '검출': y})[has_aw]
g = t.groupby(['조인 키', '자료']).agg(시료_행=('검출', 'size'), 검출=('검출', 'sum')).rename(columns={'시료_행': '시료 행'})
g['검출 중(%)'] = (g['검출'] / max(int((has_aw & y).sum()), 1) * 100).round(1)
display(g)
print(f'수분활성도 값이 있는 검출 행 {int((has_aw & y).sum()):,} / 전체 검출 {int(y.sum()):,}')

print('4) 범위를 벗어난 값')
odd_aw = ((food['aw'] < 0) | (food['aw'] > 1)).to_numpy()
print(f'수분활성도가 0~1 밖인 시료 {int(odd_aw.sum())}행:', food.loc[odd_aw].groupby(['aw_code', 'aw_src', 'aw_raw']).size().to_dict())
print('염도 값 분포:', food['na'].describe(percentiles=[.5, .9, .99]).round(3).to_dict())
top_na = food.loc[(food['na'] >= 16).to_numpy()].groupby(['na_code', 'na_src', 'na_raw']).agg(염도=('na', 'first'), 시료_행=('na', 'size')).reset_index()
top_na.insert(1, '품목 이름', top_na['na_code'].map(CODE_TABLE['na']['kor_nm']))
print('염도가 16 이상인 품목코드 (PPT 생장 한계 표의 "≤16%"와 견줄 수 있게 — 단위가 같은지는 확인 필요):')
display(top_na.sort_values('염도', ascending=False))

1) t_aw_cl 에서 같은 수분활성도 원문이 5개 이상의 품목코드에 들어 있는 경우


,t_aw_cl 원문,품목코드 수,단계(lv)별 코드 수,aw_source,엑셀에 코드가 있는 수,그중 엑셀 값이 있는 수,엑셀 값이 다른 수,이 값(t_aw_cl)이 붙은 시료 행,그중 검출
0,0.15,615,"{3: 8, 4: 139, 5: 468}",{'original': 615},6,5,5,1176,1
1,>0.98,92,"{3: 50, 4: 24, 5: 18}",{'original': 92},92,92,1,0,0
2,0.992,34,"{2: 1, 3: 1, 4: 32}","{'original': 33, 'derived_median': 1}",33,33,1,62,0
3,<0.6,22,{5: 22},{'original': 22},22,22,0,0,0
4,0.98,18,"{3: 4, 4: 12, 5: 2}",{'original': 18},18,14,0,34,0
5,0.96,18,"{3: 2, 4: 15, 5: 1}",{'original': 18},18,17,0,0,0
6,0.91,17,"{2: 1, 3: 13, 4: 2, 5: 1}","{'original': 16, 'recomputed': 1}",17,4,0,19861,2
7,0.99,14,"{2: 1, 4: 13}","{'original': 13, 'recomputed': 1}",14,9,0,1,0
8,0.986,14,"{3: 1, 4: 13}",{'original': 14},14,12,0,0,0
9,<0.7,12,"{3: 11, 5: 1}",{'original': 12},12,12,0,0,0


t_aw_cl 원문이 '0.839' 인 품목코드 9개


,품목 이름,lv,t_aw_cl 원문,엑셀에 코드가 있는가,엑셀 원문(없으면 빈칸),aw_source,시료 행(prdlst_code),시료 행(train_prdlst_code)
prdlst_cd,,,,,,,,
C0318000000000,유가공품류,3,0.839,True,-,original,0,107022
C0316000000000,식육가공품 및 포장육,3,0.839,True,-,original,0,101045
C0317000000000,알가공품류,3,0.839,True,-,original,0,29392
C0302000000000,빙과류,3,0.839,True,-,original,0,22206
C0312000000000,조미식품,3,0.839,True,-,original,0,4165
C0313000000000,절임류 또는 조림류,3,0.839,True,-,original,0,2804
C0310000000000,특수영양식품,3,0.839,True,-,original,0,2140
C0306000000000,두부류 또는 묵류,3,0.839,True,-,original,0,489
C0314000000000,주류,3,0.839,True,-,original,0,74


t_aw_cl 원문이 '0.15' 인 품목코드 615개 — 엑셀 값이 있거나 시료 500행 넘게 걸리는 7개만 표시


,품목 이름,lv,t_aw_cl 원문,엑셀에 코드가 있는가,엑셀 원문(없으면 빈칸),aw_source,시료 행(prdlst_code),시료 행(train_prdlst_code)
prdlst_cd,,,,,,,,
C0319030100000,조미건어포,5,0.15,True,0.6~0.85,original,291,0
C0316040000000,건조저장육류,4,0.15,True,0.85-0.93,original,5852,0
C0319030200000,건어포,5,0.15,True,0.6~0.85,original,44,0
C0308030000000,건면,4,0.15,True,0.51,original,70,0
A0500400200000,건포도,4,0.15,True,-,original,2,0
C0319030300000,기타 건포류,5,0.15,True,0.48~0.66,original,11,0
C0202040000000,건조저장육류,4,0.15,False,NaN,original,859,0


... (3 outputs left out)

3) 수분활성도 값이 있는 행 — 어느 키로 붙었는지


시료 행   검출  검출 중(%)
조인 키               자료                           
prdlst_code        t_aw_cl   43159   21      3.9
                   엑셀       203699  238     44.2
prdlst_code 앞 14자리 t_aw_cl    2870   10      1.9
                   엑셀        16364   32      5.9
train_prdlst_code  t_aw_cl   69328  195     36.2
                   엑셀         9816   42      7.8

수분활성도 값이 있는 검출 행 538 / 전체 검출 1,969
4) 범위를 벗어난 값
수분활성도가 0~1 밖인 시료 8행: {('C0315020200000', 't_aw_cl', '-0.85'): 8}
염도 값 분포: {'count': 281879.0, 'mean': 1.003, 'std': 2.942, 'min': 0.0, '50%': 0.25, '90%': 2.755, '99%': 5.85, 'max': 96.05}
염도가 16 이상인 품목코드 (PPT 생장 한계 표의 "≤16%"와 견줄 수 있게 — 단위가 같은지는 확인 필요):


,na_code,품목 이름,na_src,na_raw,염도,시료_행
4,C0312060300000,태움.용융소금,엑셀,95.5~96.6,96.050003,1
6,C0312060600000,가공소금,엑셀,95.5~96.6,96.050003,11
3,C0312060200000,재제소금(재제조소금),엑셀,96,96.000000,5
5,C0312060400000,정제소금,엑셀,91.5035,91.503502,85
2,C0312060100000,천일염,엑셀,85.87994,85.879936,29
1,C0312020400000,복합조미식품,엑셀,37.7698,37.769798,645
7,C0319020300000,액젓,엑셀,22.53~27.37,24.950001,70
8,C0319020400000,조미액젓,엑셀,17.81~25.26,21.535000,7
0,C0311030000000,한식간장,엑셀,16.97736,16.977360,17


In [28]:
# 저장 — 데이터셋에 넣을 값 3 + 값 표기 3 + 출처 7칼럼, 그 밖의 추적용 칼럼(어느 품목코드·원문·규칙)은 중간 파일에만
FOOD_LABEL_COLS = [f'{c}_label' for c in FOOD_COLS]
FOOD_TRACE_COLS = ['aw_key', 'aw_src', 'aw_source', 'ph_key', 'ph_src', 'na_key', 'na_src']

def arr_cat(series):
    return pa.array(pd.Series(series).replace('', np.nan).to_numpy(dtype=object), type=pa.string(), from_pandas=True).dictionary_encode()

cols = {KEY_COL: pa.array(KEYS, type=pa.large_string())}
for c in FOOD_COLS:
    cols[c] = pa.array(food[c].to_numpy(np.float32), from_pandas=True)
for c in FOOD_COLS:
    cols[f'{c}_label'] = pa.array(pd.Categorical(food[f'{c}_label'].replace('', np.nan), categories=FOOD_LABELS))
for c in FOOD_TRACE_COLS:
    cols[c] = arr_cat(food[c])
for c in FOOD_COLS:
    for suf in ('code', 'raw', 'rule'):
        cols[f'{c}_{suf}'] = pa.array(food[f'{c}_{suf}'].replace('', np.nan).to_numpy(dtype=object), type=pa.string(), from_pandas=True)
FOOD_TABLE = pa.table(cols)
pq.write_table(FOOD_TABLE, MID_DIR / 'food.parquet', row_group_size=ROW_GROUP)
print('저장:', MID_DIR / 'food.parquet', f'({FOOD_TABLE.num_rows:,}행 × {FOOD_TABLE.num_columns}칼럼) — 데이터셋에는', FOOD_COLS + FOOD_LABEL_COLS + FOOD_TRACE_COLS, '을 넣는다')

저장: C:\Users\NT751\Desktop\천영진\리스테리아ML2\data\중간\food.parquet (676,459행 × 23칼럼) — 데이터셋에는 ['aw', 'ph', 'na', 'aw_label', 'ph_label', 'na_label', 'aw_key', 'aw_src', 'aw_source', 'ph_key', 'ph_src', 'na_key', 'na_src'] 을 넣는다


## 7. 조인 — 토양 (pH·유기물·유효인산)

`api_data_info.soil_ph_info`(DB 테이블 코멘트 "토양산도 정보")를 **가능한 한 가깝고 가능한 한 최근인 검정값**으로 붙인다(사용자 결정 2026-10-08). 이 표의 한 행은 필지 한 곳의 토양검정 1건이고 좌표 칼럼이 없어, 법정동 좌표 표 `ba_data_info.korea_region_code`(DB 테이블 코멘트 "법정동_코드")의 `lat`·`lon` 으로 법정동의 위치를 얻는다.

| 칼럼 (항목마다 하나씩, `soil_ph`·`soil_om`·`soil_vldpha`) | 내용 |
|---|---|
| `soil_ph` · `soil_om` · `soil_vldpha` | `acid`(DB 코멘트 "산도") · `om`("유기물") · `vldpha`("유효인산") |
| `<항목>_days_before` | 며칠 전 값인지 = 기준일 − 그 값을 가져온 검정일 (일) |
| `<항목>_dong_code` | 그 값을 가져온 법정동 코드 |
| `<항목>_dist_km` | 시료 좌표에서 그 법정동 좌표까지 거리 (km) |
| `<항목>_exam_n` | 그 법정동의 그 검정일에 그 항목 값이 있는 검정 행 수 |

### 왜 이렇게 하는가

2026-10-08 전 규칙은 "기준일보다 앞선 검정이 한 번이라도 있는 **가장 가까운 법정동**"을 먼저 정하고, 그 법정동 안에서 가장 최근 검정을 썼다. 법정동을 정할 때 검정이 얼마나 오래됐는지는 보지 않았다. 그래서 두 가지 문제가 있었다(전 규칙으로 실행한 출력의 수치, `soil_ph` 기준).

- **아주 오래된 값이 붙었다.** 가까운 법정동에 10년 전 검정 하나만 있어도 그 값을 썼고, 조금 더 먼 법정동에 최근 검정이 있어도 쓰지 않았다. 3년 넘은 값 55,674행(검출 250), 10년 넘은 값 670행. 예: 서울 서초구 마방로 68 주소의 시료는 1.20km 떨어진 법정동의 3,668일 전 값을 받았는데, 1.37km 떨어진 법정동에는 253일 전 검정이 있었다(7-5절 출력).
- **값이 없는 국내 행이 있었다.** 고른 법정동에 그 항목 값이 없으면 다른 법정동으로 넘어가지 않아 결측이 됐다(`soil_ph` 1,564행 · `soil_om` 106행 · `soil_vldpha` 106행).

가까운 것과 최근인 것을 둘 다 만족하는 후보가 항상 있는 것은 아니다. 5km 안에서 "가장 가까운 후보가 곧 가장 최근 후보"라 고를 것이 없는 국내 행은 12.0%뿐이고, 나머지는 가까운 후보와 최근 후보 사이에서 하나를 골라야 한다(7-5절 출력). 그래서 고르는 기준을 아래처럼 정했다.

### 어떻게 고르는가

항목마다 따로, 아래 순서로 고른다.

1. **후보를 만든다.** 법정동마다 "그 항목 값이 있는 검정 중 기준일보다 앞선 가장 최근 검정일"이 그 법정동의 후보다. 후보마다 두 숫자가 있다 — 거리(km)와 며칠 전 값인지(일).
2. **두 숫자를 같은 눈금으로 바꾼다.** 1년 묵은 값을 1km 떨어진 값과 같은 크기로 본다(**1km = 1년**, `SOIL_KM_PER_YEAR = 1.0`). 며칠 전 값인지 ÷ 365.25 가 km 로 바꾼 경과다.
3. **후보의 점수 = 거리와 경과 중 큰 쪽.** 점수가 가장 작은 후보를 고른다. 곧 "둘 중 나쁜 쪽이 가장 덜 나쁜 후보"다.
4. 점수가 같으면 (거리 + 경과)가 작은 후보, 그래도 같으면 가까운 쪽(거리도 같으면 법정동 코드가 작은 쪽).

| 후보 (설명용으로 지은 숫자) | 거리 | 며칠 전 값 | 경과(km 로 바꾼 값) | 점수 = 큰 쪽 |
|---|---|---|---|---|
| 가장 가까운 법정동 | 0.2km | 3,650일 | 9.99 | 9.99 |
| 조금 먼 법정동 | 2.0km | 200일 | 0.55 | **2.00** ← 고른다 |
| 더 먼 법정동 | 5.0km | 30일 | 0.08 | 5.00 |

- **큰 쪽으로 정한 이유**: 거리와 경과를 더하는 방식은 한쪽이 아주 좋으면 다른 쪽이 많이 나빠도 뽑힌다. 큰 쪽으로 보면 아주 가깝지만 오래된 값도, 아주 최근이지만 먼 값도 피한다.
- **1km = 1년으로 정한 이유**: 토양 자료로 눈금을 정해 보려 했으나 정해지지 않았다. 같은 법정동에서 석 달 안에 다시 검정한 값도 산도가 0.60(중앙값) 다르고, 8년 넘게 지나도 0.65, 10~20km 떨어져도 0.65로 차이가 조금만 늘어난다(7-5절 출력). 시간·거리보다 필지마다 값이 다른 쪽이 훨씬 크다. 그래서 눈금은 자료가 아니라 **사용자 결정(2026-10-08)** 으로 정했고, 1km = 0.5년·2년으로 바꾸면 무엇이 달라지는지 7-5절에 적었다.
- **결과**(7-4절 출력): 국내 행에 값이 없는 행 0, 10년 넘은 값 0행, 3년 넘은 `soil_ph` 값 1,845행(검출 2). 대신 값을 가져온 법정동까지 거리는 중앙값 0.57km → 0.73km, 90% 지점 1.46km → 1.73km 로 늘었다.

### 조인 규칙

| 규칙 | 내용 |
|---|---|
| 대상 | 국내 행 전부. 해외 행은 붙일 자료가 없어 결측. 좌표는 시료 주소의 좌표다(제조 장소만이 아니다 — 2-1절) |
| 후보 법정동 | `soil_ph_info` 에 검정이 있고 `korea_region_code` 에 좌표가 있는 법정동 |
| 후보 | 항목마다, 법정동마다: 그 항목 값이 있는 검정 중 `exam_day`("토양검정일")가 기준일(`mnfctur_de`)보다 앞인 가장 최근 날. 기준일 당일·이후 검정은 쓰지 않는다 |
| 거리 | 시료 파일의 `lat`·`lon` 에서 후보 법정동 좌표까지 **실제 거리(구면 거리, km)**(4절 기상과 같은 방식). 거리 상한 없음 |
| 고르기 | 점수 = max(거리 km, 며칠 전 값인지 ÷ 365.25 × `SOIL_KM_PER_YEAR`)가 가장 작은 후보. 같으면 (거리 + 경과)가 작은 후보, 그래도 같으면 가까운 쪽(거리도 같으면 법정동 코드가 작은 쪽) |
| 항목마다 따로 | 세 항목을 따로 고르므로 한 행의 세 값이 서로 다른 법정동·검정일에서 올 수 있다(7-4절에 행 수) |
| 같은 날 여러 행 | 그 법정동에서 그날 검정한 필지가 여러 곳이면 항목마다 **중앙값**. 그 항목 값이 비어 있는 행은 그 항목에서만 빠진다 |
| 검정으로 치는 행 | 세 항목(`acid`·`om`·`vldpha`) 중 하나라도 값이 있는 행 |
| 값 | 이상값·중복 행을 지우지 않는다. 결측은 채우지 않는다 |

- **단위**: DB 코멘트·테이블정의서에는 단위가 없다. 리서치팀 엑셀(4-0절 출력)의 같은 항목은 "산도(pH)", "유효인산(mg/kg)", "유기물(g/kg)"로 적혀 있다. DB 값의 단위가 이와 같은지는 확인 필요.
- **경지구분을 가리지 않는다**: `exam_type`(DB 코멘트 "경지구분코드", 정의서 "주사유형")은 값이 8가지이고 구분에 따라 값 수준이 다른데(7-1절 출력), 코드의 뜻 설명이 없어 가리지 않고 쓴다 — 확인 필요.
- **오래된 값·먼 값에 상한은 없다**: 상한으로 자르지 않으므로 국내 행은 모두 값을 받는다. 가까이에 최근 검정이 없는 행은 여전히 오래되거나 먼 값을 받는다(7-4절 — 며칠 전 값인지 최대 2,389일, 거리 최대 29.44km). `<항목>_days_before`·`<항목>_dist_km` 로 행마다 확인할 수 있다.
- **필지 한 곳의 값일 수 있다**: 고른 검정일에 그 항목 검정이 1행뿐인 행이 36%쯤이다(7-4절). 같은 법정동 안에서도 필지마다 값이 다르므로(7-5절), 이 값은 그 법정동을 대표하는 값이 아닐 수 있다.
- **"가까운 법정동"의 실제 뜻**: 시료 좌표는 주소 자체가 아니라 주소가 속한 법정동의 대표 지점이다(3-3절). 그래서 거리는 대표 지점에서 토양 법정동 좌표까지의 거리이고, `<항목>_dist_km` 도 주소에서 잰 거리가 아니다.
- **2026-10-08 에 바꾼 점 (1)**: 그 전 판은 세 항목 중 하나라도 값이 있는 가장 최근 검정일 하루를 골라 세 항목을 모두 그날 값에서 가져왔다. 그날 어떤 항목이 비어 있으면 더 앞선 검정일에 그 항목 값이 있어도 결측이 됐다. 항목마다 그 항목 값이 있는 검정일을 쓰도록 바꿨다.
- **2026-10-08 에 바꾼 점 (2)**: 법정동을 고르는 방법을 "가장 가까운 법정동"에서 위의 "거리와 경과 중 큰 쪽이 가장 작은 후보"로 바꿨다. 법정동을 항목마다 고르게 되어 `soil_dong_code`·`soil_dist_km` 두 칼럼이 `<항목>_dong_code`·`<항목>_dist_km` 여섯 칼럼으로 바뀌었다.
- **같은 날 여러 필지는 중앙값**: 이상값(산도 14 초과 등)을 지우지 않으므로 평균 대신 중앙값으로 정했다(이 노트북에서 정한 것).
- 테이블정의서에는 `exam_day` 가 "조사일. 연 단위 집계 시 사용하지 않음"으로 적혀 있으나, `any_year`("시료채취년도")로는 검정이 기준일보다 앞인지 뒤인지 가를 수 없어 `exam_day` 를 쓴다.
- `korea_region_code` 의 `lat`·`lon` 이 법정동의 어느 지점인지는 DB 코멘트가 없다 — 확인 필요.
- `<항목>_dist_km` 는 후보를 고를 때 쓴 그 구면 거리(1절의 `haversine_km`)다.

### 7-1. 원자료 읽기

In [29]:
SOIL_ITEMS = {'soil_ph': ('acid', '산도'), 'soil_om': ('om', '유기물'), 'soil_vldpha': ('vldpha', '유효인산')}
SOIL_SRC = [v[0] for v in SOIL_ITEMS.values()]
EPOCH = pd.Timestamp('1970-01-01')

SOIL_PATH = RAW_DIR / 'raw_soil_ph_info.parquet'
krc = pd.read_parquet(RAW_DIR / 'raw_korea_region_code.parquet')
assert krc['legal_dong_code'].is_unique
print(f"ba_data_info.korea_region_code: {len(krc):,}행 · 좌표 있는 행 {int(krc['lat'].notna().sum()):,} · level {krc['level'].value_counts(dropna=False).to_dict()}")

# 필요한 칼럼만 읽는다 (날짜는 날짜 번호로 — 1970-01-01 부터 며칠째)
t = pq.read_table(SOIL_PATH, columns=['soil_ph_seq', 'legal_dong_code', 'any_year', 'exam_day', 'exam_type'] + SOIL_SRC, read_dictionary=['legal_dong_code', 'exam_type'])
assert t.column('exam_day').null_count == 0 and t.column('legal_dong_code').null_count == 0
assert pc.count_distinct(t.column('soil_ph_seq')).as_py() == t.num_rows, 'soil_ph_seq 중복'
mm = pc.min_max(t.column('exam_day')).as_py()
soil = pd.DataFrame({'legal_dong_code': t.column('legal_dong_code').to_pandas(),
                     'exam_no': t.column('exam_day').cast(pa.int32()).to_numpy(),
                     'exam_type': t.column('exam_type').to_pandas(),
                     **{c: t.column(c).to_numpy() for c in SOIL_SRC}})
year_gap = pc.year(t.column('exam_day')).to_numpy() - pc.cast(t.column('any_year'), pa.int32()).to_numpy()
del t
print(f"api_data_info.soil_ph_info: {len(soil):,}행 · 법정동 {soil['legal_dong_code'].nunique():,}개 · exam_day {mm['min']} ~ {mm['max']}")
display(soil[SOIL_SRC].describe(percentiles=[.01, .05, .5, .95, .99]).T.assign(**{'값 없는 행': [int(soil[c].isna().sum()) for c in SOIL_SRC]}))
print(f"고치지 않고 그대로 두는 값 — 산도 14 초과 {int((soil['acid'] > 14).sum())}행(최대 {soil['acid'].max()}) · 유기물 0 {int((soil['om'] == 0).sum())}행 · 유효인산 0 {int((soil['vldpha'] == 0).sum())}행")
print('검정일 연도 − 채취년도(any_year):', pd.Series(year_gap).value_counts().sort_index().to_dict())
usable = soil[SOIL_SRC].notna().any(axis=1)
print(f'세 항목이 모두 빈 행 {int((~usable).sum()):,} → 검정으로 치지 않는다')
soil = soil[usable]
del year_gap, usable
print('경지구분(exam_type)별 행 수와 중앙값 — 가리지 않고 쓴다:')
display(soil.groupby('exam_type', observed=True)[SOIL_SRC].median().assign(행=soil.groupby('exam_type', observed=True).size()).sort_values('행', ascending=False))
soil = soil.drop(columns=['exam_type'])

ba_data_info.korea_region_code: 51,884행 · 좌표 있는 행 27,401 · level {'ri': 39244, 'eupmyeondong': 12142, 'sigungu': 478, 'sido': 20}


api_data_info.soil_ph_info: 6,280,656행 · 법정동 17,425개 · exam_day 2015-01-01 ~ 2026-07-20


,count,mean,std,min,1%,5%,50%,95%,99%,max,값 없는 행
acid,6184118.0,6.283845,2.482693,0.1,4.5,5.1,6.2,7.6,8.2,5396.0,96538
om,6263331.0,26.120940,19.617095,0.0,4.0,8.0,23.0,50.6,111.2,1936.0,17325
vldpha,6239520.0,304.850220,387.006501,0.0,7.0,19.0,145.0,1063.0,1725.0,37647.6,41136


고치지 않고 그대로 두는 값 — 산도 14 초과 99행(최대 5396.0) · 유기물 0 348행 · 유효인산 0 260행
검정일 연도 − 채취년도(any_year): {0: 6234294, 1: 46113, 2: 160, 3: 45, 4: 4, 5: 39, 6: 1}
세 항목이 모두 빈 행 9,013 → 검정으로 치지 않는다
경지구분(exam_type)별 행 수와 중앙값 — 가리지 않고 쓴다:


,acid,om,vldpha,행
exam_type,,,,
1,6.10,23.0,92.7,3433811
2,6.40,22.0,349.0,1748213
4,6.40,25.0,398.1,538175
3,6.60,26.7,662.0,505030
7,5.30,28.4,34.6,29741
8,7.10,24.3,69.0,16019
5,7.80,12.0,68.0,536
6,7.35,14.0,137.0,118


### 7-2. 법정동 × 검정일 표와 후보 법정동

- 법정동·검정일마다 항목별 중앙값과 행 수(전체 · 항목별로 값이 있는 행)를 미리 구해 둔다(같은 날 여러 필지 → 중앙값).
- 후보 법정동 = 검정이 있고 좌표가 있는 법정동.
- 항목마다 "그 항목 값이 있는 (법정동, 검정일)"만 모은 표(`SOIL_DAYS`)를 만든다. 7-3절은 이 표에서 법정동마다 기준일보다 앞선 가장 최근 검정일을 찾는다.

In [30]:
g = soil.groupby(['legal_dong_code', 'exam_no'], observed=True, sort=False)
DAY = g[SOIL_SRC].median()
DAY['n'] = g.size()
for s in SOIL_SRC:
    DAY[f'n_{s}'] = g[s].count()            # 그날 그 항목 값이 있는 행 수
DAY = DAY.reset_index()
DAY['legal_dong_code'] = DAY['legal_dong_code'].astype(str)
DAY = DAY.sort_values(['legal_dong_code', 'exam_no']).reset_index(drop=True)     # 법정동 코드순 · 검정일순
print(f"(법정동, 검정일) 조합 {len(DAY):,}개 · 그날 검정 행 수 분포: 1행 {float((DAY['n'] == 1).mean()) * 100:.1f}% · 2행 {float((DAY['n'] == 2).mean()) * 100:.1f}% · 3행 이상 {float((DAY['n'] >= 3).mean()) * 100:.1f}% (최대 {int(DAY['n'].max())})")
del soil, g; gc.collect()

n_soil_dong = DAY['legal_dong_code'].nunique()
CAND = (krc.loc[krc['lat'].notna() & krc['lon'].notna() & krc['legal_dong_code'].isin(set(DAY['legal_dong_code'])), ['legal_dong_code', 'legal_dong_nm', 'level', 'lat', 'lon']]
        .sort_values('legal_dong_code').reset_index(drop=True))
print(f'검정이 있는 법정동 {n_soil_dong:,}개 중 좌표가 있는 후보 법정동 {len(CAND):,}개 ({len(CAND) / n_soil_dong * 100:.2f}%) · 좌표가 없어 후보에서 빠진 법정동 {n_soil_dong - len(CAND):,}개')
print('후보 법정동의 level:', CAND['level'].value_counts().to_dict())
DAY = DAY[DAY['legal_dong_code'].isin(set(CAND['legal_dong_code']))].reset_index(drop=True)
day_cand = pd.Index(CAND['legal_dong_code']).get_indexer(DAY['legal_dong_code'])     # DAY 행마다 후보 법정동 번호 (CAND 는 코드순)
assert (day_cand >= 0).all() and (np.diff(day_cand) >= 0).all()

# 항목별 표 — 그 항목 값이 있는 (법정동, 검정일)만. key = 후보 법정동 번호 × KEY_MUL + 검정일 번호 (오름차순이라 한 번에 찾을 수 있다)
KEY_MUL = 100_000
assert DAY['exam_no'].max() < KEY_MUL
SOIL_DAYS = {}
for c, (src, ko) in SOIL_ITEMS.items():
    has = DAY[src].notna().to_numpy()
    T = {'cand': day_cand[has], 'no': DAY['exam_no'].to_numpy()[has], 'val': DAY[src].to_numpy()[has], 'n': DAY[f'n_{src}'].to_numpy()[has]}
    T['key'] = T['cand'].astype(np.int64) * KEY_MUL + T['no']
    assert (np.diff(T['key']) > 0).all()
    SOIL_DAYS[c] = T
    print(f"{c}({ko}): 값이 있는 (법정동, 검정일) {int(has.sum()):,}개 · 그 항목 검정이 있는 후보 법정동 {len(np.unique(T['cand'])):,}개")
del DAY, day_cand; gc.collect();

(법정동, 검정일) 조합 1,076,056개 · 그날 검정 행 수 분포: 1행 31.3% · 2행 19.8% · 3행 이상 49.0% (최대 746)


검정이 있는 법정동 17,424개 중 좌표가 있는 후보 법정동 17,212개 (98.78%) · 좌표가 없어 후보에서 빠진 법정동 212개
후보 법정동의 level: {'ri': 14700, 'eupmyeondong': 2512}


soil_ph(산도): 값이 있는 (법정동, 검정일) 1,051,159개 · 그 항목 검정이 있는 후보 법정동 17,210개
soil_om(유기물): 값이 있는 (법정동, 검정일) 1,063,576개 · 그 항목 검정이 있는 후보 법정동 17,211개
soil_vldpha(유효인산): 값이 있는 (법정동, 검정일) 1,061,411개 · 그 항목 검정이 있는 후보 법정동 17,210개


### 7-3. 시료마다 후보 고르기 — 거리와 경과 중 큰 쪽이 가장 작은 후보

같은 (좌표, 기준일) 조합은 값이 같으므로 고유 조합마다 한 번만 계산한다. 항목마다 따로 고른다.

1. 좌표마다 후보 법정동을 거리순으로 정렬해 가까운 300곳을 먼저 본다.
2. 법정동마다 "그 항목 값이 있는 검정 중 기준일보다 앞선 가장 최근 검정일"을 찾아 며칠 전 값인지를 구한다.
3. 점수 = max(거리 km, 며칠 전 값인지 ÷ 365.25 × `SOIL_KM_PER_YEAR`)가 가장 작은 법정동을 고른다(같을 때의 순서는 7절 규칙 표).
4. 300곳 밖의 법정동은 거리가 300번째보다 멀고, 점수는 거리보다 작을 수 없다. 그래서 고른 후보의 점수가 300번째 거리보다 작으면 300곳만 보아도 전체에서 고른 것과 같다. 그렇지 않은 좌표는 후보를 8배씩 넓혀 다시 본다(아래 출력에 그런 좌표 수).

고르는 함수 `soil_choose` 는 눈금을 인자로 받는다. 데이터셋에는 눈금 `SOIL_KM_PER_YEAR = 1.0`(1km = 1년)으로 고른 값이 들어가고, 7-5절에서 같은 함수를 다른 눈금으로 불러 결과를 견준다.

In [31]:
SOIL_KM_PER_YEAR = 1.0        # 눈금: 1년 묵은 값을 1km 떨어진 값과 같은 크기로 본다 (사용자 결정 2026-10-08)
SOIL_DAYS_PER_YEAR = 365.25   # 며칠 전 값인지(일)를 년으로 바꿀 때 나누는 수
SOIL_TOP = 300                # 계산을 빠르게 하려고 좌표마다 가까운 300곳을 먼저 본다 (모자라면 넓혀서 다시 본다)

cand_xy = CAND[['lat', 'lon']].to_numpy(float)
base_no = (df[DATE_COL] - EPOCH).dt.days.to_numpy()
dom_idx = np.nonzero(IS_DOM)[0]
combo = pd.DataFrame({'loc': row_dom_loc, 'base_no': base_no[dom_idx]}).drop_duplicates().sort_values(['loc', 'base_no']).reset_index(drop=True)
print(f'국내 고유 좌표 {len(DOM_XY):,}곳 · 고유 (좌표, 기준일) 조합 {len(combo):,}개')
loc_xy = DOM_XY.to_numpy(float)
base_c = combo['base_no'].to_numpy()
bounds = np.append(np.nonzero(np.diff(combo['loc'].to_numpy(), prepend=-1))[0], len(combo))    # 좌표마다 combo 의 시작 위치


def soil_choose(item, km_per_year, top_n=SOIL_TOP, radius_km=None):
    """고유 (좌표, 기준일) 조합마다 그 항목의 후보 하나를 고른다.

    후보 = 법정동마다 "그 항목 값이 있는 검정 중 기준일보다 앞선 가장 최근 검정일".
    점수 = max(거리 km, 며칠 전 값인지 ÷ 365.25 × km_per_year) — 둘 중 나쁜 쪽. 점수가 가장 작은 후보를 고른다.
    점수가 같으면 (거리 + 경과 환산값)이 작은 후보, 그래도 같으면 가까운 쪽(거리도 같으면 법정동 코드가 작은 쪽).
    km_per_year=0 이면 경과를 보지 않는다 = 기준일보다 앞선 그 항목 검정이 한 번이라도 있는 가장 가까운 법정동.
    radius_km 을 주면 그 반경 안에서 "더 가깝고 더 최근인 다른 후보가 없는 후보" 수도 센다.
    반환: 후보 법정동 번호(-1 = 없음), 거리 km, 항목 표(SOIL_DAYS[item])에서의 위치, 반경 안 후보 수, 넓혀서 다시 본 좌표 수"""
    T = SOIL_DAYS[item]
    n = len(combo)
    pick, pos, dist_km = np.full(n, -1, dtype=np.int64), np.full(n, -1, dtype=np.int64), np.full(n, np.nan)
    n_front = np.zeros(n, dtype=np.int64)
    n_wide = 0
    for a, b in zip(bounds[:-1], bounds[1:]):
        li = int(combo.at[a, 'loc'])
        d = haversine_km(loc_xy[li, 0], loc_xy[li, 1], cand_xy[:, 0], cand_xy[:, 1])     # 후보 법정동까지 구면 거리 km
        order = np.argsort(d, kind='stable')              # 거리가 같으면 법정동 코드가 작은 쪽 (CAND 는 코드순)
        bn = base_c[a:b]
        k = top_n
        while True:
            top = order[:k]
            dt = d[top]
            p = np.searchsorted(T['key'], top[:, None].astype(np.int64) * KEY_MUL + bn[None, :], side='left') - 1   # exam_day < 기준일 인 마지막 검정일
            pc_ = np.clip(p, 0, None)
            ok = (p >= 0) & (T['cand'][pc_] == top[:, None])                  # (후보 법정동, 기준일) 그 법정동에 기준일보다 앞선 검정이 있는가
            days_ago = np.where(ok, bn[None, :] - T['no'][pc_], 0)            # 며칠 전 값인지 (검정이 없는 칸은 아래에서 inf 로 둔다)
            age = np.where(ok, days_ago, np.inf)
            a_km = np.where(ok, days_ago / SOIL_DAYS_PER_YEAR * km_per_year, np.inf)
            score = np.maximum(dt[:, None], a_km)                             # 둘 중 나쁜 쪽
            best = score.min(axis=0)
            j = np.argmin(np.where(score == best, dt[:, None] + a_km, np.inf), axis=0)
            # 더 먼 법정동의 점수는 그 거리보다 작을 수 없다 → 지금 본 가장 먼 곳의 거리보다 점수가 작으면 더 볼 필요가 없다
            enough = (best < dt[-1]).all() and (radius_km is None or dt[-1] > radius_km)
            if k >= len(order) or enough:
                break
            k = min(k * 8, len(order))
            n_wide += 1
        ar = np.arange(b - a)
        got = ok[j, ar]
        pick[a:b] = np.where(got, top[j], -1)
        pos[a:b] = np.where(got, p[j, ar], -1)
        dist_km[a:b] = np.where(got, dt[j], np.nan)
        if radius_km is not None:
            cm = np.minimum.accumulate(age, axis=0)                           # 가까운 순으로 볼 때 지금까지 가장 최근 값
            newer = np.vstack([np.isfinite(age[:1]), cm[1:] < cm[:-1]])       # 앞의 어느 후보보다도 최근인 후보
            n_front[a:b] = (newer & (dt <= radius_km)[:, None]).sum(axis=0)
    return pick, dist_km, pos, n_front, n_wide


t0 = time.time()
SOIL_PICK = {}
cand_code = CAND['legal_dong_code'].to_numpy()
for c, (src, ko) in SOIL_ITEMS.items():
    pick, dist_km, pos, _, n_wide = soil_choose(c, SOIL_KM_PER_YEAR)
    T = SOIL_DAYS[c]
    got_c = pos >= 0
    p = np.clip(pos, 0, None)
    combo[c] = np.where(got_c, T['val'][p], np.nan)
    combo[f'{c}_days_before'] = np.where(got_c, base_c - T['no'][p], np.nan)
    combo[f'{c}_dong_code'] = np.where(got_c, cand_code[np.clip(pick, 0, None)], None)
    combo[f'{c}_dist_km'] = dist_km
    combo[f'{c}_exam_n'] = np.where(got_c, T['n'][p], np.nan)
    assert (combo.loc[got_c, f'{c}_days_before'] >= 1).all() and (combo.loc[got_c, f'{c}_exam_n'] >= 1).all() and combo.loc[got_c, c].notna().all()
    SOIL_PICK[c] = (pick, pos)
    print(f'{c}({ko}): 후보를 고른 조합 {int(got_c.sum()):,} / {len(combo):,} · 가까운 {SOIL_TOP}곳으로 모자라 넓혀서 다시 본 좌표 {n_wide}')
print(f'계산 {time.time() - t0:.0f}초 · 눈금 1km = {1 / SOIL_KM_PER_YEAR:g}년')

# 시료 행에 붙이기
ci = pd.MultiIndex.from_frame(combo[['loc', 'base_no']]).get_indexer(pd.MultiIndex.from_arrays([row_dom_loc, base_no[dom_idx]]))
assert (ci >= 0).all()
SOIL_DAYS_COLS = [f'{c}_days_before' for c in SOIL_ITEMS]
SOIL_DONG_COLS = [f'{c}_dong_code' for c in SOIL_ITEMS]
SOIL_DIST_COLS = [f'{c}_dist_km' for c in SOIL_ITEMS]
SOIL_N_COLS = [f'{c}_exam_n' for c in SOIL_ITEMS]
SOIL_COLS = list(SOIL_ITEMS) + SOIL_DAYS_COLS + SOIL_DONG_COLS + SOIL_DIST_COLS + SOIL_N_COLS
SOIL = pd.DataFrame({KEY_COL: df[KEY_COL].to_numpy()})
for c in SOIL_COLS:
    is_code = c in SOIL_DONG_COLS
    col = np.full(N_ROWS, None if is_code else np.nan, dtype=object if is_code else float)
    col[dom_idx] = combo[c].to_numpy()[ci]
    SOIL[c] = col
assert len(SOIL) == N_ROWS

국내 고유 좌표 3,599곳 · 고유 (좌표, 기준일) 조합 264,715개


soil_ph(산도): 후보를 고른 조합 264,715 / 264,715 · 가까운 300곳으로 모자라 넓혀서 다시 본 좌표 0


soil_om(유기물): 후보를 고른 조합 264,715 / 264,715 · 가까운 300곳으로 모자라 넓혀서 다시 본 좌표 0


soil_vldpha(유효인산): 후보를 고른 조합 264,715 / 264,715 · 가까운 300곳으로 모자라 넓혀서 다시 본 좌표 0
계산 19초 · 눈금 1km = 1년


### 7-4. 결과 — 값 있는 비율, 거리, 며칠 전 값인지

- 국내 행은 세 항목 모두 값이 있어야 한다(상한으로 자르지 않으므로).
- 거리와 며칠 전 값인지는 항목마다 따로 적는다. 세 항목이 서로 다른 법정동·검정일에서 온 행 수도 적는다.

In [32]:
dong_s = df['legal_dong_code'].astype('string')
rows = []
for name, mask in (('전체', np.ones(N_ROWS, bool)), ('국내', IS_DOM), ('국내 검출', IS_DOM & y), ('국내 불검출', IS_DOM & ~y), ('해외', ~IS_DOM)):
    r = {'묶음': name, '행': int(mask.sum())}
    for c in SOIL_ITEMS:
        r[f'{c} 값 있음(%)'] = pct(SOIL[c].notna(), mask)
    for c in SOIL_ITEMS:
        r[f'{c} 법정동 = 시료의 legal_dong_code (%)'] = pct((SOIL[f'{c}_dong_code'].astype('string') == dong_s).fillna(False), mask)
    rows.append(r)
display(pd.DataFrame(rows))
print('국내 행 중 값이 없는 행:', {c: int((IS_DOM & SOIL[c].isna().to_numpy()).sum()) for c in SOIL_ITEMS})

days = SOIL[SOIL_DAYS_COLS].astype(float)
d3 = days.to_numpy()
code3 = SOIL[SOIL_DONG_COLS].to_numpy(dtype=object)
all3 = ~np.isnan(d3).any(axis=1)
same_dong = all3 & (code3[:, 0] == code3[:, 1]) & (code3[:, 1] == code3[:, 2])
same_day = same_dong & (d3[:, 0] == d3[:, 1]) & (d3[:, 1] == d3[:, 2])
print(f'세 항목 값이 모두 있는 국내 행 {int(all3.sum()):,} 중 세 값의 법정동이 모두 같은 행 {int(same_dong.sum()):,} · 법정동과 검정일이 모두 같은 행 {int(same_day.sum()):,}')

pcts = [.5, .75, .9, .99]
display(pd.DataFrame({**{f'{c} 법정동까지 거리(km)': SOIL[f'{c}_dist_km'].astype(float).describe(percentiles=pcts) for c in SOIL_ITEMS},
                      **{f'{c} 며칠 전 값인지(일)': days[f'{c}_days_before'].describe(percentiles=pcts) for c in SOIL_ITEMS},
                      **{f'{c} 그날 검정 행 수': SOIL[f'{c}_exam_n'].astype(float).describe(percentiles=pcts) for c in SOIL_ITEMS}}).round(2).T)
yr = df[DATE_COL].dt.year.to_numpy()
for c in SOIL_ITEMS:
    dd_all = days[f'{c}_days_before'].to_numpy()
    has_c = ~np.isnan(dd_all)
    dk = SOIL[f'{c}_dist_km'].astype(float).to_numpy()[has_c]
    dn = SOIL[f'{c}_exam_n'].astype(float).to_numpy()[has_c]
    print(f'[{c}] 값 있는 행 {int(has_c.sum()):,} · 거리 2km / 5km / 10km 이내: {float((dk <= 2).mean()) * 100:.1f}% / {float((dk <= 5).mean()) * 100:.1f}% / {float((dk <= 10).mean()) * 100:.1f}%'
          f' · 100일 / 365일 이내 값: {float((dd_all[has_c] <= 100).mean()) * 100:.1f}% / {float((dd_all[has_c] <= 365).mean()) * 100:.1f}% · 그날 그 항목 검정이 1행뿐: {float((dn == 1).mean()) * 100:.1f}%')
    for lim, name in ((365, '1년'), (365 * 3, '3년'), (365 * 5, '5년'), (365 * 10, '10년')):
        m_ = has_c & (dd_all > lim)
        print(f'    며칠 전 값인지가 {name}({lim}일) 넘는 행: {int(m_.sum()):,} (국내의 {m_.sum() / IS_DOM.sum() * 100:.2f}%) · 그중 검출 {int((m_ & y).sum())}')
    print('    기준일 연도별 "며칠 전 값인지" 중앙값(일):', {int(k): float(v) for k, v in pd.Series(dd_all[has_c]).groupby(yr[has_c]).median().items()})
    code_c = SOIL[f'{c}_dong_code'].astype('string')
    has_d = dong_s.notna().to_numpy() & has_c
    same8 = (code_c.str[:8] == dong_s.str[:8]).fillna(False).to_numpy()
    print(f'    시료에 법정동 코드가 있는 행 {int(has_d.sum()):,} 중, 값을 가져온 법정동이 같은 읍면동(코드 앞 8자리)인 행 {int((has_d & same8).sum()):,} ({(has_d & same8).sum() / has_d.sum() * 100:.1f}%)'
          f' · 거리가 0.01km 미만인 행 {int((SOIL[f"{c}_dist_km"].astype(float).to_numpy() < 0.01).sum()):,}')
display(SOIL.loc[IS_DOM, list(SOIL_ITEMS)].astype(float).describe(percentiles=[.01, .5, .99]).T)

SOIL_TABLE = pa.table({
    KEY_COL: pa.array(KEYS, type=pa.large_string()),
    **{c: pa.array(SOIL[c].to_numpy(np.float32), from_pandas=True) for c in SOIL_ITEMS},
    **{c: pa.array(SOIL[c].to_numpy(float), from_pandas=True).cast(pa.int32()) for c in SOIL_DAYS_COLS},
    **{c: pa.array(SOIL[c].to_numpy(dtype=object), type=pa.string(), from_pandas=True).dictionary_encode() for c in SOIL_DONG_COLS},
    **{c: pa.array(SOIL[c].to_numpy(np.float32), from_pandas=True) for c in SOIL_DIST_COLS},
    **{c: pa.array(SOIL[c].to_numpy(float), from_pandas=True).cast(pa.int32()) for c in SOIL_N_COLS},
})
assert SOIL_TABLE.column_names == [KEY_COL] + SOIL_COLS
pq.write_table(SOIL_TABLE, MID_DIR / 'soil.parquet', row_group_size=ROW_GROUP)
print('저장:', MID_DIR / 'soil.parquet', f'({SOIL_TABLE.num_rows:,}행 × {SOIL_TABLE.num_columns}칼럼)', {f.name: str(f.type) for f in SOIL_TABLE.schema})

,묶음,행,soil_ph 값 있음(%),soil_om 값 있음(%),soil_vldpha 값 있음(%),soil_ph 법정동 = 시료의 legal_dong_code (%),soil_om 법정동 = 시료의 legal_dong_code (%),soil_vldpha 법정동 = 시료의 legal_dong_code (%)
0,전체,676459,92.97,92.97,92.97,15.70,15.73,15.73
1,국내,628895,100.00,100.00,100.00,16.88,16.92,16.92
2,국내 검출,1844,100.00,100.00,100.00,14.10,14.15,14.15
3,국내 불검출,627051,100.00,100.00,100.00,16.89,16.93,16.92
4,해외,47564,0.00,0.00,0.00,0.00,0.00,0.00


국내 행 중 값이 없는 행: {'soil_ph': 0, 'soil_om': 0, 'soil_vldpha': 0}
세 항목 값이 모두 있는 국내 행 628,895 중 세 값의 법정동이 모두 같은 행 625,953 · 법정동과 검정일이 모두 같은 행 616,746


,count,mean,std,min,50%,75%,90%,99%,max
soil_ph 법정동까지 거리(km),628895.0,0.91,1.17,0.0,0.73,1.18,1.73,5.86,29.44
soil_om 법정동까지 거리(km),628895.0,0.91,1.17,0.0,0.73,1.17,1.73,5.86,29.44
soil_vldpha 법정동까지 거리(km),628895.0,0.91,1.17,0.0,0.73,1.17,1.72,5.86,29.44
soil_ph 며칠 전 값인지(일),628895.0,150.63,185.61,1.0,82.00,204.00,384.00,848.00,2389.00
soil_om 며칠 전 값인지(일),628895.0,148.92,184.19,1.0,81.00,202.00,376.00,846.00,2389.00
soil_vldpha 며칠 전 값인지(일),628895.0,149.13,184.20,1.0,82.00,202.00,377.00,844.00,2389.00
soil_ph 그날 검정 행 수,628895.0,4.17,8.38,1.0,2.00,4.00,8.00,35.00,324.00
soil_om 그날 검정 행 수,628895.0,4.17,8.26,1.0,2.00,4.00,8.00,34.00,324.00
soil_vldpha 그날 검정 행 수,628895.0,4.16,8.26,1.0,2.00,4.00,8.00,34.00,324.00


[soil_ph] 값 있는 행 628,895 · 거리 2km / 5km / 10km 이내: 92.3% / 98.8% / 99.6% · 100일 / 365일 이내 값: 55.5% / 89.2% · 그날 그 항목 검정이 1행뿐: 36.6%
    며칠 전 값인지가 1년(365일) 넘는 행: 67,791 (국내의 10.78%) · 그중 검출 168
    며칠 전 값인지가 3년(1095일) 넘는 행: 1,845 (국내의 0.29%) · 그중 검출 2
    며칠 전 값인지가 5년(1825일) 넘는 행: 83 (국내의 0.01%) · 그중 검출 0
    며칠 전 값인지가 10년(3650일) 넘는 행: 0 (국내의 0.00%) · 그중 검출 0
    기준일 연도별 "며칠 전 값인지" 중앙값(일): {2015: 12.0, 2016: 63.0, 2017: 98.0, 2018: 249.0, 2019: 167.0, 2020: 68.0, 2021: 67.0, 2022: 66.0, 2023: 68.0, 2024: 61.0, 2025: 68.0, 2026: 72.0}
    시료에 법정동 코드가 있는 행 628,895 중, 값을 가져온 법정동이 같은 읍면동(코드 앞 8자리)인 행 446,211 (71.0%) · 거리가 0.01km 미만인 행 108,305
[soil_om] 값 있는 행 628,895 · 거리 2km / 5km / 10km 이내: 92.3% / 98.8% / 99.6% · 100일 / 365일 이내 값: 55.8% / 89.5% · 그날 그 항목 검정이 1행뿐: 36.3%
    며칠 전 값인지가 1년(365일) 넘는 행: 65,753 (국내의 10.46%) · 그중 검출 167
    며칠 전 값인지가 3년(1095일) 넘는 행: 1,866 (국내의 0.30%) · 그중 검출 2
    며칠 전 값인지가 5년(1825일) 넘는 행: 83 (국내의 0.01%) · 그중 검출 0
    며칠 전 값인지가 10년(3650일) 넘는 행: 0 (국내의 0.00%) · 그

,count,mean,std,min,1%,50%,99%,max
soil_ph,628895.0,6.478243,0.840418,3.1,4.5,6.45,8.400,9.50
soil_om,628895.0,25.278923,19.359412,0.0,3.0,22.00,94.312,411.15
soil_vldpha,628895.0,433.742184,474.094214,0.0,9.0,242.50,2033.500,7496.00


저장: C:\Users\NT751\Desktop\천영진\리스테리아ML2\data\중간\soil.parquet (676,459행 × 16칼럼) {'lims_data_key': 'large_string', 'soil_ph': 'float', 'soil_om': 'float', 'soil_vldpha': 'float', 'soil_ph_days_before': 'int32', 'soil_om_days_before': 'int32', 'soil_vldpha_days_before': 'int32', 'soil_ph_dong_code': 'dictionary<values=string, indices=int32, ordered=0>', 'soil_om_dong_code': 'dictionary<values=string, indices=int32, ordered=0>', 'soil_vldpha_dong_code': 'dictionary<values=string, indices=int32, ordered=0>', 'soil_ph_dist_km': 'float', 'soil_om_dist_km': 'float', 'soil_vldpha_dist_km': 'float', 'soil_ph_exam_n': 'int32', 'soil_om_exam_n': 'int32', 'soil_vldpha_exam_n': 'int32'}


### 7-5. 규칙을 이렇게 정한 근거 — 눈금별 결과와 토양 값의 성질

7절 처음에 적은 "왜·어떻게"의 근거가 되는 계산이다. 데이터셋에 들어가는 값은 7-3절에서 이미 정해졌고, 이 절은 값을 바꾸지 않는다.

**(1) 눈금을 바꾸면 무엇이 달라지나** — `soil_ph` 를 네 가지 눈금으로 다시 골라 견준다.

| 눈금 | 뜻 |
|---|---|
| 경과를 보지 않음 | 기준일보다 앞선 그 항목 검정이 한 번이라도 있는 가장 가까운 법정동. 2026-10-08 전 규칙과 같은 방식이다(전 규칙은 법정동을 세 항목 공통으로 골랐다는 점만 다르다) |
| 1km = 2년 | 가까운 쪽을 더 중하게 본다 |
| 1km = 1년 | **채택** |
| 1km = 0.5년 | 최근인 쪽을 더 중하게 본다 |

- 읽는 법: 눈금이 "최근 쪽"으로 갈수록 오래된 값이 줄고 거리가 늘어난다. 표의 "채택한 눈금과 다른 후보를 고른 행"이 눈금을 달리 정했을 때 값이 바뀌는 행 수다.

**(2) 고를 여지가 있는 행이 얼마나 되나** — 5km 안의 후보 중 "더 가깝고 더 최근인 다른 후보가 없는 후보"를 센다. 1개면 가장 가까운 후보가 가장 최근 후보라 어떤 눈금이든 같은 후보를 고른다. 2개 이상이면 가까운 것과 최근 것 사이에서 골라야 한다.

**(3) 예** — 경과를 보지 않으면 10년 넘은 값이 붙는 행이 가장 많은 법정동의 시료 한 행을, 눈금별로 어느 법정동의 며칠 전 값을 받는지 적는다.

**(4) 토양 값은 시간이 지나면, 거리가 떨어지면 얼마나 달라지나** — 눈금을 자료로 정할 수 있는지 보려고, (법정동, 검정일) 값(그날 중앙값) 두 개의 차이를 두 가지로 잰다.

| 견주는 것 | 쌍을 만드는 방법 |
|---|---|
| 시간 | 같은 법정동의 서로 다른 두 검정일. 검정일마다 같은 법정동의 다른 검정일을 무작위로 3개 짝지음 |
| 거리 | 20km 안의 서로 다른 두 법정동(법정동마다 이웃을 무작위로 최대 20곳, 검정일은 최대 30개). 검정일이 가장 가까운 날끼리 짝짓고, 그 간격이 90일 이내인 쌍만 |

- 출력에서 읽은 것: 같은 법정동에서 90일 안에 다시 검정한 값끼리도 산도가 0.60(중앙값) 다르다. 8년 넘게 떨어지면 0.65, 10~20km 떨어지면 0.65다. 유기물·유효인산도 같은 모양이다. 곧 **시간·거리에 따른 차이보다 검정마다(필지마다) 값이 다른 쪽이 훨씬 크다.**
- 그래서 "몇 km 가 몇 년과 같은가"는 이 자료로 정해지지 않았고, 눈금 1km = 1년은 사용자 결정이다. 또 이 표는 10년 묵은 값이 최근 값보다 눈에 띄게 틀리다는 근거가 되지 못한다 — 규칙을 바꿔 얻은 것은 값의 정확도가 아니라 "오래된 값과 결측이 없다"는 점이다.
- 주의: 두 표는 쌍을 만든 법정동이 다르다(0~1km 쌍은 법정동이 촘촘한 곳에서만 나온다). 두 표의 숫자를 일대일로 맞대어 눈금을 계산하는 데 쓰지 않는다. 검정 대상은 농경지 필지이고, 시료 주소는 공장·도시 지역이 많다.

In [33]:
# (1) 눈금을 바꾸면 무엇이 달라지나 — soil_ph. 눈금 0 = 경과를 보지 않고 가장 가까운 법정동만 (2026-10-08 전 규칙 방식)
ITEM = 'soil_ph'
T = SOIL_DAYS[ITEM]
dom_y = y[dom_idx]
SCALE_NAMES = {0.0: '경과를 보지 않음 (가장 가까운 법정동)', 0.5: '1km = 2년', 1.0: '1km = 1년 (채택)', 2.0: '1km = 0.5년'}
scale_res, rows = {}, []
for kpy, name in SCALE_NAMES.items():
    pick, dist_km, pos, n_front, _ = soil_choose(ITEM, kpy, radius_km=5.0 if kpy == SOIL_KM_PER_YEAR else None)
    age = np.where(pos >= 0, base_c - T['no'][np.clip(pos, 0, None)], np.nan)
    scale_res[kpy] = (pick, pos, age, dist_km, n_front)
for kpy, name in SCALE_NAMES.items():
    pick, pos, age, dist_km, _ = scale_res[kpy]
    a_row, d_row = age[ci], dist_km[ci]                       # 국내 행 단위
    has = ~np.isnan(a_row)
    r = {'눈금': name, '값 없는 국내 행': int((~has).sum()),
         '거리 50% (km)': np.nanpercentile(d_row, 50), '거리 90% (km)': np.nanpercentile(d_row, 90), '거리 99% (km)': np.nanpercentile(d_row, 99),
         '며칠 전 50%': np.nanpercentile(a_row, 50), '며칠 전 90%': np.nanpercentile(a_row, 90), '며칠 전 99%': np.nanpercentile(a_row, 99)}
    for lim, nm in ((365, '1년'), (365 * 3, '3년'), (365 * 5, '5년'), (365 * 10, '10년')):
        r[f'{nm} 넘는 행'] = int((a_row > lim).sum())
    r['3년 넘는 행 중 검출'] = int(((a_row > 365 * 3) & dom_y).sum())
    dif = (pick != scale_res[SOIL_KM_PER_YEAR][0]) | (pos != scale_res[SOIL_KM_PER_YEAR][1])
    r['채택한 눈금과 다른 후보를 고른 행'] = int(dif[ci].sum())
    r['그중 검출'] = int((dif[ci] & dom_y).sum())
    rows.append(r)
print(f'[{ITEM}] 눈금별 결과 — 국내 {len(dom_idx):,}행')
display(pd.DataFrame(rows).round(2).set_index('눈금').T)

# (2) 고를 여지가 있는 행이 얼마나 되나 — 5km 안에서 "더 가깝고 더 최근인 다른 후보가 없는 후보" 수
nf = scale_res[SOIL_KM_PER_YEAR][4][ci]
vc = pd.Series(np.minimum(nf, 6)).value_counts().sort_index()
print('5km 안 후보 수별 국내 행 (6 = 6개 이상):', {int(k): f'{int(v):,} ({v / len(nf) * 100:.1f}%)' for k, v in vc.items()})
print('  0개 = 5km 안에 기준일보다 앞선 검정이 없음 · 1개 = 가장 가까운 후보가 가장 최근 후보라 고를 것이 없음 · 2개 이상 = 가까운 것과 최근 것 사이에서 골라야 함')

# (3) 예 — 경과를 보지 않으면 10년 넘은 값이 붙는 행이 가장 많은 법정동
age0, pick0 = scale_res[0.0][2][ci], scale_res[0.0][0][ci]
old = age0 > 3650
if old.any():
    worst = pd.Series(cand_code[pick0[old]]).value_counts().index[0]
    m = old & (cand_code[np.clip(pick0, 0, None)] == worst)
    i = np.nonzero(m)[0][0]
    r_df = dom_idx[i]
    print(f"경과를 보지 않으면 10년 넘은 {ITEM} 값이 붙는 국내 행 {int(old.sum()):,} · 그중 법정동 {worst}({CAND.loc[CAND['legal_dong_code'] == worst, 'legal_dong_nm'].iloc[0]})에서 온 행 {int(m.sum()):,}")
    ex = []
    for kpy, name in SCALE_NAMES.items():
        pick, pos, age, dist_km, _ = scale_res[kpy]
        c_i = pick[ci][i]
        ex.append({'눈금': name, '고른 법정동': cand_code[c_i], '법정동 이름': CAND.at[c_i, 'legal_dong_nm'], '거리(km)': round(float(dist_km[ci][i]), 2),
                   '며칠 전 값인지': int(age[ci][i]), '값': float(T['val'][pos[ci][i]])})
    print(f"  예: {df.at[r_df, KEY_COL]} · 기준일 {df.at[r_df, DATE_COL].date()} · 주소 {df.at[r_df, 'addr']}")
    display(pd.DataFrame(ex))
del scale_res, rows

[soil_ph] 눈금별 결과 — 국내 628,895행


눈금,경과를 보지 않음 (가장 가까운 법정동),1km = 2년,1km = 1년 (채택),1km = 0.5년
값 없는 국내 행,0.00,0.00,0.00,0.00
거리 50% (km),0.57,0.65,0.73,0.82
거리 90% (km),1.46,1.59,1.73,2.10
거리 99% (km),5.86,5.86,5.86,6.15
며칠 전 50%,101.00,91.00,82.00,69.00
며칠 전 90%,989.00,580.00,384.00,260.00
며칠 전 99%,2877.00,1277.00,848.00,545.00
1년 넘는 행,141988.00,106161.00,67791.00,23006.00
3년 넘는 행,55724.00,12473.00,1845.00,316.00
5년 넘는 행,24252.00,631.00,83.00,53.00


5km 안 후보 수별 국내 행 (6 = 6개 이상): {0: '7,810 (1.2%)', 1: '75,528 (12.0%)', 2: '164,825 (26.2%)', 3: '176,937 (28.1%)', 4: '115,220 (18.3%)', 5: '58,420 (9.3%)', 6: '30,155 (4.8%)'}
  0개 = 5km 안에 기준일보다 앞선 검정이 없음 · 1개 = 가장 가까운 후보가 가장 최근 후보라 고를 것이 없음 · 2개 이상 = 가까운 것과 최근 것 사이에서 골라야 함
경과를 보지 않으면 10년 넘은 soil_ph 값이 붙는 국내 행 674 · 그중 법정동 1168011800(서울특별시 강남구 도곡동)에서 온 행 208
  예: O000473_250102468_3_1 · 기준일 2025-04-11 · 주소 서울특별시 서초구 마방로 68


,눈금,고른 법정동,법정동 이름,거리(km),며칠 전 값인지,값
0,경과를 보지 않음 (가장 가까운 법정동),1168011800,서울특별시 강남구 도곡동,1.20,3668,5.8
1,1km = 2년,1165010800,서울특별시 서초구 서초동,1.37,253,6.6
2,1km = 1년 (채택),1165010800,서울특별시 서초구 서초동,1.37,253,6.6
3,1km = 0.5년,1165010800,서울특별시 서초구 서초동,1.37,253,6.6


In [34]:
# (4) 토양 값은 시간이 지나면, 거리가 떨어지면 얼마나 달라지나 — (법정동, 검정일) 값(그날 중앙값) 두 개의 차이
GAP_BINS, GAP_LABELS = [0, 90, 365, 730, 1095, 1826, 2922, 99999], ['1~90일', '91일~1년', '1~2년', '2~3년', '3~5년', '5~8년', '8년 넘음']
DIST_BINS, DIST_LABELS = [0, 1, 2, 3, 5, 10, 20], ['0~1km', '1~2km', '2~3km', '3~5km', '5~10km', '10~20km']
NEAR_DAYS, MAX_NEIGHBORS, MAX_DAYS = 90, 20, 30
rng = np.random.default_rng(SEED)

def diff_table(bins, labels, x, v):
    g = pd.DataFrame({'구간': pd.cut(x, bins, labels=labels, include_lowest=True), 'v': v}).groupby('구간', observed=True)['v']
    return pd.DataFrame({'쌍 수': g.size(), '차이 중앙값': g.median(), '차이 75%': g.quantile(.75)})

t0 = time.time()
near = []                                                 # 후보 법정동마다 20km 안의 다른 후보 법정동(많으면 무작위 20곳)과 거리
for c0 in range(0, len(CAND), 500):
    dm = haversine_km(cand_xy[c0:c0 + 500, None, 0], cand_xy[c0:c0 + 500, None, 1], cand_xy[None, :, 0], cand_xy[None, :, 1])
    for r in range(dm.shape[0]):
        nb = np.nonzero((dm[r] <= DIST_BINS[-1]) & (np.arange(len(CAND)) != c0 + r))[0]
        if len(nb) > MAX_NEIGHBORS:
            nb = rng.choice(nb, MAX_NEIGHBORS, replace=False)
        near.append((nb, dm[r, nb]))
for c, (src, ko) in SOIL_ITEMS.items():
    T = SOIL_DAYS[c]
    st = np.searchsorted(T['cand'], np.arange(len(CAND) + 1), side='left')      # 후보 법정동마다 항목 표의 시작 위치
    days_, vals_ = T['no'], T['val']
    # 시간: 같은 법정동의 서로 다른 두 검정일 (검정일마다 무작위 상대 3개)
    gi, gj = [], []
    for a, b in zip(st[:-1], st[1:]):
        if b - a >= 2:
            i = np.repeat(np.arange(a, b), 3)
            j = a + rng.integers(0, b - a, len(i))
            gi.append(i[i != j]); gj.append(j[i != j])
    gi, gj = np.concatenate(gi), np.concatenate(gj)
    by_time = diff_table(GAP_BINS, GAP_LABELS, np.abs(days_[gi] - days_[gj]), np.abs(vals_[gi] - vals_[gj]))
    # 거리: 20km 안의 다른 법정동과, 검정일이 90일 이내인 날끼리 (법정동마다 검정일 최대 30개)
    dd, vv = [], []
    for i in range(len(CAND)):
        a, b = st[i], st[i + 1]
        if a == b:
            continue
        sel = np.arange(a, b) if b - a <= MAX_DAYS else rng.choice(np.arange(a, b), MAX_DAYS, replace=False)
        for j, dij in zip(*near[i]):
            a2, b2 = st[j], st[j + 1]
            if a2 == b2:
                continue
            p = np.searchsorted(days_[a2:b2], days_[sel])
            lo, hi = np.clip(p - 1, 0, b2 - a2 - 1), np.clip(p, 0, b2 - a2 - 1)
            q = a2 + np.where(np.abs(days_[a2 + lo] - days_[sel]) <= np.abs(days_[a2 + hi] - days_[sel]), lo, hi)   # 검정일이 가장 가까운 날
            okq = np.abs(days_[q] - days_[sel]) <= NEAR_DAYS
            if okq.any():
                dd.append(np.full(int(okq.sum()), dij)); vv.append(np.abs(vals_[q[okq]] - vals_[sel[okq]]))
    by_dist = diff_table(DIST_BINS, DIST_LABELS, np.concatenate(dd), np.concatenate(vv))
    print(f'[{c} · {ko}] 같은 법정동, 두 검정일 사이 간격별 값 차이 / 서로 다른 법정동(검정일 {NEAR_DAYS}일 이내), 거리별 값 차이')
    display(by_time.round(3).T)
    display(by_dist.round(3).T)
print(f'계산 {time.time() - t0:.0f}초')
del near, combo, SOIL_DAYS, SOIL_PICK; gc.collect();

[soil_ph · 산도] 같은 법정동, 두 검정일 사이 간격별 값 차이 / 서로 다른 법정동(검정일 90일 이내), 거리별 값 차이


구간,1~90일,91일~1년,1~2년,2~3년,3~5년,5~8년,8년 넘음
쌍 수,164365.00,409391.0,482392.0,406399.0,693915.0,649083.0,296253.00
차이 중앙값,0.60,0.6,0.6,0.6,0.6,0.6,0.65
차이 75%,1.05,1.1,1.1,1.1,1.1,1.1,1.10


구간,0~1km,1~2km,2~3km,3~5km,5~10km,10~20km
쌍 수,5227.00,60974.0,124602.00,359611.00,1441146.0,5170189.00
차이 중앙값,0.50,0.5,0.55,0.60,0.6,0.65
차이 75%,0.95,1.0,1.00,1.05,1.1,1.15


[soil_om · 유기물] 같은 법정동, 두 검정일 사이 간격별 값 차이 / 서로 다른 법정동(검정일 90일 이내), 거리별 값 차이


구간,1~90일,91일~1년,1~2년,2~3년,3~5년,5~8년,8년 넘음
쌍 수,165897.0,412433.00,485782.00,411858.0,702999.0,660599.0,299342.0
차이 중앙값,7.8,8.25,8.25,8.5,8.5,8.9,9.0
차이 75%,15.0,16.00,16.00,16.0,16.2,16.5,17.0


구간,0~1km,1~2km,2~3km,3~5km,5~10km,10~20km
쌍 수,5299.0,61553.0,125636.0,362947.0,1455056.0,5220974.0
차이 중앙값,6.5,7.0,7.1,8.0,8.5,9.0
차이 75%,13.0,14.0,14.4,15.0,16.0,17.0


[soil_vldpha · 유효인산] 같은 법정동, 두 검정일 사이 간격별 값 차이 / 서로 다른 법정동(검정일 90일 이내), 거리별 값 차이


구간,1~90일,91일~1년,1~2년,2~3년,3~5년,5~8년,8년 넘음
쌍 수,165229.0,411265.0,484962.0,410748.0,700835.00,660024.000,299508.0
차이 중앙값,194.5,211.0,208.1,211.5,216.05,219.300,232.0
차이 75%,461.0,480.0,478.3,482.5,488.50,493.112,514.5


구간,0~1km,1~2km,2~3km,3~5km,5~10km,10~20km
쌍 수,5336.0,61557.00,125663.0,362716.0,1455028.0,5219008.00
차이 중앙값,144.0,151.40,164.3,184.0,207.5,230.95
차이 75%,386.0,409.75,425.0,449.5,481.1,511.50


계산 43초


## 8. 조인 — 검사 목적·시료 상태·유통기한 변수

사용자 메모(2026-10-06)에서 "제조 공정 관련 변수"로 지정된 네 칼럼이다. 내용은 제조 공정이 아니라 **검사를 의뢰할 때 적힌 정보(검사 목적·시료 상태·유통 일)와 품목제조대장의 유통기한**이어서, 이 노트북에서는 묶음 이름을 내용대로 적는다.

| 칼럼 | 원천 (DB 코멘트) | 조인 |
|---|---|---|
| `inspct_purps_code` · `inspct_purps_nm` | `ba_data.m_org_lims_data_info.inspct_purps_code` ("ZIM02 검사 목적 코드") | `lims_data_key` 로 1:1 |
| `splore_sttus_code` · `splore_sttus_nm` | 같은 테이블 `splore_sttus_code` ("ZIM11 시료 상태 코드") | 〃 |
| `distb_de` | 같은 테이블 `distb_de` ("유통 일") | 〃 |
| `pog_daycnt` | `ba_data.t_prdlst_mnf_ledg.pog_daycnt` ("유통기한") — 문장 그대로 | 품목제조번호 (8-2절) |

- 값은 DB 값 그대로 둔다(변환·이상값 처리 없음). `distb_de` 가 어떤 날짜인지는 DB 코멘트·테이블정의서에 위 문구 말고는 설명이 없다 — 확인 필요.
- 같은 테이블의 `cnvync_sttus_tp`("운반 상태 온도")는 **넣지 않는다**(사용자 결정 2026-10-06). 값이 있는 시료가 2,474행(0.37%)뿐이다(2026-10-06 조회).
- 코드 칼럼은 DB 코드(`IM02000007` 등)를 그대로 두고, `ba_data.t_cm_cmmn_code`(`cmmn_cl_code` = `IM02`·`IM11`)의 `cmmn_code_nm` 을 `_nm` 칼럼으로 함께 붙인다. DB 코멘트에는 "ZIM02"·"ZIM11" 로 적혀 있으나 공통코드표의 분류 이름은 `IM02`·`IM11` 이다.
- `ba_data.m_org_lims_data_info` 는 식약처 LIMS 원본이라, 이 테이블에 행이 없는 시료는 세 칼럼이 모두 결측이다.

### 8-1. 식약처 LIMS 원본 3칼럼 (`lims_data_key` 로 1:1)

In [35]:
m_org = pd.read_parquet(RAW_DIR / 'raw_listeria_m_org.parquet')
print(f'raw_listeria_m_org: {len(m_org):,}행 · lims_data_key 중복 {int(m_org[KEY_COL].duplicated().sum())}')
assert m_org[KEY_COL].is_unique, '같은 lims_data_key 가 두 행 이상 — 1:1 조인이 안 됨'
assert df[KEY_COL].isin(set(m_org[KEY_COL])).all(), '시료 파일의 키 중 DB 에 없는 것이 있음'
m_org = m_org.set_index(KEY_COL).reindex(df[KEY_COL]).reset_index()   # 시료 파일 행 순서로
assert len(m_org) == N_ROWS and (m_org[KEY_COL].to_numpy() == df[KEY_COL].to_numpy()).all()
has_m = m_org['has_m_org'].to_numpy(bool)
print(f'시료 {N_ROWS:,}행 중 ba_data.m_org_lims_data_info 에 행이 있는 시료 {int(has_m.sum()):,} · 없는 시료 {int((~has_m).sum()):,}'
      f' (없는 시료 중 국내 {int((~has_m & IS_DOM).sum()):,} · 해외 {int((~has_m & ~IS_DOM).sum()):,})')

codes = pd.read_parquet(RAW_DIR / 'raw_t_cm_cmmn_code_IM02_IM11.parquet')
assert codes['cmmn_code'].is_unique
display(codes[['cmmn_cl_code', 'cmmn_code', 'cmmn_code_nm']].sort_values('cmmn_code').reset_index(drop=True))
CODE_NM = codes.set_index('cmmn_code')['cmmn_code_nm']

PROC = pd.DataFrame({KEY_COL: df[KEY_COL].to_numpy()})
for c in ('inspct_purps_code', 'splore_sttus_code'):
    s = m_org[c]
    PROC[c] = s.to_numpy()
    PROC[c.replace('_code', '_nm')] = s.map(CODE_NM).to_numpy()
    no_name = s.notna() & s.map(CODE_NM).isna()
    print(f"{c}: 값 있음 {int(s.notna().sum()):,} · 빈 문자열 {int((s.astype('string').str.strip() == '').sum())} · 공통코드표에 이름이 없는 값 {int(no_name.sum())}{' ' + str(sorted(s[no_name].unique())) if no_name.any() else ''}")
PROC['distb_de'] = m_org['distb_de'].to_numpy()

for c, nm in (('inspct_purps_code', 'inspct_purps_nm'), ('splore_sttus_code', 'splore_sttus_nm')):
    t = pd.DataFrame({c: PROC[c].fillna('(결측)'), nm: PROC[nm].fillna(''), '검출': y}).groupby([c, nm]).agg(행=('검출', 'size'), 검출=('검출', 'sum'))
    t['검출 비율(%)'] = (t['검출'] / t['행'] * 100).round(3)
    display(t.sort_values('행', ascending=False))

d = pd.Series(PROC['distb_de'])
dv = d.dropna()
print(f"distb_de: 값 있음 {len(dv):,} · 가장 이른 값 {dv.min()} · 가장 늦은 값 {dv.max()}")
yr = dv.map(lambda x: x.year)
print(f"  연도가 2000 보다 앞인 값 {int((yr < 2000).sum())}건 · 2035 보다 뒤인 값 {int((yr > 2035).sum())}건 (고치지 않고 그대로 둔다)")
base_date = df[DATE_COL].dt.date
later = pd.Series([a > b for a, b in zip(d[d.notna()], base_date[d.notna().to_numpy()])])
print(f"  distb_de 가 기준일(mnfctur_de)보다 뒤인 행 {int(later.sum()):,} / {len(later):,}")

raw_listeria_m_org: 765,891행 · lims_data_key 중복 0


시료 676,459행 중 ba_data.m_org_lims_data_info 에 행이 있는 시료 669,257 · 없는 시료 7,202 (없는 시료 중 국내 0 · 해외 7,202)


,cmmn_cl_code,cmmn_code,cmmn_code_nm
0,IM02,IM02000001,수입검사
1,IM02,IM02000002,검사명령검사
2,IM02,IM02000003,수거검사
3,IM02,IM02000004,자가품질위탁검사
4,IM02,IM02000005,품질검사
5,IM02,IM02000006,허가.인증 신청 및 신고 검사
6,IM02,IM02000007,기타
7,IM02,IM02000008,압·몰수품검사
8,IM02,IM02000009,모니터링검사
9,IM02,IM02000010,국가출하승인


inspct_purps_code: 값 있음 669,257 · 빈 문자열 0 · 공통코드표에 이름이 없는 값 0
splore_sttus_code: 값 있음 56,703 · 빈 문자열 0 · 공통코드표에 이름이 없는 값 0


,,행,검출,검출 비율(%)
inspct_purps_code,inspct_purps_nm,,,
IM02000007,기타,326825,1619,0.495
IM02000004,자가품질위탁검사,198217,127,0.064
IM02000003,수거검사,39501,66,0.167
IM02000001,수입검사,39468,96,0.243
IM02000005,품질검사,32608,3,0.009
IM02000014,식품안전관리인증검사,26426,19,0.072
(결측),,7202,10,0.139
IM02000015,자가품질직접검사(영업자),4322,0,0.000
IM02000009,모니터링검사,1870,29,1.551


,,행,검출,검출 비율(%)
splore_sttus_code,splore_sttus_nm,,,
(결측),,619756,1823,0.294
IM11000002,냉장,29959,60,0.200
IM11000003,냉동,17047,77,0.452
IM11000001,실온,9697,9,0.093


distb_de: 값 있음 190,471 · 가장 이른 값 0019-01-19 · 가장 늦은 값 2999-08-02
  연도가 2000 보다 앞인 값 2건 · 2035 보다 뒤인 값 2건 (고치지 않고 그대로 둔다)
  distb_de 가 기준일(mnfctur_de)보다 뒤인 행 184,172 / 190,471


### 8-2. 품목제조대장 유통기한 `pog_daycnt`

2026-09-28 에 정한 규칙(노트북 `리스테리아ML/notebook/6-2_품목제조대장 pog_daycnt 조인 결측.ipynb`) 그대로다.

- 경로: 시료 `lims_data_key` → `ready_data.hazard_preprocessed_data.prdlst_mnfctur_sttemnt_no`(정의서 "품목제조번호") → `ba_data.t_prdlst_mnf_ledg.prdlst_report_no`(DB 코멘트 "품목 보고 번호")
- 번호는 **원문 그대로 같을 때만** 잇는다(글자를 지우는 등의 변환 없음).
- 대장에 **같은 번호가 여러 행이면 연결하지 않는다**(결측).
- `pog_daycnt` 는 "제조일로부터 12개월" 같은 문장이다. 숫자로 바꾸지 않고 원문 그대로 둔다.
- 품목제조대장은 받은 시점의 현재 값이다(시료를 검사한 때의 값이 아닐 수 있다).

In [36]:
ledg = pd.read_parquet(LEDG_PATH)
print(f'raw_t_prdlst_mnf_ledg_listeria: {len(ledg):,}행 × {ledg.shape[1]}칼럼 · 고유 번호 {ledg["prdlst_report_no"].nunique():,}')
n_per_no = ledg.groupby('prdlst_report_no').size()
no = m_org['prdlst_mnfctur_sttemnt_no']
has_no = (no.notna() & (no.astype('string').str.strip() != '')).to_numpy(bool)
n_ledg = no.map(n_per_no).fillna(0).to_numpy(int)          # 그 번호의 대장 행 수 (없으면 0)
ledg1 = ledg[ledg['prdlst_report_no'].isin(n_per_no[n_per_no == 1].index)].set_index('prdlst_report_no')
pog = no.where(has_no & (n_ledg == 1)).map(ledg1['pog_daycnt'])
PROC['pog_daycnt'] = pog.to_numpy()
PROC_LEDG_NO = no.where(has_no & (n_ledg == 1)).to_numpy()      # 연결된 대장 번호 (8-4절에서 대장의 다른 칼럼을 볼 때 쓴다)
has_pog = (pog.notna() & (pog.astype('string').str.strip() != '')).to_numpy(bool)

case = np.select([~has_no, n_ledg == 0, n_ledg > 1, ~has_pog], ['① 시료에 품목제조번호 없음', '② 번호가 대장에 없음', '③ 대장에 같은 번호 여러 행 → 연결 안 함', '④ 대장 1행인데 pog_daycnt 가 비어 있음'], '⑤ 값 있음')
t = pd.DataFrame({'구분': case, '검출': y, '국내': IS_DOM}).groupby('구분').agg(행=('검출', 'size'), 검출=('검출', 'sum'), 국내=('국내', 'sum'))
t['비율(%)'] = (t['행'] / N_ROWS * 100).round(2)
display(t)
print('2026-09-28 측정값(작업기록): ① 472,422 · ② 1,189 · ③ 2,770 · ④ 8,525 · ⑤ 191,631')
print(f"pog_daycnt 빈 문자열 {int((pog.astype('string').str.strip() == '').sum())}건 · 고유 문장 {pog.nunique():,}개")
print('많이 나오는 문장 10개:')
display(pog.value_counts().head(10).rename('행').rename_axis('pog_daycnt 문장').to_frame())

raw_t_prdlst_mnf_ledg_listeria: 35,808행 × 37칼럼 · 고유 번호 35,089


,행,검출,국내,비율(%)
구분,,,,
① 시료에 품목제조번호 없음,472344,1744,424780,69.83
② 번호가 대장에 없음,1189,1,1189,0.18
③ 대장에 같은 번호 여러 행 → 연결 안 함,2770,3,2770,0.41
④ 대장 1행인데 pog_daycnt 가 비어 있음,8525,0,8525,1.26
⑤ 값 있음,191631,221,191631,28.33


2026-09-28 측정값(작업기록): ① 472,422 · ② 1,189 · ③ 2,770 · ④ 8,525 · ⑤ 191,631
pog_daycnt 빈 문자열 0건 · 고유 문장 6,984개
많이 나오는 문장 10개:


,행
pog_daycnt 문장,
제조일로부터 12개월,5505
제조일로부터 6개월,3471
제조일로부터 60일,3372
제조일로부터 12개월까지,3199
제조일로부터 30일,2458
제조일로부터 1년,2324
제조일로부터 20일,2264
제조일로부터 3개월,2220
12개월,1718


### 8-3. 저장과 결측 비율

"값 있음"은 NULL·빈 문자열이 아닌 것. 0 은 값으로 센다.

In [37]:
PROC_COLS = ['inspct_purps_code', 'inspct_purps_nm', 'splore_sttus_code', 'splore_sttus_nm', 'distb_de', 'pog_daycnt']
PROC_KO = {'inspct_purps_code': '검사 목적 코드', 'inspct_purps_nm': '검사 목적 이름', 'splore_sttus_code': '시료 상태 코드',
           'splore_sttus_nm': '시료 상태 이름', 'distb_de': '유통 일', 'pog_daycnt': '유통기한(문장)'}

def has_value(s):
    s = pd.Series(s)
    if s.dtype == object or str(s.dtype) in ('str', 'string'):
        return (s.notna() & (s.astype('string').str.strip() != '').fillna(True)).to_numpy(bool)
    return s.notna().to_numpy(bool)

rows = []
for c in PROC_COLS:
    h = has_value(PROC[c])
    rows.append({'칼럼': c, '이름': PROC_KO[c], '값 있음': int(h.sum()), '결측(%)': round((1 - h.mean()) * 100, 2),
                 '국내 결측(%)': round((1 - h[IS_DOM].mean()) * 100, 2), '해외 결측(%)': round((1 - h[~IS_DOM].mean()) * 100, 2),
                 '검출 행 결측(%)': round((1 - h[y].mean()) * 100, 2), '불검출 행 결측(%)': round((1 - h[~y].mean()) * 100, 2),
                 '고유값 수': int(pd.Series(PROC[c])[h].nunique())})
PROC_SUMMARY = pd.DataFrame(rows)
display(PROC_SUMMARY)

# 저장 — 날짜는 DB 자료형(date) 그대로, 코드·이름은 범주형, 문장은 문자열
def arr_str(col, typ=pa.string()):
    return pa.array(PROC[col].to_numpy(dtype=object), type=typ, from_pandas=True)   # 결측(NaN·None)은 null 로

PROC_TABLE = pa.table({
    KEY_COL: pa.array(KEYS, type=pa.large_string()),
    'inspct_purps_code': arr_str('inspct_purps_code').dictionary_encode(),
    'inspct_purps_nm': arr_str('inspct_purps_nm').dictionary_encode(),
    'splore_sttus_code': arr_str('splore_sttus_code').dictionary_encode(),
    'splore_sttus_nm': arr_str('splore_sttus_nm').dictionary_encode(),
    'distb_de': pa.array([None if pd.isna(v) else v for v in PROC['distb_de'].tolist()], type=pa.date32()),
    'pog_daycnt': arr_str('pog_daycnt', pa.large_string()),
})
assert PROC_TABLE.num_rows == N_ROWS
pq.write_table(PROC_TABLE, MID_DIR / 'process.parquet', row_group_size=ROW_GROUP)
print('저장:', MID_DIR / 'process.parquet', f'({PROC_TABLE.num_rows:,}행 × {PROC_TABLE.num_columns}칼럼)')
print({f.name: str(f.type) for f in PROC_TABLE.schema})
del ledg, ledg1, m_org; gc.collect();

,칼럼,이름,값 있음,결측(%),국내 결측(%),해외 결측(%),검출 행 결측(%),불검출 행 결측(%),고유값 수
0,inspct_purps_code,검사 목적 코드,669257,1.06,0.00,15.14,0.51,1.07,11
1,inspct_purps_nm,검사 목적 이름,669257,1.06,0.00,15.14,0.51,1.07,11
2,splore_sttus_code,시료 상태 코드,56703,91.62,97.38,15.46,92.59,91.61,3
3,splore_sttus_nm,시료 상태 이름,56703,91.62,97.38,15.46,92.59,91.61,3
4,distb_de,유통 일,190471,71.84,69.73,99.79,87.71,71.80,4539
5,pog_daycnt,유통기한(문장),191631,71.67,69.53,100.00,88.78,71.62,6984


저장: C:\Users\NT751\Desktop\천영진\리스테리아ML2\data\중간\process.parquet (676,459행 × 7칼럼)
{'lims_data_key': 'large_string', 'inspct_purps_code': 'dictionary<values=string, indices=int32, ordered=0>', 'inspct_purps_nm': 'dictionary<values=string, indices=int32, ordered=0>', 'splore_sttus_code': 'dictionary<values=string, indices=int32, ordered=0>', 'splore_sttus_nm': 'dictionary<values=string, indices=int32, ordered=0>', 'distb_de': 'date32[day]', 'pog_daycnt': 'large_string'}


### 8-4. 이 변수들에서 설명이 필요한 점 (2026-10-06 추가)

값을 고치지는 않는다. 보고서에 쓸 때 설명이 필요한 점을 수치로 적어 둔다.

1. **검사 목적은 제조 공정 정보가 아니다.** 검사를 의뢰한 목적이고, 값 '기타'에 행과 검출이 가장 많이 몰려 있다.
2. **다른 세 칼럼은 검사 목적에 따라 값이 있고 없고가 갈린다.** 검사 목적별로 값이 있는 행 수를 적는다.
3. **`distb_de`("유통 일")는 값으로 보면 유통기한이 끝나는 날짜로 보인다.** 기준일이 제조일인 행에서 (`distb_de` − 기준일)의 중앙값이 같은 행의 `pog_daycnt` 문장과 맞는지 아래에서 본다. DB 코멘트·테이블정의서에는 "유통 일"이라는 말뿐이라 **식정원 확인이 필요하다.** 맞다면 `pog_daycnt`(유통기한 문장)와 같은 정보가 두 칼럼에 들어 있는 셈이다.
4. **`distb_de` 에 날짜로 보기 어려운 값이 그대로 있다** — 연도가 2000 보다 앞이거나 2035 보다 뒤인 값, 기준일보다 앞선 값.

In [38]:
purp = pd.Series(PROC['inspct_purps_nm']).fillna('(결측)').to_numpy()
has_col = {c: has_value(PROC[c]) for c in ('splore_sttus_code', 'distb_de', 'pog_daycnt')}
t = pd.DataFrame({'검사 목적': purp, '행': 1, '검출': y.astype(int), '시료 상태 값 있음': has_col['splore_sttus_code'].astype(int),
                  '유통 일 값 있음': has_col['distb_de'].astype(int), '유통기한 값 있음': has_col['pog_daycnt'].astype(int)}).groupby('검사 목적').sum()
t.insert(2, '검출 중(%)', (t['검출'] / y.sum() * 100).round(1))
t.insert(1, '행(%)', (t['행'] / N_ROWS * 100).round(2))
print('1)·2) 검사 목적별 행·검출과, 다른 세 칼럼에 값이 있는 행 수')
display(t.sort_values('행', ascending=False))

print('3) 기준일이 제조일인 행에서 (distb_de − 기준일) 일수 — 같은 행의 pog_daycnt 문장별')
base_d = df[DATE_COL].dt.date.to_numpy()
dist_d = PROC['distb_de'].to_numpy()
made = (df['date_basis'].astype(str) == '제조일').to_numpy() & has_col['distb_de']
gap = pd.Series(np.array([(a - b).days for a, b in zip(dist_d[made], base_d[made])]), index=np.nonzero(made)[0])
pog_made = pd.Series(PROC['pog_daycnt']).iloc[gap.index]
g = pd.DataFrame({'pog_daycnt 문장': pog_made.to_numpy(), '일수': gap.to_numpy()}).dropna(subset=['pog_daycnt 문장'])
top = g['pog_daycnt 문장'].value_counts().head(12).index
gg = g[g['pog_daycnt 문장'].isin(top)].groupby('pog_daycnt 문장')['일수']
display(pd.DataFrame({'행': gg.size(), '중앙값': gg.median(), '하위 25%': gg.quantile(.25), '상위 25%': gg.quantile(.75), '최솟값': gg.min(), '최댓값': gg.max()})
        .sort_values('행', ascending=False))
print(f'기준일이 제조일이고 distb_de 가 있는 행 {int(made.sum()):,} · 그중 pog_daycnt 도 있는 행 {len(g):,}')

print('4) distb_de 에 그대로 있는 값')
hv = has_col['distb_de']
yrs = np.array([d.year for d in dist_d[hv]])
cmp_ = np.array([(a - b).days for a, b in zip(dist_d[hv], base_d[hv])])
print(f'값 있는 행 {int(hv.sum()):,} · 연도가 2000 보다 앞 {int((yrs < 2000).sum())}행 · 2035 보다 뒤 {int((yrs > 2035).sum())}행'
      f' · 기준일보다 앞 {int((cmp_ < 0).sum()):,}행 · 기준일과 같은 날 {int((cmp_ == 0).sum()):,}행 · 기준일보다 뒤 {int((cmp_ > 0).sum()):,}행')
del gap, g, cmp_, yrs

1)·2) 검사 목적별 행·검출과, 다른 세 칼럼에 값이 있는 행 수


,행,행(%),검출,검출 중(%),시료 상태 값 있음,유통 일 값 있음,유통기한 값 있음
검사 목적,,,,,,,
기타,326825,48.31,1619,82.2,1,26990,7479
자가품질위탁검사,198217,29.30,127,6.4,19,114148,146598
수거검사,39501,5.84,66,3.4,16464,24577,15811
수입검사,39468,5.83,96,4.9,39458,1,1
품질검사,32608,4.82,3,0.2,0,20816,18432
식품안전관리인증검사,26426,3.91,19,1.0,0,1261,565
(결측),7202,1.06,10,0.5,0,0,0
자가품질직접검사(영업자),4322,0.64,0,0.0,0,2604,2703
모니터링검사,1870,0.28,29,1.5,761,70,40


3) 기준일이 제조일인 행에서 (distb_de − 기준일) 일수 — 같은 행의 pog_daycnt 문장별


,행,중앙값,하위 25%,상위 25%,최솟값,최댓값
pog_daycnt 문장,,,,,,
제조일로부터 12개월,2780,364.0,361.0,364.0,-330,730
제조일로부터 12개월까지,1849,364.0,364.0,364.0,0,730
제조일로부터 6개월,1656,181.0,180.0,183.0,0,549
제조일로부터 60일,1538,59.0,44.0,59.0,-303,425
제조일로부터 3개월,1362,91.0,89.0,91.0,19,821
제조일로부터 1년,1194,364.0,293.0,364.0,-468,381
제조일로부터 20일,1187,17.0,14.0,19.0,0,273
제조일로부터 30일,1158,29.0,28.0,29.0,8,396
제조시간으로부터 48시간까지,1094,2.0,2.0,2.0,0,4


기준일이 제조일이고 distb_de 가 있는 행 124,737 · 그중 pog_daycnt 도 있는 행 93,144
4) distb_de 에 그대로 있는 값


값 있는 행 190,471 · 연도가 2000 보다 앞 2행 · 2035 보다 뒤 2행 · 기준일보다 앞 690행 · 기준일과 같은 날 5,609행 · 기준일보다 뒤 184,172행


품목제조대장 값의 시점과, 대장에 있지만 넣지 않은 칼럼:

- 대장(`ba_data.t_prdlst_mnf_ledg`)은 받은 시점의 현재 값이다. 수정 이력이 없어, 기준일 뒤에 수정된 행의 유통기한이 그때도 같았는지는 알 수 없다.
- `pog_daycnt` 의 "값 있음"에는 숫자가 없는 문장("해당없음"·"-" 등)도 들어 있다.
- 같은 대장 행에 `frmlc_mtrqlt`(DB 코멘트 "포장 재질")가 있다. PPT 추가 예측인자의 "포장 유형"에 가까운 칼럼이지만 요청에 없어 넣지 않았다.

In [39]:
led = pd.read_parquet(LEDG_PATH, columns=['prdlst_report_no', 'last_updt_dtm', 'prms_dt', 'del_yn', 'frmlc_mtrqlt', 'prsrv_distb_stdr_cn'])
led = led[~led['prdlst_report_no'].duplicated(keep=False)].set_index('prdlst_report_no')       # 번호가 한 행뿐인 대장 행
no_used = pd.Series(PROC_LEDG_NO)
linked = no_used.notna().to_numpy()
has_pog_ = has_value(PROC['pog_daycnt'])
base_ts = df[DATE_COL].to_numpy().astype('datetime64[D]')
def later_than_base(col):
    v = pd.to_datetime(no_used.map(led[col]), errors='coerce').to_numpy().astype('datetime64[D]')
    return (v > base_ts) & ~np.isnat(v)
upd_later, prms_later = later_than_base('last_updt_dtm'), later_than_base('prms_dt')
print(f'대장에 연결된 시료 {int(linked.sum()):,}행 · pog_daycnt 값이 있는 시료 {int(has_pog_.sum()):,}행')
print(f"  pog_daycnt 값이 있는 행 중 last_updt_dtm(최종 수정 일시)이 기준일보다 뒤인 행 {int((has_pog_ & upd_later).sum()):,} ({(has_pog_ & upd_later).sum() / has_pog_.sum() * 100:.1f}%)"
      f" · prms_dt 가 기준일보다 뒤인 행 {int((has_pog_ & prms_later).sum()):,} · del_yn 이 'Y' 인 행 {int((has_pog_ & (no_used.map(led['del_yn']) == 'Y').to_numpy()).sum()):,}")
pog_s = pd.Series(PROC['pog_daycnt']).astype('string')
no_digit = has_pog_ & ~pog_s.str.contains(r'[0-9]', regex=True).fillna(False).to_numpy()
print(f'  pog_daycnt 문장에 숫자가 하나도 없는 행 {int(no_digit.sum()):,} — 많이 나오는 문장:', pog_s[no_digit].value_counts().head(6).to_dict())
for col, cmt in (('frmlc_mtrqlt', '포장 재질'), ('prsrv_distb_stdr_cn', '보존 유통 기준 내용')):
    hv_ = has_value(no_used.map(led[col]))
    print(f'  (넣지 않은 칼럼) {col} "{cmt}" — 대장에 연결된 시료 중 값이 있는 행 {int((linked & hv_).sum()):,} ({(linked & hv_).sum() / max(int(linked.sum()), 1) * 100:.1f}%)')
del led

대장에 연결된 시료 200,156행 · pog_daycnt 값이 있는 시료 191,631행
  pog_daycnt 값이 있는 행 중 last_updt_dtm(최종 수정 일시)이 기준일보다 뒤인 행 123,741 (64.6%) · prms_dt 가 기준일보다 뒤인 행 624 · del_yn 이 'Y' 인 행 65
  pog_daycnt 문장에 숫자가 하나도 없는 행 6,132 — 많이 나오는 문장: {'해당없음': 833, '-': 822, '없음': 627, '제조일자 표기': 413, '제조일표기': 340, '해당 없음': 314}
  (넣지 않은 칼럼) frmlc_mtrqlt "포장 재질" — 대장에 연결된 시료 중 값이 있는 행 179,925 (89.9%)


  (넣지 않은 칼럼) prsrv_distb_stdr_cn "보존 유통 기준 내용" — 대장에 연결된 시료 중 값이 있는 행 307 (0.2%)


## 9. 합치기·저장 — `데이터셋.parquet`

- 시료 파일 23칼럼(원본 파일의 값·자료형 그대로) 뒤에 검사 목적·시료 상태·유통기한 → 식품 → 토양 → 기상·대기·풍향 순으로 붙인다.
- 모든 중간 파일은 시료 파일과 같은 행 순서다. 묶음(5만 행)마다 행 키 `lims_data_key` 가 같은지 확인하고 옆으로 붙인다(행을 늘리거나 줄이는 조인은 없다).
- 결측은 채우지 않는다. 숫자 칼럼의 결측은 parquet 의 null 로 적힌다.

In [40]:
orig = pq.read_table(BODY_PATH)                       # 시료 파일 원본 (자료형 그대로)
orig = orig.combine_chunks()
assert orig.num_rows == N_ROWS and orig.column_names == BODY_COLS
assert (orig.column(KEY_COL).to_numpy(zero_copy_only=False) == df[KEY_COL].to_numpy()).all()
base = orig.replace_schema_metadata(None)

PARTS = [('검사 목적·시료 상태·유통기한', MID_DIR / 'process.parquet', PROC_COLS),
         ('식품', MID_DIR / 'food.parquet', FOOD_COLS + FOOD_LABEL_COLS + FOOD_TRACE_COLS),
         ('토양', MID_DIR / 'soil.parquet', SOIL_COLS)] + \
        [(f'기상 {v}', MID_DIR / f'wx_{v}.parquet', wx_cols(v)) for v in WX_VARS]
files = [(name, pq.ParquetFile(path), cols) for name, path, cols in PARTS]
n_groups = -(-N_ROWS // ROW_GROUP)
for name, pf, cols in files:
    assert pf.metadata.num_rows == N_ROWS and pf.metadata.num_row_groups == n_groups, f'{name}: 행 수·묶음 수가 다름'
ADDED_COLS = [c for _, _, cols in PARTS for c in cols]
assert len(set(BODY_COLS + ADDED_COLS)) == len(BODY_COLS) + len(ADDED_COLS), '칼럼 이름이 겹침'

t0 = time.time()
tmp = DATASET_PATH.with_suffix('.parquet.part')
writer = None
for g in range(n_groups):
    lo = g * ROW_GROUP
    part = base.slice(lo, min(ROW_GROUP, N_ROWS - lo))
    key_chunk = part.column(KEY_COL).combine_chunks()
    names, arrays = list(part.column_names), list(part.columns)
    for name, pf, cols in files:
        t = pf.read_row_group(g, columns=[KEY_COL] + cols)
        assert t.num_rows == part.num_rows and t.column(KEY_COL).combine_chunks().equals(key_chunk), f'{name}: 묶음 {g} 의 행 키가 시료 파일과 다름'
        names += cols
        arrays += [t.column(c) for c in cols]
    tbl = pa.Table.from_arrays(arrays, names=names)
    if writer is None:
        writer = pq.ParquetWriter(tmp, tbl.schema)
    writer.write_table(tbl.cast(writer.schema), row_group_size=ROW_GROUP)
    del tbl, arrays, part
writer.close()
os.replace(tmp, DATASET_PATH)
print(f'저장: {DATASET_PATH} ({DATASET_PATH.stat().st_size / 1e9:.2f}GB, {time.time() - t0:.0f}초)')

저장: C:\Users\NT751\Desktop\천영진\리스테리아ML2\data\데이터셋.parquet (2.16GB, 41초)


### 9-1. 저장한 파일 확인 — 행 수·행 키·시료 파일 23칼럼

저장한 `데이터셋.parquet` 를 다시 읽어 확인한다.

In [41]:
DS = pq.ParquetFile(DATASET_PATH)
DS_NAMES = DS.schema_arrow.names
print(f'행 {DS.metadata.num_rows:,} · 칼럼 {len(DS_NAMES):,} · 묶음 {DS.metadata.num_row_groups}')
assert DS.metadata.num_rows == N_ROWS == 676_537, '행 수가 시료 파일과 다름'
assert len(DS_NAMES) == len(set(DS_NAMES)) == len(BODY_COLS) + len(ADDED_COLS)

saved_body = DS.read(columns=BODY_COLS)
keys_saved = saved_body.column(KEY_COL).to_numpy(zero_copy_only=False)
assert (keys_saved == orig.column(KEY_COL).to_numpy(zero_copy_only=False)).all(), '행 키 순서가 시료 파일과 다름'
assert len(set(keys_saved)) == N_ROWS, '행 키 중복'
print(f'행 수 {N_ROWS:,} = 시료 파일 행 수 · 행 키 순서 같음 · 행 키 중복 0')
rows = []
for c in BODY_COLS:
    a, b = saved_body.column(c).to_pandas(), orig.column(c).to_pandas()
    n_diff = int(((a.isna() != b.isna()) | (a.notna() & b.notna() & (a.astype(str) != b.astype(str)))).sum())   # 값으로 비교 (결측끼리는 같음)
    rows.append({'칼럼': c, '자료형': str(saved_body.schema.field(c).type), '원본 자료형과 같음': saved_body.schema.field(c).type == orig.schema.field(c).type,
                 '시료 파일과 같음': n_diff == 0, '다른 행 수': n_diff})
body_check = pd.DataFrame(rows)
display(body_check)
assert body_check['시료 파일과 같음'].all() and body_check['원본 자료형과 같음'].all()   # 23칼럼 모두 시료 파일 그대로
print(f"{TARGET_COL} 값:", saved_body.column(TARGET_COL).to_pandas().value_counts().to_dict())
print('칼럼 묶음별 개수:', {'시료 파일': len(BODY_COLS), '검사 목적·시료 상태·유통기한': len(PROC_COLS), '식품': len(FOOD_COLS + FOOD_LABEL_COLS + FOOD_TRACE_COLS), '토양': len(SOIL_COLS),
                      '기상·대기·풍향': len(WX_COLS), '합': len(DS_NAMES)})
del saved_body, orig, base

행 676,459 · 칼럼 2,166 · 묶음 14


행 수 676,459 = 시료 파일 676,537행 − 지운 78행 · 남은 행의 행 키 순서 같음 · 행 키 중복 0 · 지운 행 키 0개


,칼럼,자료형,원본 자료형과 같음,시료 파일과 같음,다른 행 수
0,lims_data_key,large_string,True,True,0
1,hazard_label,"dictionary<values=string, indices=int8, ordered=0>",True,True,0
2,judge_result,"dictionary<values=string, indices=int8, ordered=0>",True,True,0
3,src_route,"dictionary<values=string, indices=int8, ordered=0>",True,True,0
4,data_source,"dictionary<values=string, indices=int8, ordered=0>",True,True,0
5,expr_iem_nm,"dictionary<values=string, indices=int8, ordered=0>",True,True,0
6,result_value,large_string,True,True,0
7,result_value_num,float,True,True,0
8,judge_stdr,large_string,True,True,0
9,judge_stdr_num,float,True,True,0


judge_result 값: {'불검출': 674490, '검출': 1969}
칼럼 묶음별 개수: {'시료 파일': 23, '검사 목적·시료 상태·유통기한': 6, '식품': 13, '토양': 15, '기상·대기·풍향': 2109, '합': 2166}


### 9-2. 붙인 값이 모두 같은 행 묶음 — 행 수와 "서로 다른 입력" 수는 다르다

기상·대기·풍향·토양은 (좌표, 기준일)로, 식품은 (`prdlst_code`, `train_prdlst_code`)로 붙는다. 그래서 이 값들이 같은 행은 **붙인 값이 전부 같다.** 여기에 기준일 출처(`date_basis`)까지 같은 행을 한 묶음으로 세어, 묶음 수와 검출·불검출이 섞인 묶음을 적는다. 676,537행이 서로 다른 입력 676,537개라는 뜻이 아니라는 점을 보고서에 밝혀야 한다.

- 검출·불검출이 섞인 묶음의 행은 품목·기준일 출처·기상·토양·식품 값만으로는 가를 수 없다. (8절의 검사 목적·시료 상태·유통기한 변수는 행마다 다를 수 있다.)
- 행 키 `lims_data_key` 는 DB 코멘트가 "LIMS 데이터 키 (검사기관코드_시료의뢰번호_시료순번_시험항목순번)"이다. 한 번의 의뢰에 시료 여러 개가 들어 있을 수 있어, 식약처 LIMS 형식 키(밑줄 3개)인 행에 대해 앞 두 토막(검사기관코드_시료의뢰번호)이 같은 행을 한 의뢰로 세어 본다.

In [42]:
gkey = pd.DataFrame({'lat': XY['lat'].to_numpy(), 'lon': XY['lon'].to_numpy(), 'base': df[DATE_COL].to_numpy(),
                     'prdlst_code': df['prdlst_code'].astype(str).to_numpy(), 'train_prdlst_code': df['train_prdlst_code'].astype(str).to_numpy(),
                     'date_basis': df['date_basis'].astype(str).to_numpy()})
gid = gkey.groupby(list(gkey.columns), sort=False, dropna=False).ngroup().to_numpy().astype(np.int64)   # 품목코드가 빈 행도 한 묶음으로 센다
g_size = np.bincount(gid)
g_pos = np.bincount(gid, weights=y.astype(float)).astype(int)
mixed = (g_pos > 0) & (g_pos < g_size)
print(f'(좌표, 기준일, prdlst_code, train_prdlst_code, date_basis)가 같은 묶음: {len(g_size):,}개 (행 {N_ROWS:,})')
print(f'  2행 이상인 묶음에 든 행 {int(g_size[g_size >= 2].sum()):,} ({g_size[g_size >= 2].sum() / N_ROWS * 100:.1f}%) · 가장 큰 묶음 {int(g_size.max()):,}행')
print(f'  검출과 불검출이 섞인 묶음 {int(mixed.sum()):,}개 — 그 안의 검출 {int(g_pos[mixed].sum()):,}행 (전체 검출의 {g_pos[mixed].sum() / y.sum() * 100:.1f}%) · 불검출 {int((g_size - g_pos)[mixed].sum()):,}행')
print(f'  검출만 있는 묶음 {int(((g_pos == g_size) & (g_pos > 0)).sum()):,}개 (검출 {int(g_pos[g_pos == g_size].sum()):,}행)')
key_s = df[KEY_COL].astype(str)
lims3 = (key_s.str.count('_') == 3).to_numpy()
req = key_s[lims3].str.split('_', n=2).str[:2].str.join('_')
req_size = req.value_counts()
req_pos = pd.Series(y[lims3]).groupby(req.to_numpy()).sum()
print(f"식약처 LIMS 형식 키(밑줄 3개) {int(lims3.sum()):,}행 — 의뢰(검사기관코드_시료의뢰번호) {len(req_size):,}건 · 2행 이상인 의뢰에 든 행 {int(req_size[req_size >= 2].sum()):,} ({req_size[req_size >= 2].sum() / lims3.sum() * 100:.1f}%)")
print(f"  검출 {int(req_pos.sum()):,}행이 의뢰 {int((req_pos > 0).sum()):,}건에서 나옴 · 한 의뢰에 검출이 2행 이상인 의뢰 {int((req_pos >= 2).sum()):,}건 (그 검출 {int(req_pos[req_pos >= 2].sum()):,}행)")
del gkey, gid

(좌표, 기준일, prdlst_code, train_prdlst_code, date_basis)가 같은 묶음: 373,206개 (행 676,459)
  2행 이상인 묶음에 든 행 398,812 (59.0%) · 가장 큰 묶음 306행
  검출과 불검출이 섞인 묶음 779개 — 그 안의 검출 1,407행 (전체 검출의 71.5%) · 불검출 7,184행
  검출만 있는 묶음 489개 (검출 562행)


식약처 LIMS 형식 키(밑줄 3개) 669,257행 — 의뢰(검사기관코드_시료의뢰번호) 349,443건 · 2행 이상인 의뢰에 든 행 421,170 (62.9%)
  검출 1,959행이 의뢰 1,295건에서 나옴 · 한 의뢰에 검출이 2행 이상인 의뢰 247건 (그 검출 911행)


## 10. 컬럼설명 — `모든_컬럼설명.parquet`

`리스테리아ML/EXP/1/1_데이터정제.ipynb` 의 `df_컬럼요약` 과 같은 형식(자료형·결측 수·결측 비율·유니크 값 수·최빈값 5개)에 한글명·정의·출처·단위·비고를 더한다.

- **정의** 칸에는 `테이블명세서(모델팀)/테이블정의서.xlsx` 에 **같은 이름의 칼럼 설명이 있을 때만 그 문구를 그대로** 넣는다. 정의서에 그 이름이 없으면 비워 둔다(null). 문구를 고치거나 새로 지어 넣지 않는다(사용자 지시 2026-10-06).
  - 시료 파일 23칼럼은 `ready_data` 시트의 `hazard_preprocessed_data`, `inspct_purps_code`·`splore_sttus_code`·`distb_de` 는 `ba_data 테이블` 시트의 `m_org_lims_data_info`, `pog_daycnt` 는 `t_prdlst_mnf_ledg`, `aw`·`ph`·`na`·`aw_source` 는 `t_aw_cl` 에서 같은 이름을 찾는다.
  - 이 노트북이 새로 만든 칼럼(기상·대기·풍향, 토양, 값 표기, 출처 칼럼 등)은 정의서에 이름이 없어 정의 칸이 비어 있다.
- 이 노트북이 붙인 설명(무슨 값인지, 어떻게 조인했는지, 주의할 점)은 **비고** 칸에 적는다.
- 기상·대기·풍향은 변수마다 111칼럼이라 **변수 하나를 한 줄**로 적는다(`<변수>_d000~d110`). 이 줄의 결측 수·비율은 111칼럼 × 676,537행 칸 기준이다.
- 결측 수는 저장한 `데이터셋.parquet` 에서 다시 센 값이다.

In [43]:
# 테이블정의서의 모든 표 → {(표 이름, 칼럼 이름): 설명 원문}
SPEC_ALL = {}
for sh in ('api_data_info', 'ba_data 테이블', 'ready_data'):
    t = pd.read_excel(SPEC_PATH, sheet_name=sh, header=0)
    t['테이블'] = t['테이블'].ffill().astype(str).str.split('(').str[0].str.strip()      # 표 이름 줄: "이름   (TABLE · …)"
    t = t.dropna(subset=['컬럼'])
    for tb, col, desc in zip(t['테이블'], t['컬럼'].astype(str).str.strip(), t['설명']):
        SPEC_ALL.setdefault((tb, col), None if pd.isna(desc) else str(desc))

SPEC_TABLE_OF = {**{c: 'hazard_preprocessed_data' for c in BODY_COLS},
                 'inspct_purps_code': 'm_org_lims_data_info', 'splore_sttus_code': 'm_org_lims_data_info', 'distb_de': 'm_org_lims_data_info',
                 'pog_daycnt': 't_prdlst_mnf_ledg', 'aw': 't_aw_cl', 'ph': 't_aw_cl', 'na': 't_aw_cl', 'aw_source': 't_aw_cl'}
def spec_def(col):
    """정의서에 같은 이름의 칼럼이 있으면 그 설명 원문, 없으면 None"""
    return SPEC_ALL.get((SPEC_TABLE_OF.get(col), col))

SRC_BODY = 'staging_data.export_body_listeria (시료 파일 listeria_body.parquet)'
BODY_KO = {'lims_data_key': '행 키', 'hazard_label': '위해요소명', 'judge_result': '판정 결과 (타깃)', 'src_route': '국내·해외 구분',
           'data_source': '자료 출처', 'expr_iem_nm': '시험항목명', 'result_value': '검출값 (원본 문자열)', 'result_value_num': '검출값에서 뽑은 숫자',
           'judge_stdr': '규격 기준값', 'judge_stdr_num': '규격 기준값에서 뽑은 숫자', 'inspct_instt_code': '검사기관 코드',
           'prdlst_code': '원본 식품공전 품목코드', 'train_prdlst_code': '학습용 품목코드', 'train_prdlst_nm': '학습용 품목명', 'prduct_nm': '제품명',
           'mnfctur_cmpny_nm': '제조회사명', 'mnfctur_de': '기준일', 'date_basis': '기준일 출처', 'addr': '주소', 'addr_source': '주소 출처',
           'legal_dong_code': '법정동 코드', 'lat': '위도', 'lon': '경도'}
basis_share = ' · '.join(f'{k} {v / N_ROWS * 100:.2f}%' for k, v in df['date_basis'].astype(str).value_counts().items())
BODY_NOTE = {   # 비고 칸에 적을 말 (이 노트북이 확인한 사실)
    'src_route': "정의서에 이 이름의 칼럼 없음. DB 함수 staging_data.build_export_qual 이 ready_data.hazard_preprocessed_data.geo_route 를 src_route 로 내보낸 값(작업기록 2026-10-01). geo_route 의 정의서 설명: "
                 + str(SPEC.get('geo_route', '')) + " / 한국 주소가 로마자로 적힌 행도 시료 파일 값 그대로 overseas 다(2절·5-2절)",
    'data_source': "정의서에 이 이름의 칼럼 없음 — 설명 없음, 확인 필요",
    'result_value': "판정 결과와 같은 검사에서 나온 값 — 예측 변수로 쓰면 정답이 샌다",
    'result_value_num': "정의서에 이 이름의 칼럼 없음. DB 함수 staging_data.build_export_qual 의 식 round(ba_data.jz_num(result_value,'micro'),4) (작업기록 2026-09-22 확인)",
    'judge_stdr': "정의서에 이 이름의 칼럼 없음. DB 함수 staging_data.build_export_qual 이 ready_data.hazard_preprocessed_data.mxmm_value 를 내보낸 값(작업기록 2026-09-22 확인). mxmm_value 의 정의서 설명: " + str(SPEC.get('mxmm_value', '')),
    'judge_stdr_num': "정의서에 이 이름의 칼럼 없음. DB 함수 staging_data.build_export_qual 의 식 round(ba_data.jz_stdr(mxmm_value),4) (작업기록 2026-09-22 확인)",
    'train_prdlst_code': "6절 식품 조인의 둘째 키. 시료 파일의 값은 지금 DB 값과 다른 행이 있다(3-3절)",
    'mnfctur_de': "기상·토양 조인의 기준 날짜. 이 칼럼의 값은 DB ready_data.hazard_preprocessed_data.mnfctur_de_or_fallback(정의서 설명: " + str(SPEC.get('mnfctur_de_or_fallback', ''))
                  + ")과 전 행이 같다(3-3절). 정의 칸의 문구는 이름이 같은 DB 칼럼 mnfctur_de 의 설명이다. 제조일만이 아니다 — 어떤 날짜인지는 date_basis 에 있다(" + basis_share + ")",
    'date_basis': "실제 값은 " + ' · '.join(df['date_basis'].astype(str).value_counts().index) + " 이다. 정의 칸 문구의 '수거일'은 값에 없고 '선적일'은 문구에 없다",
    'addr': "기상·토양 조인에 쓰는 좌표(lat·lon)는 이 주소 자체의 위치가 아니라 이 주소가 속한 법정동의 대표 지점이다(3-3절). 제조 장소만도 아니다 — 주소 출처는 addr_source 에 있다",
    'addr_source': "실제 값은 " + ' · '.join(df['addr_source'].astype(str).value_counts().index) + " 이다. '업소주소'가 누구의 주소인지는 정의서·DB 코멘트에 설명 없음 — 확인 필요",
    'legal_dong_code': "해외 행은 비어 있다(채우지 않음)",
    'lat': "국내 행은 addr 이 속한 법정동(읍면동·시군구·시도 단계) 대표 지점의 위도 — 주소 자체의 위치가 아니다(3-3절). 기상 관측소·토양 법정동을 고르는 기준 좌표",
    'lon': "국내 행은 addr 이 속한 법정동(읍면동·시군구·시도 단계) 대표 지점의 경도 — 주소 자체의 위치가 아니다(3-3절). 기상 관측소·토양 법정동을 고르는 기준 좌표",
}

def col_stats(name):
    s = DS.read(columns=[name]).column(0).to_pandas()
    top = s.value_counts(dropna=True).head(5)
    return {'데이터타입': str(s.dtype), '결측 수': int(s.isna().sum()), '결측 비율(%)': round(float(s.isna().mean()) * 100, 2),
            '유니크 값 수': int(s.nunique(dropna=True)), '최빈값 5개': [f'{v} ({c:,})' for v, c in top.items()]}

def add_row(col, group, ko, src, unit, note):
    rows.append({'컬럼명': col, '묶음': group, '한글명': ko, '정의': spec_def(col), '출처': src, '단위': unit, '비고': note, **col_stats(col)})

rows = []
for c in BODY_COLS:
    add_row(c, '시료 파일', BODY_KO[c], SRC_BODY, '', BODY_NOTE.get(c, ''))

M_ORG = 'ba_data.m_org_lims_data_info'
G8 = '검사 목적·시료 상태·유통기한'
add_row('inspct_purps_code', G8, '검사 목적 코드', f'{M_ORG}.inspct_purps_code', '', 'lims_data_key 로 1:1. 이 테이블에 행이 없는 시료는 결측. DB 코드 그대로. 제조 공정 정보가 아니라 검사를 의뢰한 목적이다(8-4절)')
add_row('inspct_purps_nm', G8, '검사 목적 이름', 'ba_data.t_cm_cmmn_code.cmmn_code_nm', '', '검사 목적 코드의 이름 — 공통코드표에서 cmmn_cl_code = IM02, cmmn_code = inspct_purps_code 인 행의 cmmn_code_nm')
add_row('splore_sttus_code', G8, '시료 상태 코드', f'{M_ORG}.splore_sttus_code', '', 'lims_data_key 로 1:1. DB 코드 그대로. 수입검사·수거검사 행에만 값이 있다(8-4절)')
add_row('splore_sttus_nm', G8, '시료 상태 이름', 'ba_data.t_cm_cmmn_code.cmmn_code_nm', '', '시료 상태 코드의 이름(실온·냉장·냉동 등) — 공통코드표에서 cmmn_cl_code = IM11, cmmn_code = splore_sttus_code 인 행의 cmmn_code_nm')
add_row('distb_de', G8, '유통 일', f'{M_ORG}.distb_de', '', 'lims_data_key 로 1:1. DB 값 그대로(연도가 0019·2999 같은 값, 기준일보다 앞선 값도 고치지 않음). 자료형은 날짜(date). 어떤 날짜인지 설명이 없다. 값으로 보면 유통기한이 끝나는 날짜로 보인다(8-4절) — 식정원 확인 필요. 맞다면 pog_daycnt 와 같은 정보')
add_row('pog_daycnt', G8, '유통기한 (문장)', 'ba_data.t_prdlst_mnf_ledg.pog_daycnt', '', '"제조일로부터 12개월" 같은 문장 그대로. ready_data.hazard_preprocessed_data.prdlst_mnfctur_sttemnt_no = prdlst_report_no 원문 일치로 연결, 대장에 같은 번호가 여러 행이면 연결하지 않음. 대장은 받은 시점의 현재 값이라 기준일 뒤에 수정된 행이 있고, "해당없음"·"-" 같은 문장도 값으로 들어 있다(8-4절)')

key_order_txt = ' → '.join(FOOD_KEY_ORDER)
food_rule = (f"품목코드 단위 값 — 그 시료를 잰 값이 아니다. 조인: {key_order_txt} 순서로 값이 있는 첫 코드(train_prdlst_code 로 붙은 행은 상위 품목 단계의 값). "
             f"두 자료 중 {'엑셀에 값이 있으면 엑셀, 없으면 ba_data.t_aw_cl' if FOOD_PRIORITY == 'excel' else 'ba_data.t_aw_cl 에 값이 있으면 그 값, 없으면 엑셀'}. "
             "원문이 범위면 가운데 값, 부등호면 경계값, 그 밖은 결측. 품목코드가 Z(기준규격외)로 시작하는 행은 어느 자료에도 값이 없어 전부 결측(6-2절)")
FOOD_NOTE = {'aw': ' / 0~1 밖인 값(−0.85, t_aw_cl 원문 "-0.85")이 그대로 있다. t_aw_cl 에는 같은 값이 여러 품목코드에 똑같이 들어 있는 경우가 있다(6-4절). 20자리 품목코드 행은 앞 14자리(그 품목의 14자리 코드)로 붙는다(6-2절)',
             'na': ' / 값 범위는 6-4절. 단위가 %인지는 확인 필요'}
FOOD_SRC = {'aw': 'ba_data.t_aw_cl.aw · 리서치팀 엑셀 `값` 시트 `수분활성도`', 'ph': 'ba_data.t_aw_cl.ph · 리서치팀 엑셀 `값` 시트 `식품 pH`', 'na': 'ba_data.t_aw_cl.na · 리서치팀 엑셀 `값` 시트 `염도`'}
for c in FOOD_COLS:
    add_row(c, '식품', FOOD_KO[c], FOOD_SRC[c], '설명 없음 — 확인 필요', food_rule + FOOD_NOTE.get(c, ''))
for c in FOOD_COLS:
    add_row(f'{c}_label', '식품', f'{FOOD_KO[c]} 값 표기', FOOD_SRC[c], '', f'{c} 값이 원문에서 어떤 표기였는지: 단일값 · 중앙(범위의 가운데 값) · 이상(≥) · 이하(≤) · 초과(>) · 미만(<). 값이 없으면 비어 있음')
FOOD_TRACE_DEF = {
    'aw_key': ('수분활성도 값이 붙은 조인 키', 'aw 값이 어느 키로 붙었는지: prdlst_code(그 품목코드) · train_prdlst_code(학습용 품목코드 — 상위 품목 단계) · prdlst_code 앞 14자리'),
    'aw_src': ('수분활성도 값의 자료', 'aw 값을 가져온 자료: 엑셀(리서치팀 통합 파일) · t_aw_cl(ba_data.t_aw_cl)'),
    'aw_source': ('t_aw_cl 의 aw_source', 'aw 값을 ba_data.t_aw_cl 에서 가져온 행의 t_aw_cl.aw_source 원값(original · recomputed(...) 등). DB 코멘트 없음 — 뜻 확인 필요. 엑셀 값을 쓴 행은 비어 있음'),
    'ph_key': ('식품pH 값이 붙은 조인 키', 'ph 값이 어느 키로 붙었는지 (aw_key 와 같은 값 종류)'),
    'ph_src': ('식품pH 값의 자료', 'ph 값을 가져온 자료: 엑셀 · t_aw_cl'),
    'na_key': ('염도 값이 붙은 조인 키', 'na 값이 어느 키로 붙었는지 (aw_key 와 같은 값 종류)'),
    'na_src': ('염도 값의 자료', 'na 값을 가져온 자료: 엑셀 · t_aw_cl'),
}
for c in FOOD_TRACE_COLS:
    add_row(c, '식품', FOOD_TRACE_DEF[c][0], 'ba_data.t_aw_cl.aw_source' if c == 'aw_source' else '이 노트북 6절의 조인 기록', '', FOOD_TRACE_DEF[c][1])

soil_rule = ("국내 행만. 항목마다 따로 고른다. 검정이 있고 좌표(ba_data_info.korea_region_code)가 있는 법정동마다 '그 항목 값이 있는 검정 중 기준일(mnfctur_de)보다 앞선 가장 최근 exam_day'를 후보로 삼고, "
             f"점수 = max(시료 좌표에서 그 법정동까지 실제 거리(구면 거리, km), 며칠 전 값인지 ÷ {SOIL_DAYS_PER_YEAR:g} × {SOIL_KM_PER_YEAR:g})가 가장 작은 후보의 값(그날 여러 행이면 중앙값). 곧 1km 를 {1 / SOIL_KM_PER_YEAR:g}년과 같은 크기로 보고 거리와 경과 중 큰 쪽이 가장 작은 후보다(사용자 결정 2026-10-08). "
             "점수가 같으면 (거리 + 경과)가 작은 후보, 그래도 같으면 가까운 쪽. 세 항목의 법정동·검정일이 서로 다를 수 있음. 기준일 당일·이후 검정은 쓰지 않음. 이상값·중복 행을 지우지 않음. 경지구분(exam_type)을 가리지 않음. 거리·경과에 상한 없음(<항목>_days_before · <항목>_dist_km 참고). "
             "시료 좌표가 법정동 대표 지점이라 거리는 대표 지점끼리의 거리다(7절)")
SOIL_UNIT = {'soil_ph': 'DB 코멘트·정의서에 단위 없음 (리서치팀 엑셀 표기 "산도(pH)")', 'soil_om': 'DB 코멘트·정의서에 단위 없음 (리서치팀 엑셀 표기 "유기물(g/kg)") — 확인 필요',
             'soil_vldpha': 'DB 코멘트·정의서에 단위 없음 (리서치팀 엑셀 표기 "유효인산(mg/kg)") — 확인 필요'}
for c, (src, ko) in SOIL_ITEMS.items():
    add_row(c, '토양', f'토양 {ko}', f'api_data_info.soil_ph_info.{src}', SOIL_UNIT[c], f'가능한 한 가깝고 가능한 한 최근인 토양검정 {ko} (DB 코멘트 "{ko}"). ' + soil_rule)
for c, (src, ko) in SOIL_ITEMS.items():
    add_row(f'{c}_days_before', '토양', f'토양 {ko} 값이 기준일 기준 며칠 전 값인지', 'api_data_info.soil_ph_info.exam_day', '일', f'기준일(mnfctur_de) − {c} 값을 가져온 검정일(exam_day, DB 코멘트 "토양검정일"). 1 이상. {c} 가 비어 있으면 비어 있다. ' + soil_rule)
for c, (src, ko) in SOIL_ITEMS.items():
    add_row(f'{c}_dong_code', '토양', f'토양 {ko} 값을 가져온 법정동 코드', 'api_data_info.soil_ph_info.legal_dong_code', '', f'{c} 값을 가져온 법정동의 코드. 시료의 legal_dong_code 와 다를 수 있고, 항목마다 다를 수 있다. ' + soil_rule)
for c, (src, ko) in SOIL_ITEMS.items():
    add_row(f'{c}_dist_km', '토양', f'토양 {ko} 값을 가져온 법정동까지 거리', 'ba_data_info.korea_region_code.lat · lon', 'km', f'시료 좌표(법정동 대표 지점)에서 {c} 값을 가져온 법정동 좌표까지 구면 거리(후보를 고를 때 쓴 거리). 주소에서 잰 거리가 아니다. 법정동 좌표가 법정동의 어느 지점인지는 DB 코멘트 없음 — 확인 필요. ' + soil_rule)
for c, (src, ko) in SOIL_ITEMS.items():
    add_row(f'{c}_exam_n', '토양', f'그날 토양 {ko} 값이 있는 검정 행 수', f'api_data_info.soil_ph_info.{src}', '행', f'{c} 값을 가져온 법정동·검정일에 {ko} 값이 있는 검정 행 수. 2 이상이면 값은 그 행들의 중앙값. ' + soil_rule)

# 기상·대기·풍향 — 변수 하나를 한 줄로. 결측 수는 파일에 적힌 칼럼 통계(null 개수)를 더한다
PQ_IDX = {DS.schema.column(i).path: i for i in range(len(DS.schema))}
def null_cells(cols):
    return sum(DS.metadata.row_group(g).column(PQ_IDX[c]).statistics.null_count for g in range(DS.metadata.num_row_groups) for c in cols)
OV_TABLE = 'api_data_info.global_factory_weather'
UNIT_NOTE = {'AIR': ' (리서치팀 엑셀 표기. DB·정의서에는 단위 없음 — 확인 필요)'}
ASOS_JUDGED = {'avg_wind', 'max_wind', 'sunshine', 'solar_rad'}      # ASOS 는 정의서에 단위가 없어 값 범위로 판단한 변수
for v in ALL_WX:
    dom_txt = f"{NET_TABLE[v['prim'][0]]}.{v['prim'][1].removesuffix('_sin').removesuffix('_cos')}" + (
        f" (규칙상 그날 값 있는 관측소가 전국에 없으면 {NET_TABLE[v['fb'][0]]}.{v['fb'][1]} — 이번 데이터셋에서 실제로 쓰인 칸은 0)" if v['fb'] else '')
    ov_txt = (f"{OV_TABLE}.{v['ov'][0]}" + (f" ÷ {v['ov'][1]}" if v['ov'][1] != 1 else '')) if v['ov'] else '없음(결측)'
    cols = wx_cols(v['var'])
    n_null = null_cells(cols)
    extra = ' 최다풍향(도로 본 값)을 관측소마다 ' + v['var'][-3:] + ' 으로 바꾼 뒤 평균.' if v['var'].startswith('wind_dir') else ''
    ov_def = ('해외 행 = 시료와 같은 좌표의 그날 값' + (f"({v['ov'][0]} 을 {v['ov'][1]} 로 나눠 국내 단위로 맞춤)" if v['ov'][1] != 1 else '')
              + ' — 국내 값과 자료 종류가 달라 같은 장소·같은 날에도 값이 다르고, 날짜는 UTC 기준이다(5-2절).') if v['ov'] else '해외 행 = 해외 자료가 없어 전부 결측.'
    unit = v['unit'] + UNIT_NOTE.get(v['prim'][0], '') + (' (ASOS 는 정의서에 단위가 없어 값 범위로 판단)' if v['var'] in ASOS_JUDGED else '')
    rows.append({'컬럼명': f"{v['var']}_d000~d{LAG_MAX:03d}", '묶음': '기상·대기·풍향', '한글명': v['ko'], '정의': None,
                 '출처': f'국내 {dom_txt} / 해외 {ov_txt}', '단위': unit,
                 '비고': (f"{v['ko']} 일자료. 국내 행 = 시료 좌표(법정동 대표 지점)에서 그날 값 있는 관측소 중 실제 거리(km)로 가까운 3곳 평균(거리 상한 없음, 이상값을 지우지 않음), {ov_def}{extra} "
                         f"{len(cols)}칼럼: _d000 = 기준일 당일(기준일이 끝나야 알 수 있는 값), _d001 = 1일 전 … _d{LAG_MAX:03d} = {LAG_MAX}일 전. 기준일은 제조일만이 아니다(2-1절). "
                         f"결측 수·비율은 {len(cols)}칼럼 × {N_ROWS:,}행 칸 기준"),
                 '데이터타입': 'float32', '결측 수': int(n_null), '결측 비율(%)': round(n_null / (len(cols) * N_ROWS) * 100, 2), '유니크 값 수': None, '최빈값 5개': []})

df_컬럼설명 = pd.DataFrame(rows)[['컬럼명', '묶음', '한글명', '정의', '출처', '단위', '데이터타입', '결측 수', '결측 비율(%)', '유니크 값 수', '최빈값 5개', '비고']]
df_컬럼설명['유니크 값 수'] = df_컬럼설명['유니크 값 수'].astype('Int64')
df_컬럼설명['정의'] = df_컬럼설명['정의'].astype(object).where(df_컬럼설명['정의'].notna(), None)
n_described = int((df_컬럼설명['묶음'] != '기상·대기·풍향').sum()) + len(WX_COLS)
assert n_described == len(DS_NAMES), '컬럼설명이 다루는 칼럼 수가 데이터셋 칼럼 수와 다름'
# 정의 칸은 정의서 원문과 글자 하나까지 같아야 한다
for c_, d_ in zip(df_컬럼설명['컬럼명'], df_컬럼설명['정의']):
    assert d_ is None or d_ == SPEC_ALL[(SPEC_TABLE_OF[c_], c_)], c_
df_컬럼설명.to_parquet(COLDESC_PATH, index=False)
print(f'저장: {COLDESC_PATH} — {len(df_컬럼설명)}줄 (데이터셋 {len(DS_NAMES):,}칼럼을 모두 다룸)')
print(f"정의 칸: 정의서 원문이 들어간 줄 {int(df_컬럼설명['정의'].notna().sum())} · 정의서에 이름이 없어 빈 줄 {int(df_컬럼설명['정의'].isna().sum())}")
with pd.option_context('display.max_rows', 100, 'display.max_colwidth', 60):
    display(df_컬럼설명)

저장: C:\Users\NT751\Desktop\천영진\리스테리아ML2\data\모든_컬럼설명.parquet — 76줄 (데이터셋 2,166칼럼을 모두 다룸)
정의 칸: 정의서 원문이 들어간 줄 25 · 정의서에 이름이 없어 빈 줄 51


,컬럼명,묶음,한글명,정의,출처,단위,데이터타입,결측 수,결측 비율(%),유니크 값 수,최빈값 5개,비고
0,lims_data_key,시료 파일,행 키,PK. 소스별 조합 키,staging_data.export_body_listeria (시료 파일 listeria_body.p...,,str,0,0.00,676459,"[O000001_160100103_5_4 (1), O000001_160100300_4_4 (1), O...",
1,hazard_label,시료 파일,위해요소명,위해요소명 (10종),staging_data.export_body_listeria (시료 파일 listeria_body.p...,,category,0,0.00,1,"[리스테리아 (676,459)]",
2,judge_result,시료 파일,판정 결과 (타깃),목표변수. 검출 / 불검출. 병합 시 '하나라도 검출이면 검출',staging_data.export_body_listeria (시료 파일 listeria_body.p...,,category,0,0.00,2,"[불검출 (674,490), 검출 (1,969)]",
3,src_route,시료 파일,국내·해외 구분,None,staging_data.export_body_listeria (시료 파일 listeria_body.p...,,category,0,0.00,2,"[domestic (628,895), overseas (47,564)]",정의서에 이 이름의 칼럼 없음. DB 함수 staging_data.build_export_qual 이...
4,data_source,시료 파일,자료 출처,None,staging_data.export_body_listeria (시료 파일 listeria_body.p...,,category,0,0.00,1,"[식약처 (676,459)]","정의서에 이 이름의 칼럼 없음 — 설명 없음, 확인 필요"
5,expr_iem_nm,시료 파일,시험항목명,시험항목명(대표행 기준). 농관원 행은 조사 구분(anals_se)이 들어간다,staging_data.export_body_listeria (시료 파일 listeria_body.p...,,category,0,0.00,4,"[리스테리아모노사이토제네스 (660,052), 리스테리아 모노사이토제네스 (15,798), 리스테리아...",
6,result_value,시료 파일,검출값 (원본 문자열),검출값(원본 문자열). 정량·정성이 섞여 있어 물질 간 직접 비교 불가,staging_data.export_body_listeria (시료 파일 listeria_body.p...,,str,0,0.00,100,"[음성 (376,707), 음성,음성,음성,음성,음성 (281,857), 불검출 (5,149), 0,...",판정 결과와 같은 검사에서 나온 값 — 예측 변수로 쓰면 정답이 샌다
7,result_value_num,시료 파일,검출값에서 뽑은 숫자,None,staging_data.export_body_listeria (시료 파일 listeria_body.p...,,float32,672682,99.44,2,"[0.0 (3,775), 2.0 (2)]",정의서에 이 이름의 칼럼 없음. DB 함수 staging_data.build_export_qual 의...
8,judge_stdr,시료 파일,규격 기준값,None,staging_data.export_body_listeria (시료 파일 listeria_body.p...,,str,669187,98.92,17,"[n=5,c=0,m=0/25g (4,692), n=5, c=0, m=0/25g (1,817), 음성 ...",정의서에 이 이름의 칼럼 없음. DB 함수 staging_data.build_export_qual 이...
9,judge_stdr_num,시료 파일,규격 기준값에서 뽑은 숫자,None,staging_data.export_body_listeria (시료 파일 listeria_body.p...,,float32,676391,99.99,5,"[100.0 (62), 20.0 (3), 25.0 (1), 1000.0 (1), 0.5 (1)]",정의서에 이 이름의 칼럼 없음. DB 함수 staging_data.build_export_qual 의...


## 11. 원자료로 다시 계산해 대조 — 저장한 `데이터셋.parquet` 기준

저장한 파일을 다시 읽어, `raw/` 원자료에서 **조인 코드와 다른 방법으로** 다시 계산한 값과 비교한다.

**이 대조가 확인하는 것과 확인하지 않는 것**
- 확인하는 것: 조인이 **이 노트북에 적은 규칙대로 계산됐는지**.
- 확인하지 않는 것: 시료 파일의 좌표·기준일·품목코드가 맞는지, 원자료 값 자체가 맞는지(이상한 값도 그대로 들어간다), 규칙이 적절한지.
- 기상·대기·풍향 대조는 표본이다: 50행 × 19개 변수 × 4날짜 = 3,800칸으로, 전체 칸(676,537행 × 2,109칼럼, 약 14억 3천만 칸)의 일부다. 3,800칸 중 결측끼리 같은 칸에는 해외 자료가 없는 변수의 해외 행 칸(5행 × 11개 변수 × 4날짜 = 220칸)이 들어 있다.
- 식품 대조는 값 변환 함수(`parse_food`)를 그대로 다시 써서, 어느 키·어느 자료의 값이 붙었는지는 확인하지만 **변환 규칙 자체가 맞는지는 확인하지 못한다**(예: 원문 "-0.85" 가 음수로 읽히는 것, "6..00~6.70" 이 결측이 되는 것은 그대로 통과한다).
- (참고) 2026-10-06 검수에서 이 노트북의 함수를 쓰지 않고 따로 짠 코드로 국내 전체 행 × D-0·D-110 × 10개 변수를 다시 계산해 다른 행이 없음을 확인했다(노트북 밖에서 한 확인).

| 묶음 | 대조 방법 | 범위 |
|---|---|---|
| 기상·대기·풍향 | 5-1절과 같은 함수 — 원자료 행에서 한 칸씩 직접 계산 | 무작위 50행 × 19개 변수 × D-0·D-1·D-55·D-110 |
| 검사 목적·시료 상태·유통기한 | 원자료 표를 `merge` 로 다시 붙여 비교 | 전체 행 |
| 식품 | 고유 (prdlst_code, train_prdlst_code) 짝마다 두 자료의 원문을 직접 찾아 규칙대로 다시 계산 (값·값 표기·어느 키·어느 자료·`aw_source`) | 전체 행 |
| 토양 | 원자료 행에서 직접: 항목마다 후보 법정동을 실제 거리(km)순으로 하나씩 보며, 그 법정동의 기준일보다 앞선 가장 최근 검정일로 점수(거리와 경과 중 큰 쪽)를 매겨 가장 작은 법정동을 찾고, 그 검정일 행들의 중앙값 | 무작위 국내 300행 + 무작위 해외 20행 |

In [44]:
def read_final(var, cols):
    t = pq.read_table(DATASET_PATH, columns=cols)
    return {c: t.column(c).to_numpy() for c in cols}

verify_weather(read_final, '데이터셋.parquet')

def same_series(a, b):
    """결측끼리는 같다고 보고 두 열을 비교 — 다른 행 수"""
    a, b = pd.Series(a).reset_index(drop=True), pd.Series(b).reset_index(drop=True)
    na, nb = a.isna().to_numpy(), b.isna().to_numpy()
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        eq = np.isclose(a.fillna(0).to_numpy(float), b.fillna(0).to_numpy(float), rtol=1e-6, atol=1e-9)
    else:
        eq = a.astype(str).to_numpy() == b.astype(str).to_numpy()
    return int(((na != nb) | (~na & ~nb & ~eq)).sum())

# 검사 목적·시료 상태·유통기한 — merge 로 다시 붙이기 (전체 행)
saved = DS.read(columns=[KEY_COL] + PROC_COLS).to_pandas()
raw_m = pd.read_parquet(RAW_DIR / 'raw_listeria_m_org.parquet')
raw_codes = pd.read_parquet(RAW_DIR / 'raw_t_cm_cmmn_code_IM02_IM11.parquet')[['cmmn_code', 'cmmn_code_nm']]
raw_ledg = pd.read_parquet(LEDG_PATH, columns=['prdlst_report_no', 'pog_daycnt'])
one = raw_ledg[~raw_ledg['prdlst_report_no'].duplicated(keep=False)]
chk = (saved[[KEY_COL]].merge(raw_m, on=KEY_COL, how='left', validate='one_to_one')
       .merge(raw_codes.rename(columns={'cmmn_code': 'inspct_purps_code', 'cmmn_code_nm': 'inspct_purps_nm'}), on='inspct_purps_code', how='left')
       .merge(raw_codes.rename(columns={'cmmn_code': 'splore_sttus_code', 'cmmn_code_nm': 'splore_sttus_nm'}), on='splore_sttus_code', how='left')
       .merge(one.rename(columns={'prdlst_report_no': 'prdlst_mnfctur_sttemnt_no'}), on='prdlst_mnfctur_sttemnt_no', how='left'))
assert len(chk) == N_ROWS
proc_check = pd.DataFrame([{'칼럼': c, '대조한 행': N_ROWS, '다른 행': same_series(saved[c], chk[c])} for c in PROC_COLS])
display(proc_check)
assert (proc_check['다른 행'] == 0).all()
del saved, raw_m, raw_ledg, one, chk

# 식품 — 고유 품목코드 짝마다 직접 다시 계산 (전체 행)
DB_RAW = pd.read_parquet(RAW_DIR / 'raw_t_aw_cl.parquet').set_index('prdlst_cd')
XL_RAW = pd.read_excel(XLSX_PATH, sheet_name='값', dtype=object).set_index('PRDLST_CD').rename(columns={'수분활성도': 'aw', '식품 pH': 'ph', '염도': 'na'})
def food_by_hand(code_full, code_train, c):
    """→ (숫자, 값 표기, 어느 키, 어느 자료, t_aw_cl.aw_source)"""
    codes = {'prdlst_code': code_full, 'train_prdlst_code': code_train, 'prdlst_code 앞 14자리': code_full[:14]}
    for key in FOOD_KEY_ORDER:
        code = codes[key]
        cand = {'t_aw_cl': DB_RAW.at[code, c] if code in DB_RAW.index else None, '엑셀': XL_RAW.at[code, c] if code in XL_RAW.index else None}
        for src in (('엑셀', 't_aw_cl') if FOOD_PRIORITY == 'excel' else ('t_aw_cl', '엑셀')):
            num, _, label = parse_food(cand[src])
            if not np.isnan(num):
                return num, label, key, src, (DB_RAW.at[code, 'aw_source'] if src == 't_aw_cl' else None)
    return np.nan, '', '', '', None
saved = DS.read(columns=FOOD_COLS + FOOD_LABEL_COLS + FOOD_TRACE_COLS).to_pandas()
pair = pd.DataFrame({'a': df['prdlst_code'].astype(str).to_numpy(), 'b': df['train_prdlst_code'].astype(str).to_numpy()})
uniq = pair.drop_duplicates().reset_index(drop=True)
pos = pd.MultiIndex.from_frame(uniq).get_indexer(pd.MultiIndex.from_frame(pair))
def blank_to_nan(values):
    v = pd.Series(values, dtype=object)
    return v.mask(v.isna() | (v == ''), np.nan)
food_rows = []
for c in FOOD_COLS:
    hand = [food_by_hand(a, b, c) for a, b in zip(uniq['a'], uniq['b'])]
    num = np.array([h[0] for h in hand], dtype=np.float32)[pos]
    checks = [(c, saved[c], num), (f'{c}_label', saved[f'{c}_label'].astype(object), blank_to_nan(np.array([h[1] for h in hand], dtype=object)[pos])),
              (f'{c}_key', saved[f'{c}_key'].astype(object), blank_to_nan(np.array([h[2] for h in hand], dtype=object)[pos])),
              (f'{c}_src', saved[f'{c}_src'].astype(object), blank_to_nan(np.array([h[3] for h in hand], dtype=object)[pos]))]
    if c == 'aw':
        checks.append(('aw_source', saved['aw_source'].astype(object), blank_to_nan(np.array([h[4] for h in hand], dtype=object)[pos])))
    for name, got_col, exp_col in checks:
        food_rows.append({'칼럼': name, '대조한 행': N_ROWS, '고유 품목코드 짝': len(uniq), '다른 행': same_series(got_col, exp_col)})
food_check = pd.DataFrame(food_rows)
display(food_check)
assert (food_check['다른 행'] == 0).all()
del saved, pair

# 토양 — 원자료 행에서 직접 다시 계산 (무작위 국내 300행 + 무작위 해외 20행)
saved = DS.read(columns=SOIL_COLS).to_pandas()
t = pq.read_table(SOIL_PATH, columns=['legal_dong_code', 'exam_day'] + SOIL_SRC, read_dictionary=['legal_dong_code'])
soil_raw = pd.DataFrame({'legal_dong_code': t.column('legal_dong_code').to_pandas().astype(str).to_numpy(),
                         'exam_no': t.column('exam_day').cast(pa.int32()).to_numpy(), **{c: t.column(c).to_numpy() for c in SOIL_SRC}})
del t
soil_raw = soil_raw[soil_raw[SOIL_SRC].notna().any(axis=1)]
krc_raw = pd.read_parquet(RAW_DIR / 'raw_korea_region_code.parquet', columns=['legal_dong_code', 'lat', 'lon']).dropna(subset=['lat', 'lon'])
cand_raw = krc_raw[krc_raw['legal_dong_code'].isin(set(soil_raw['legal_dong_code']))].sort_values('legal_dong_code').reset_index(drop=True)
rows_by_dong = {k: g for k, g in soil_raw[soil_raw['legal_dong_code'].isin(set(cand_raw['legal_dong_code']))].groupby('legal_dong_code')}
del soil_raw

def soil_by_hand(lat, lon, base):
    """항목마다: 후보 법정동을 가까운 순으로 하나씩 보며, 그 법정동에서 그 항목 값이 있는 검정 중 기준일보다 앞선 가장 최근 검정일로
    점수 max(거리 km, 며칠 전 값인지 ÷ 365.25 × 눈금)을 매겨 가장 작은 법정동을 고른다. 점수가 같으면 (거리 + 경과 환산값)이 작은 쪽, 그래도 같으면 먼저 본(가까운) 쪽.
    거리가 지금까지 가장 작은 점수보다 큰 법정동부터는 보지 않는다(점수는 거리보다 작을 수 없다)"""
    b = (pd.Timestamp(base) - pd.Timestamp('1970-01-01')).days
    d = haversine_km(np.float64(lat), np.float64(lon), cand_raw['lat'].to_numpy(float), cand_raw['lon'].to_numpy(float))
    order = np.argsort(d, kind='stable')
    out = {}
    for c, (src, _) in SOIL_ITEMS.items():
        best = None                                   # (점수, 거리 + 경과 환산값, 후보 번호, 그 검정일의 행들)
        for i in order:
            if best is not None and d[i] > best[0]:
                break
            g = rows_by_dong[cand_raw.at[i, 'legal_dong_code']]
            g = g[(g['exam_no'] < b) & g[src].notna()]
            if not len(g):
                continue
            last = int(g['exam_no'].max())
            a_km = (b - last) / SOIL_DAYS_PER_YEAR * SOIL_KM_PER_YEAR
            cur = (max(float(d[i]), a_km), float(d[i]) + a_km)
            if best is None or cur < best[:2]:
                best = (*cur, i, g[g['exam_no'] == last])
        if best is None:
            out.update({c: np.nan, f'{c}_days_before': np.nan, f'{c}_dong_code': None, f'{c}_dist_km': np.nan, f'{c}_exam_n': np.nan})
        else:
            i, gi = best[2], best[3]
            out.update({c: float(gi[src].median()), f'{c}_days_before': float(b - int(gi['exam_no'].iloc[0])),
                        f'{c}_dong_code': cand_raw.at[i, 'legal_dong_code'], f'{c}_dist_km': float(d[i]), f'{c}_exam_n': float(len(gi))})
    return out

def same_num(a, b, tol=0.0):
    """둘 다 결측이거나, 둘 다 값이 있고 차이가 tol 이내"""
    return (pd.isna(a) and pd.isna(b)) or (pd.notna(a) and pd.notna(b) and abs(float(a) - float(b)) <= tol)

rng = np.random.default_rng(SEED)
soil_rows_chk = np.concatenate([rng.choice(np.nonzero(IS_DOM)[0], 300, replace=False),
                                rng.choice(np.nonzero(~IS_DOM)[0], 20, replace=False)])
n_bad, examples = 0, []
for r in soil_rows_chk:
    exp = soil_by_hand(XY.at[r, 'lat'], XY.at[r, 'lon'], df.at[r, DATE_COL]) if IS_DOM[r] else None
    if exp is None:
        ok = all(pd.isna(saved.at[r, c]) for c in SOIL_COLS)
    else:
        ok = all((pd.isna(saved.at[r, f'{c}_dong_code']) and exp[f'{c}_dong_code'] is None) or str(saved.at[r, f'{c}_dong_code']) == exp[f'{c}_dong_code'] for c in SOIL_ITEMS)
        ok &= all(same_num(saved.at[r, f'{c}_dist_km'], exp[f'{c}_dist_km'], tol=1e-3) and same_num(saved.at[r, f'{c}_days_before'], exp[f'{c}_days_before'])
                  and same_num(saved.at[r, f'{c}_exam_n'], exp[f'{c}_exam_n'])
                  and same_num(saved.at[r, c], np.float32(exp[c]), tol=1e-5 * max(1, abs(exp[c])) if not np.isnan(exp[c]) else 0.0) for c in SOIL_ITEMS)
        if len(examples) < 4:
            examples.append({KEY_COL: df.at[r, KEY_COL], '기준일': df.at[r, DATE_COL].date(), '시료 legal_dong_code': df.at[r, 'legal_dong_code'],
                             **{f'다시 계산한 {k}': (round(v, 4) if isinstance(v, float) else v) for k, v in exp.items()}})
    n_bad += not ok
print(f'토양: 대조한 행 {len(soil_rows_chk)}개(국내 300 · 해외 20) 중 다른 행 {n_bad}개')
display(pd.DataFrame(examples).T)
display(saved.loc[soil_rows_chk[:4], SOIL_COLS].assign(**{KEY_COL: df.loc[soil_rows_chk[:4], KEY_COL].to_numpy()}).T)
assert n_bad == 0
del saved, rows_by_dong

[데이터셋.parquet] 대조한 칸 3,800개 중 같은 칸 3,800개, 다른 칸 0개 (대조한 칸 중 결측 220개 — 결측끼리도 같아야 같은 칸)


,칼럼,대조한 행,다른 행
0,inspct_purps_code,676459,0
1,inspct_purps_nm,676459,0
2,splore_sttus_code,676459,0
3,splore_sttus_nm,676459,0
4,distb_de,676459,0
5,pog_daycnt,676459,0


,칼럼,대조한 행,고유 품목코드 짝,다른 행
0,aw,676459,1251,0
1,aw_label,676459,1251,0
2,aw_key,676459,1251,0
3,aw_src,676459,1251,0
4,aw_source,676459,1251,0
5,ph,676459,1251,0
6,ph_label,676459,1251,0
7,ph_key,676459,1251,0
8,ph_src,676459,1251,0
9,na,676459,1251,0


토양: 대조한 행 320개(국내 300 · 해외 20) 중 다른 행 0개


,0,1,2,3
lims_data_key,O000728_240101368_8_2,O000392_240705341_3_4,O000168_230702355_7_2,O000392_250108488_16_3
기준일,2024-03-08,2024-08-23,2023-10-11,2025-05-12
시료 legal_dong_code,4611016300,4471025000,5173031000,4167025000
다시 계산한 soil_ph,6.55,6.5,6.0,6.85
다시 계산한 soil_ph_days_before,191.0,350.0,79.0,21.0
다시 계산한 soil_ph_dong_code,4611016300,4471025024,5173031032,4167025021
다시 계산한 soil_ph_dist_km,0.0001,0.3735,0.6163,0.8398
다시 계산한 soil_ph_exam_n,4.0,5.0,1.0,2.0
다시 계산한 soil_om,25.0,16.5,26.0,8.0
다시 계산한 soil_om_days_before,191.0,350.0,79.0,21.0


,344512,524170,85639,296237
soil_ph,6.55,6.5,6.0,6.85
soil_om,25.0,16.5,26.0,8.0
soil_vldpha,233.149994,348.0,885.0,307.0
soil_ph_days_before,191.0,350.0,79.0,21.0
soil_om_days_before,191.0,350.0,79.0,21.0
soil_vldpha_days_before,191.0,350.0,79.0,21.0
soil_ph_dong_code,4611016300,4471025024,5173031032,4167025021
soil_om_dong_code,4611016300,4471025024,5173031032,4167025021
soil_vldpha_dong_code,4611016300,4471025024,5173031032,4167025021
soil_ph_dist_km,0.0001,0.373535,0.616346,0.839768


### 끝 — 산출물

In [45]:
for p in (DATASET_PATH, COLDESC_PATH):
    print(f'{p.name:16s} {p.stat().st_size / 1e6:>10,.1f} MB  {datetime.fromtimestamp(p.stat().st_mtime):%Y-%m-%d %H:%M}')
print(f'데이터셋: {DS.metadata.num_rows:,}행 × {len(DS_NAMES):,}칼럼 · 검출 {int(y.sum()):,}행')
print('읽는 법(예): pd.read_parquet(데이터셋 경로, columns=[...]) — 기상 칼럼만 약 5.7GB 라 필요한 칼럼만 골라 읽는 편이 안전하다')

데이터셋.parquet        2,157.1 MB  2026-10-08 17:35
모든_컬럼설명.parquet         0.0 MB  2026-10-08 17:35
데이터셋: 676,459행 × 2,166칼럼 · 검출 1,969행
읽는 법(예): pd.read_parquet(데이터셋 경로, columns=[...]) — 기상 칼럼만 약 5.7GB 라 필요한 칼럼만 골라 읽는 편이 안전하다
